In [1]:
''' main.ipynb ''
Notebook: Rigid Body Angular Dynamics with Magnetic Moment in a Rotating Field (with Damping)
Authors: Hamdi Ucar, Daniel Paschall
Date         : 2025-05-01
Revision Date: 2026-05-19
Revision     : 0519.0

This notebook derives equations of motion of a rigid body with diagonal moi carrying a magnetic
moment subject to a magnetic field of constant strength rotating about the z-axis and
numerically integrates them. It includes viscous damping in each Euler rotations.
Features:
1. Simulation of Euler angles θ, φ, ψ
2. Eigenvalue anaysis
3. CSV export of results (θ, φ, ψ and their derivatives)
4. MP4 export of angular motion
5. Symbolic variables and equations in the class EOMDiag (within instance em)

𝗛𝗼𝘄 𝘁𝗼 𝘂𝘀𝗲:
    𝗖𝗹𝗶𝗰𝗸 𝘁𝗵𝗲 𝗯𝘂𝘁𝘁𝗼𝗻 ▶ 𝗮𝘁 𝘁𝗵𝗲 𝘁𝗼𝗼𝗹𝗯𝗮𝗿 𝗷𝘂𝘀𝘁 𝗮𝗯𝗼𝘃𝗲 𝘁𝗵𝗶𝘀 𝗳𝗿𝗮𝗺𝗲 𝗼𝗿 𝗳𝗿𝗼𝗺 𝘁𝗵𝗲 𝘁𝗼𝗽 𝗺𝗲𝗻𝘂 "𝗥𝘂𝗻 - 𝗥𝘂𝗻 𝗦𝗲𝗹𝗲𝗰𝘁𝗲𝗱 𝗖𝗲𝗹𝗹" 𝘁𝗼 𝘀𝘁𝗮𝗿𝘁 𝘁𝗵𝗶𝘀 𝗻𝗼𝘁𝗲𝗯𝗼𝗼𝗸 𝗮𝗽𝗽𝗹𝗶𝗰𝗮𝘁𝗶𝗼𝗻.
    𝗪𝗮𝗶𝘁 𝗳𝗼𝗿 𝘀𝗲𝘃𝗲𝗿𝗮𝗹 𝘀𝗲𝗰𝗼𝗻𝗱𝘀 𝗳𝗼𝗿 𝘁𝗵𝗲 𝗮𝗽𝗽𝗹𝗶𝗰𝗮𝘁𝗶𝗼𝗻 𝘁𝗼 𝘀𝘁𝗮𝗿𝘁. 𝗪𝗵𝗲𝗻 𝗶𝘁 𝘀𝘁𝗮𝗿𝘁, 𝗼𝗻𝗲 𝗰𝗮𝗻 𝘀𝗲𝗲 𝗮 𝘁𝗶𝘁𝗹𝗲 "𝗦𝗼𝗹𝘂𝘁𝗶𝗼𝗻, 𝘀𝗶𝗺𝘂𝗹𝗮𝘁𝗶𝗼𝗻 𝗮𝗻𝗱 𝗮𝗻𝗮𝗹𝘆𝘀𝗶𝘀...". 𝗜𝗳 𝗮𝗻
    𝗲𝗿𝗿𝗼𝗿 𝗼𝗰𝗰𝘂𝗿𝘀 𝗳𝗼𝗿 𝘀𝗼𝗺𝗲 𝗿𝗲𝗮𝘀𝗼𝗻 𝗼𝗿 𝘄𝗲𝗯 𝗽𝗮𝗴𝗲 𝗳𝗮𝗶𝗹𝘀 𝘁𝗼 𝗿𝗲𝘀𝗽𝗼𝗻𝗱, 𝗼𝗻𝗲 𝗰𝗮𝗻 𝗿𝗲𝗽𝗲𝗮𝘁 𝘁𝗵𝗶𝘀 𝗽𝗿𝗼𝗰𝗲𝗱𝘂𝗿𝗲 𝗮𝗻𝗱 𝗮𝗹𝘀𝗼 𝗰𝗮𝗻 𝗰𝗹𝗶𝗰𝗸 𝗯𝘂𝘁𝘁𝗼𝗻 ⟳ "𝗥𝗲𝘀𝘁𝗮𝗿𝘁 𝘁𝗵𝗲 𝗸𝗲𝗿𝗻𝗲𝗹".

How to use:
    Click the button ▶ at the toolbar just above this frame or from the top menu "Run - Run Selected Cell" to start this notebook application.
    Wait for several seconds for the application to start. When it start, one can see a title "Solution, simulation and analysis...". If an
    error occurs for some reason or web page fails to respond, one can repeat this procedure and also can click button ⟳ "Restart the kernel".
'''

%matplotlib ipympl
from   typing import Callable, Optional, Literal, Tuple, Dict, Any
from   ipywidgets import interact, FloatText, IntText, BoundedIntText, BoundedFloatText, Label, Button, ToggleButtons, Text, Checkbox, Dropdown, Select, IntProgress, FloatSlider, SelectionRangeSlider, FloatLogSlider, HBox, VBox, GridBox, Accordion, HTML as iHTML, HTMLMath, Layout, Output, ColorPicker, FloatRangeSlider, interactive_output
#from   ipyfilechooser import FileChooser
import ipywidgets as W
import ipyvuetify as VU
#from ipyevents import Event
import numpy as np
import sympy as sp
import math
from   mpmath import findroot,mpc
from   sympy import Eq, latex, simplify, linear_eq_to_matrix
from   sympy import Add, Mul, sin as sin, cos as cos, tan as tan, asin as asin
from   scipy.optimize import linear_sum_assignment
from   scipy.integrate import solve_ivp

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from   matplotlib.figure import Figure
from   matplotlib.widgets import RangeSlider, Button as mButton
from   matplotlib.animation import FuncAnimation
from   matplotlib.backend_tools import ToolBase
import matplotlib.ticker as mticker
from   matplotlib.ticker import AutoMinorLocator, FormatStrFormatter
from   matplotlib.font_manager import FontProperties
from   matplotlib.backend_bases import MouseEvent, MouseButton
from   matplotlib.transforms import IdentityTransform
from   matplotlib.collections import LineCollection

import plotly.graph_objs as go
import plotly.io as pio

from   ipywidgets import interact
from   IPython.utils import io
from   IPython.display import display, clear_output, Latex, Math,  Markdown, HTML, Video, Javascript
import re
import csv
import inspect
import time
from   datetime import datetime
import os
import shutil
import json
import pyperclip
import ast
import threading
import hashlib
import base64
import pandas as pd
import ffmpeg
from   IPython import get_ipython
import traceback
#from   tornado.ioloop import IOLoop
from   mpl_toolkits.mplot3d import Axes3D  # Needed for 3D projection
from   scipy import signal
from   scipy.signal import periodogram, welch, find_peaks
from   scipy.fft import rfft, rfftfreq
import scipy.linalg as la

from   eom_diag import EOMDiag
from   animate import AnimateSim, GetViewAngles, SetViewAngles, GetShape, GetScale, SetShape, SetScale, xyz_buttons_css, CancelAnim

import importlib
import doc_boxes
importlib.reload(doc_boxes)

from   doc_boxes import rem_par, ResearchContext, QuickGuideAccordion, DerivationAccordion, R_MatrixAccordion, SupBoxAccordion, SemiEmpAccordion

import warnings
warnings.filterwarnings("error", category=RuntimeWarning)

#import logging

# Set all loggers to DEBUG and add traceback info
#logging.basicConfig(
#    level=logging.DEBUG,
#    format='%(name)s - %(levelname)s - %(filename)s:%(lineno)d - %(message)s',
#    force=True
#)

def make_img_css(class_name, png_path):
    b64 = base64.b64encode(open(png_path, "rb").read()).decode("ascii")
    return f"""
.{class_name} {{
  width: 28px;
  height: 28px;
  padding: 0;
  border: none;
  background-image: url("data:image/png;base64,{b64}");
  background-repeat: no-repeat;
  background-position: center;L
  background-size: contain;
}}
"""

css = "\n".join([
    make_img_css("xy", "view_xy.png"),
    make_img_css("yz", "view_yz.png"),
    make_img_css("xz", "view_xz.png"),
])
display(HTML(f"<style>\n{css}\n</style>"))
display(HTML("""
<style>
.sb {font-weight: 600}
/* target JupyterLab’s rendered Markdown tables */
.jp-RenderedHTMLCommon table td,
.jp-RenderedHTMLCommon table th {
  padding: 2px 6px !important;
  line-height: 1.1 !important;
}

/* target accordeon large paddings */

.jupyter-widget-Collapse-header {
    padding-top: 6px !important;
    padding-bottom: 6px !important;
    padding-left: 6px !important;
    min-height: unset !important;
    line-height: 1.2 !important;
}

.lm-Widget.lm-Panel.jupyter-widget-Collapse-contents {
    padding: 0 0 0 6px !important;
}

/* Target only range sliders */
:root {
  --jp-widgets-slider-handle-size: 12px;   /* default ~24px */
  --jp-widgets-slider-track-thickness: 3px;
}

/* target the button element when the class is applied to the button itself */
button.btn-center,
.lm-Widget.btn-center > button,
.jp-RenderedHTMLCommon .btn-center button {
  display: inline-flex !important;
  align-items: center !important;       /* vertical centering */
  justify-content: center !important;   /* horizontal centering */
  padding: 0 6px !important;
  /* height: 20px !important; */
  line-height: normal !important;       /* prevent inherited line-height oddness */
  box-sizing: border-box !important;
}

/* also ensure any internal span doesn't pull text down */
button.btn-center > span,
.lm-Widget.btn-center > button > span {
  display: inline-block !important;
  vertical-align: middle !important;
  line-height: normal !important;
}
.sanino select {
  /* font-size: 9pt !important; */
  padding-left: 2px !important;
  padding-right: 2px !important;
}

/* fix the large gap between the bar and the readout */
/*
.jupyter-widget-hslider .slider-container {
    margin-right: 4px !important;
    flex: 1 1 auto !important;
}
*/
.widget-readout {
    min-width: 32px !important;
    padding: 2px 4px !important;
}
</style>
"""))

#.lm-Widget.jupyter-widget-Collapse.jupyter-widget-Accordion-child {
#    margin-left: 0 !important;
#    padding-left: 0 !important;
#}


#──────────────────────────────────────────────────────────────────────
N_PLOTS = 11
DEGREE_MODE = False
VEL_UNITS = ('rad/s','rev/s')
ANG_UNITS = ('rad','deg')
ACC_UNITS = ('rad/s²','rev/s²')
deg_mode_action_suppress = False
glw = 0.8 # general line width of plots
#kernel = get_ipython().kernel
io_loop = get_ipython().kernel.io_loop

def angunit():
    return ANG_UNITS[DEGREE_MODE]

def velunit():
    return VEL_UNITS[DEGREE_MODE]

def accunit():
    return ACC_UNITS[DEGREE_MODE]

# ——————————————————————————————————————————————————————————————————————
class BaseSaveDialog:
    def __init__(self, resource_name: str, callback: Callable[[str], None], btn_color: str = "success"):
        """
        resource_name: e.g. "simulation data", "derivation data", "animation file"
        callback: called with "a" (add) or "r" (replace)
        """
        self.resource_name = resource_name
        self.callback = callback

        title = f"Save {resource_name.title()}"
        msg   = (f"One or more {resource_name} already present on this project.")

        # Dialog container
        self.dialog = VU.Dialog(v_model=False, max_width="500px", persistent=False)

        # Buttons
        add_label     = f"Add new {resource_name}"
        replace_label = f"Replace last {resource_name}"
        self.btn_add     = VU.Btn(children=[add_label],     color=btn_color, small=True, style_="text-transform: none;")
        self.btn_replace = VU.Btn(children=[replace_label], color=btn_color, small=True, style_="text-transform: none;")
        self.btn_cancel  = VU.Btn(children=["Cancel"],      color="grey",    small=True, style_="text-transform: none;")

        # Events
        self.btn_add.on_event('click',     self._on_add)
        self.btn_replace.on_event('click', self._on_replace)
        self.btn_cancel.on_event('click',  self._on_cancel)

        # Assemble
        card = VU.Card(
            style_="padding: 10px;",
            children=[
                VU.CardTitle(children=[title], class_="headline"),
                VU.CardText(children=[msg], style_="font-size: 14px;"),
                VU.CardActions(children=[self.btn_add, self.btn_replace, VU.Spacer(), self.btn_cancel])
            ]
        )
        self.dialog.children = [card]

    def _on_add(self, *args):
        self.dialog.v_model = False
        self.callback("a")

    def _on_replace(self, *args):
        self.dialog.v_model = False
        self.callback("r")

    def _on_cancel(self, *args):
        self.dialog.v_model = False

    def show(self):
        """Pop up the dialog (you must have already `display(dialog)` once)."""
        self.dialog.v_model = True

# ──────────────────────────────────────────────────────────────────────
class SimSaveDialog(BaseSaveDialog):
    def __init__(self, callback: Callable[[str], None]):
        super().__init__("simulation data", callback, btn_color="success")

# ──────────────────────────────────────────────────────────────────────
class AnimSaveDialog(BaseSaveDialog):
    def __init__(self, callback: Callable[[str], None]):
        super().__init__("animation file", callback, btn_color="success")

# ——————————————————————————————————————————————————————————————————————
class SelectDeleteDialog:
    def __init__(self, title_select: str, title_delete: str, on_select: Callable[[str], None], on_delete: Callable[[str], None], on_clean: Callable[[str], None]):
        self.title_select = title_select
        self.title_delete = title_delete
        self.on_select = on_select
        self.on_delete = on_delete
        self.on_clean = on_clean
        self.action = True  # True=select, False=delete
        # The dialog (hidden by default)
        self.dialog = VU.Dialog(v_model=False, max_width="500px", persistent=True)

        # Cancel/OK button (we’ll relabel in show_select/show_delete)
        self.btn_cancel = VU.Btn(children=["Cancel"], color="blue-grey-darken-1", text=True, small=True, style_="text-transform: none;" )
        self.btn_cancel.on_event('click', self._close)

        self.btn_clean = VU.Btn(children=["Clean"], color="blue-grey-darken-1", text=True, small=True, style_="text-transform: none;" )
        self.btn_clean.on_event('click', self._clean)

        # Build the static card shell
        self._build_static_card(self.title_select)

        # Display it once so it’s wired into the notebook UI
        display(self.dialog)

    def _build_static_card(self, title):
        """Construct title, text container, and actions bar."""
        self.card_title = VU.CardTitle(
            children=[title],
            style_="font-size: 16px; font-weight: 500;"
        )
        self.card_text = VU.CardText(style_="font-size: 14px;")
        actions = VU.CardActions(children=[self.btn_clean, VU.Spacer(), self.btn_cancel])
        self.card = VU.Card(
            children=[self.card_title, self.card_text, actions],
            style_="padding: 10px;"
        )
        self.dialog.children = [self.card]

    def _close(self, *args):
        self.dialog.v_model = False

    def _clean(self, *args):
        self.file_list = self.on_clean()
        self.set_file_list()

    def _make_click_handler(self, filename):
        def handler(*_):
            self._close()
            if self.action:
                self.on_select(filename)
            else:
                self.on_delete(filename)
        return handler

    def set_file_list(self):
        """Populate the list of filename-buttons."""
        btns = []
        for entry in self.file_list:
            if entry:
                btn = VU.Btn(
                    children=[entry],
                    color="blue-grey-darken-1",
                    text=True,
                    small=True,
                    style_="text-transform: none; font-size: 13px;",
                    class_="ma-0"
                )
                btn.on_event('click', self._make_click_handler(entry))
                btns.append(btn)

        # stack vertically with a small gap
        container = VU.Layout(
            children=btns,
            class_='d-flex flex-column',
            style_="gap: 6px;"
        )
        self.card_text.children = [container]

    def show_select(self, file_list):
        """Open in SELECT mode."""
        self.action = True
        self.file_list = file_list
        self.btn_cancel.children = ["Cancel"]
        self.card_title.children = [self.title_select]
        self.set_file_list()
        self.dialog.v_model = True

    def show_delete(self, file_list):
        """Open in DELETE mode."""
        if file_list:
            self.action = False
            self.file_list = file_list
            self.btn_cancel.children = ["Close"]
            self.card_title.children = [self.title_delete]
            self.set_file_list()
            self.dialog.v_model = True

# ──────────────────────────────────────────────────────────────────────
class AngleText(FloatText):
    # registry of all instances
    _registry = []

    def __init__(self, *, value=0.0, rad_min=-math.inf, rad_max=math.inf, rad_step=0.1, **kwargs):
        # store internal radian value and bounds
        self._rad      = value
        self._rad_min  = rad_min
        self._rad_max  = rad_max
        self._rad_step = rad_step
        disp_step = self._to_display(self._rad_step)

        super().__init__(value=0.0, step=disp_step, **kwargs)

        # push our true initial rad through the property
        self.value = value

        # register this instance
        AngleText._registry.append(self)

        # Observe changes and clamp
        self.observe(self._on_display_change, names='value')

    def _to_display(self, rad_val: float) -> float:
        return round(math.degrees(rad_val), 12) if DEGREE_MODE else rad_val

    def _from_display(self, disp_val) -> float:
        return math.radians(disp_val) if DEGREE_MODE else disp_val

    def _on_display_change(self, change):
        try:
            rad = self._from_display(change['new'])
            self._rad = min(max(rad, self._rad_min), self._rad_max)
            if self._rad != rad:
                object.__setattr__(self, 'value', self._to_display(self._rad))  # silently update
        except Exception:
            pass  # optionally add handling

    @property
    def valRad(self) -> float:
        """Return the angle in radians."""
        return float(self._rad)

    @valRad.setter
    def valRad(self, rad: float):
        """Accept a radian value, store it, then update the displayed number."""
        # store radian
        self._rad = float(rad)

        # compute what the user should see
        self.value = self._to_display(self._rad)

    def toggle_mode(self):
        # FloatText widgets in ipywidgets can round values unexpectedly when their step attribute is set
        self.step = self._rad_step if DEGREE_MODE else self._rad_step * 0.1
        self.value = dval = self._to_display(self._rad)

    @classmethod
    def update_all_modes(cls):
        """Call toggle_mode on every registered widget."""
        for w in cls._registry:
            w.toggle_mode()

# ──────────────────────────────────────────────────────────────────────
class AngVelocityText(FloatText):
    # registry of all instances
    _registry = []

    def __init__(self, *, value=0.0, rad_min=-math.inf, rad_max=math.inf, rad_step=0.1, **kwargs):
        # store internal radian value and bounds
        self._rad      = value
        self._rad_min  = rad_min
        self._rad_max  = rad_max
        self._rad_step = rad_step

        # compute initial display bounds & value
        disp_step = self._to_display(self._rad_step)

        super().__init__(
            value=0.0,    # temp placeholder
            step=disp_step,
            **kwargs
        )

        # push our true initial rad through the property
        self.value = value

        # 2) register this instance
        AngVelocityText._registry.append(self)

        # Observe changes and clamp
        self.observe(self._on_display_change, names='value')

    def _to_display(self, rad_val: float) -> float:
        #return rad_val/(np.pi*2) if DEGREE_MODE else rad_val
        return round(rad_val/(np.pi*2), 12) if DEGREE_MODE else rad_val

    def _from_display(self, disp_val) -> float:
        return disp_val*np.pi*2 if DEGREE_MODE else disp_val

    def _on_display_change(self, change):
        try:
            rad = self._from_display(change['new'])
            self._rad = min(max(rad, self._rad_min), self._rad_max)
            if self._rad != rad:
                object.__setattr__(self, 'value', self._to_display(self._rad))  # silently update
        except Exception:
            pass  # optionally add handling

    @property
    def valRad(self) -> float:
        """Return the angle in radians."""
        return self._rad

    @valRad.setter
    def valRad(self, rad: float):
        """Accept a radian value, store it, then update the displayed number."""
        # store radian
        self._rad = float(rad)
        self.value =self._to_display(self._rad) # compute what the user should see

    def toggle_mode(self):
        #FloatText widgets in ipywidgets can round values unexpectedly when their step attribute is set
        self.step = self._rad_step*0.1 if DEGREE_MODE else self._rad_step
        self.value = dval = self._to_display(self._rad)

    @classmethod
    def update_all_modes(cls):
        """Call toggle_mode on every registered widget."""
        for w in cls._registry:
            w.toggle_mode()

# ——————————————————————————————————————————————————————————————————————
def LabeledFloat(desc, val, unit, step_size, minv, maxv, desctip):
    def clamp(change):
        v = change['new']
        if v < w.min:
            w.value = w.min
        elif v > w.max:
            w.value = w.max

    w = BoundedFloatText(value=val,
            step=step_size,
            min=minv,
            max=maxv,
            description=desc,
            tooltip=desctip,
            layout=Layout(width='130px', margin='0 3px 0 0'),  # top right bottom left
            style= {'description_width': '57px'})

    w.observe(clamp, names='value')
    if unit is not None:
        lb_unit = Label(value=unit, margin='0 10px 0 5px', layout=Layout(width='20px')) # top right bottom left
        box = HBox([w, lb_unit],layout=Layout(margin="0"))
    else:
        box = HBox([w],layout=Layout(margin="0"))
    return w, box

# ——————————————————————————————————————————————————————————————————————
def LabeledFloatK(desc, val, unit, step_size, minv, maxv, desctip):
    def clamp(change):
        v = change['new']
        if v < w.min:
            w.value = w.min
        elif v > w.max:
            w.value = w.max

    w = BoundedFloatText(value=val,
            step=step_size,
            min=minv,
            max=maxv,
            description=desc,
            tooltip=desctip,
            layout=Layout(width='110px', margin='0 3px 0 0'),  # top right bottom left
            style= {'description_width': '35px'})

    w.observe(clamp, names='value')
    if unit is not None:
        lb_unit = Label(value=unit, margin='0 10px 0 5px', layout=Layout(width='20px')) # top right bottom left
        box = HBox([w, lb_unit],layout=Layout(margin="0"))
    else:
        box = HBox([w],layout=Layout(margin="0"))
    return w, box

# ——————————————————————————————————————————————————————————————————————
def LabeledFloat2(desc, val, unit, step_size, minv, maxv, desctip=''):
    def clamp(change):
        v = change['new']
        if v < w.min:
            w.value = w.min
        elif v > w.max:
            w.value = w.max

    w = BoundedFloatText(value=val,
            step=step_size,
            min=minv,
            max=maxv,
            description=desc,
            tooltip=desctip,
            layout=Layout(width='193px', margin='0 5px 0 0'),
            style={'description_width': '24px'})

    lb_unit = Label(value=unit, layout=Layout(width='50px'))
    box = HBox([w, lb_unit])
    return w, box

# ——————————————————————————————————————————————————————————————————————
# angle entry widget, can work with rad and degrees
class LabeledFloatA:
    _registry = []  # will hold tuples (AngleText instance, Label instance)

    @staticmethod
    def create(desc, val, step_size, minv, maxv, desctip=''):
        w = AngleText(value=val,
                rad_step=step_size,
                rad_min=minv,
                rad_max=maxv,
                description=desc,
                tooltip=desctip,
                layout=Layout(width='191px', margin='0 5px 0 0'),
                style={'description_width': '24px'})


        #lb_desc = Label(value=desc, layout=Layout(width='25px', margin='8px 2px 0 0'), tooltip=desctip)
        lb_unit = Label(value='rad', layout=Layout(width='50px'))

        LabeledFloatA._registry.append((w, lb_unit))
        return w, HBox([w, lb_unit])

    @staticmethod
    def update_all():
        """Call after you flip DEGREE_MODE; updates every unit label."""
        for w, lb in LabeledFloatA._registry:
            lb.value = 'deg' if DEGREE_MODE else 'rad'

# ——————————————————————————————————————————————————————————————————————
# angle/s entry widget, can work with rad/s and cycles per second
class LabeledFloatB:
    _registry = []  # will hold tuples (AngleText instance, Label instance)

    @staticmethod
    def create(desc, val, step_size, minv, maxv, units: tuple|None, desctip=''):
        w = AngVelocityText(value=val,
                rad_step=step_size,
                rad_min=minv,
                rad_max=maxv,
                description=desc,
                tooltip=desctip,
                layout=Layout(width='191px', margin='0 5px 0 0'),
                style={'description_width': '24px'})

        #lb_desc = Label(value=desc, layout=Layout(width='25px', margin='8px 2px 0 0'), tooltip=desctip)
        if units is None:
            units = VEL_UNITS
        lb_unit = Label(value=units[0], layout=Layout(width='50px'))
        lb_unit.units = units
        LabeledFloatB._registry.append((w, lb_unit))
        return w, HBox([w, lb_unit])

    @staticmethod
    def update_all():
        """Call after you flip DEGREE_MODE; updates every unit label."""
        for w, lb in LabeledFloatB._registry:
            lb.value = lb.units[DEGREE_MODE]

# ──────────────────────────────────────────────────────────────────────
#anim_done_event = threading.Event()
class StatusLabel(W.HTML):
    def __init__(self, text='', color='--jp-ui-font-color1'):
        super().__init__()
        self.set(text, color)

    def set(self, text, color='--jp-ui-font-color1'):
        def replace_subscripts(s):
            # Replace _{...} with HTML subscript
            return re.sub(r'_\{([^}]*)\}', r'<sub>\1</sub>', s)

        styled_text = (
            f"<span style='color:{color}; font-size: 14px; font-weight: 600; height:18px'>"
            f"{replace_subscripts(text)}</span>"
        )
        self.value = styled_text

    def setM(self, text, color='--jp-ui-font-color1'):

        styled_text = (
                f"<div style='color:{color}; font-size: 14px; font-weight: 400; height:18px'>"
                f"{text}"
                f"</div>"
        )
        self.value = styled_text


# ──────────────────────────────────────────────────────────────────────
def push_minus_inside(expr):
    if expr.is_Mul and expr.as_coeff_mul()[0] == -1:
        factors = expr.as_coeff_mul()[1]
        # Case: -(A + B + ...) → (-A - B - ...)
        if any(f.is_Add for f in factors):
            return Mul(*[(-f if f.is_Add else f) for f in factors])
        # Case: -(-X) → X
        if len(factors) == 1:
            inner = factors[0]
            ci, fi = inner.as_coeff_mul()
            if ci == -1:
                return Mul(*fi)
    return expr.xreplace({arg: push_minus_inside(arg) for arg in expr.args})

# ──────────────────────────────────────────────────────────────────────
def clean_small_entries(Anum, tol=1e-15):
    """
    Replace entries of numeric sympy.Matrix A_num that are closer than tol to 0
    with exact integer 0. Returns a new sympy.Matrix.
    """
    if not isinstance(Anum, sp.Matrix):
        Anum = sp.Matrix(Anum)

    M = Anum.copy()
    for i in range(M.rows):
        for j in range(M.cols):
            val = M[i, j]
            # evaluate to a numeric Python complex number (works for real/complex sympy numbers)
            try:
                valc = complex(val.evalf())
            except Exception:
                # fallback to float conversion (shouldn't normally be needed after numeric substitution)
                valc = complex(float(val))
            if abs(valc) < tol:
                M[i, j] = sp.Integer(0)
            else:
                # keep a numeric value (reduce symbolic noise)
                M[i, j] = sp.N(val, 15)   # 15 digits; change if you want more precision
    return M

# ──────────────────────────────────────────────────────────────────────

#rhs_drive_amp=0.1
#rhs_freq0=60*np.pi
#rhs_drift=-1

def rhs(t, y, I1, I2, I3, mB, om, gam, ps_acc, xi=0):
    th, ph, ps, thd, phd, psd = y
    thdd = em.f_th(th, ph, ps, thd, phd, psd, I1, I2, I3, mB, om, gam, xi, t)
    phdd = em.f_ph(th, ph, ps, thd, phd, psd, I1, I2, I3, mB, om, gam, xi, t)
    psdd = em.f_ps(th, ph, ps, thd, phd, psd, I1, I2, I3, mB, om, gam, xi, t) + ps_acc  # ps_acc is actually added

#    freq_t = rhs_freq0 + rhs_drift * t
#    # Phase from integrating freq over time
#    phase = 2 * np.pi * (rhs_freq0 * t + 0.5 * rhs_drift * t**2)
#    # Add periodic acceleration to theta equation
#    thdd += rhs_drive_amp * np.sin(phase)

    return [thd, phd, psd, thdd, phdd, psdd]

# ──────────────────────────────────────────────────────────────────────
# Simulation and export
def simulate_euler(params, init, omega_body:float, t_start:float, t_end:float, ps_acc:float, n_steps: int, solver_params:tuple):
    """
    Integrate the Euler‐angle ODEs [θ,φ,ψ,θ̇,φ̇,ψ̇] with optional damping.

    Uses solve_ivp with:
      • method='DOP853'  (8th‐order RK)
      • rtol=1e-9, atol=1e-12
      • max_step = t_end / n_steps

    Returns:
      t_vals: array of length n_steps
      th_vals, ph_vals, ps_vals: arrays of angles over time
    """
    #print('t_start=',t_start)
    #stop_event.clear()
    I1_val, I2_val, I3_val, mB_val, omega_val, gamma_val, xi = params
    th0, ph0, ps0 = init
    th_dot0, ph_dot0, ps_dot0 = omega_body
    solv_method, solv_max_step, rel_tol, abs_tol = solver_params
    max_step = 5e-4
    t_eval = np.linspace(t_start, t_end, n_steps+1) # +1 !!!
    try:
        sol = solve_ivp(
            lambda tt, yy: rhs(tt, yy, I1_val, I2_val, I3_val, mB_val, omega_val, gamma_val, ps_acc, xi),
            [t_start, t_end],
            [th0, ph0, ps0, th_dot0, ph_dot0, ps_dot0],
            method=solv_method,  # 'DOP853',
            t_eval=t_eval,
            rtol=rel_tol,
            atol=abs_tol,
            max_step=solv_max_step
        )
        if not sol.success:
            print(f"Integration failed: {sol.message}")
            return None, None

        return sol.t, sol.y
    except Exception as e:
        print (f"Integration failed: {str(e)}")
        return None, None

# ──────────────────────────────────────────────────────────────────────
# Euler rates conversion (body ω → Euler rates) - Not used
def euler_rates_from_omega(omega_body, _th0, _ph0, _ps0):
    """
    Convert body-frame angular velocity ω_body = (ωx,ωy,ωz)
    to Euler-angle rates [θ̇, φ̇, ψ̇] for ZXZ convention by inverting:
        [ωx]   [ cosψ        sinθ·sinψ   0 ] [θ̇]
        [ωy] = [ -sinψ       sinθ·cosψ   0 ] [φ̇]
        [ωz]   [ 0           cosθ        1 ] [ψ̇]
    """
    wx, wy, wz = omega_body
    s_th, c_th = np.sin(_th0), np.cos(_th0)
    s_ps, c_ps = np.sin(_ps0), np.cos(_ps0)
    # check for singularity (sinθ ≈ 0)
    if abs(s_th) < 1e-8:
        raise ValueError('Gimbal lock: θ near 0 or π')
    # Build transformation matrix A
    A = np.array([
        [    c_ps,        s_th * s_ps, 0],
        [   -s_ps,        s_th * c_ps, 0],
        [      0,              c_th,    1]
    ])
    # Solve A · [θ̇, φ̇, ψ̇]^T = [ωx, ωy, ωz]
    rates = np.linalg.solve(A, np.array([wx, wy, wz]))
    return rates[0], rates[1], rates[2]

#——————————————————————————————————————————————————————————————————————
# Plotting procedures
#——————————————————————————————————————————————————————————————————————
def compute_T(I1, I2, I3, th, ph, ps, phd, thd, psd):
    """
    Compute kinetic energy array T for a spinning axisymmetric rigid body:
      T = I1*(sin^2 θ·φ̇^2/2 + θ̇^2/2)
        + I3*(−sin^2 θ·φ̇^2/2 + cosθ·φ̇·ψ̇ + φ̇^2/2 + ψ̇^2/2)

    Parameters
    ----------
    I1, I3      : scalars, principal moments of inertia
    theta       : array_like, θ(t) values
    phi_dot     : array_like, φ̇(t) values
    theta_dot   : array_like, θ̇(t) values
    psi_dot     : array_like, ψ̇(t) values

    Returns
    -------
    T : ndarray, same shape as theta
    """
    s_ps = np.sin(ps)
    c_ps = np.cos(ps)
    s_th = np.sin(th)
    c_th = np.cos(th)

    t1 = I1 * (s_ps * s_th *  phd + c_ps * thd)**2
    t2 = I2 * (s_ps * thd  - s_th * c_ps * phd)**2
    t3 = I3 * (c_th * phd + psd)**2

    return (t1+t2+t3)/2

    #sin2 = np.sin(th)**2
    #T1 = I1 * ( sin2 * phd**2 / 2  +  thd**2 / 2 )
    #T3 = I3 * ( -sin2 * phd**2 / 2 + np.cos(th) * phd * psd + phd**2 / 2 + psd**2 / 2 )
    #return T1 + T3

#——————————————————————————————————————————————————————————————————————
def compute_V(omega_, mB_, gamma_, t_arr, theta_arr, phi_arr):
    """
    Compute potential energy array V(t) using:
        V(t) = -mB [sin(γ)·cos(θ) + sin(θ)·cos(γ)·cos(ωt − φ)]
             = TS·cos(θ) + TC·sin(θ)·cos(ωt − φ)

    Parameters
    ----------
    omega_     : scalar, angular frequency of rotating field (rad/s)
    mB_        : scalar, magnitude of magnetic moment * magnitude of rotating magnetic field
    gammma_    : scalar, γ , tilt angle of the rotating field (radians)
    t_arr      : array, time values
    theta_arr  : array, θ(t)
    phi_arr    : array, φ(t)

    Returns
    -------
    V : ndarray, potential energy values at each time
    """
    TS = -mB_ * np.sin(gamma_)
    TC = -mB_ * np.cos(gamma_)
    return TS * np.cos(theta_arr) + TC * np.sin(theta_arr) * np.cos(omega_ * t_arr - phi_arr)

def set_ani_beg(xrange):
    w_ani_beg.value = xrange[0]


#══════════════════════════════════════════════════════════════════════
class PlotBaseTimeSeries:
    def __init__(self, t, lines=0, xrange_callback: Callable|None=None):
        """
        t : 1D array of time values
        w_xrange_widget : optional ipywidget (e.g. FloatRangeSlider) to
                          sync with the x−limits
        """
        self.t = t
        if self.t is not None:
            self.time_range = (self.t[0], self.t[-1])
        self.lines = lines
        self.xrange_callback = xrange_callback
        self.fig = None
        self.ax  = None
        self._orig_xlim = None
        self._is_zoomed = False
        self._guard = False
        self._stats_y = None          # store the last y array for stats
        self.meas1_text = None
        self.meas2_text = None
        self.x_is_time = True

    def create_figure(self):
        self.fig, self.ax = plt.subplots(figsize=self.figsize)
        # hide header/footer, enable toolbar
        self.fig.canvas.header_visible = False
        self.fig.canvas.footer_visible = False
        self.fig.canvas.toolbar_visible = True

    def configure_axes(self):
        # common styling
        if self.x_is_time:
            self.ax.set_xlim(self.t[0], self.t[-1])
            self.ax.set_xlabel('Time (sec)');
        self.ax.tick_params(axis='both', labelcolor='#303030', labelsize=8)
        self.ax.grid(True, which='both', color='lightgray', linestyle='--', linewidth=0.3)
        self.ax.locator_params(axis='x', nbins=20)
        self.ax.locator_params(axis='y', nbins=20)
        self.ax.xaxis.set_minor_locator(AutoMinorLocator(2))
        self.ax.yaxis.set_minor_locator(AutoMinorLocator(2))

        # remember original limits
        self._orig_xlim = self.ax.get_xlim()

    def adjust_layout(self):
        """Hook for subclasses to call fig.subplots_adjust(...)."""
        pass

    def update_plot_range(self, t_range):
        """Hook for subclasses to call update_plot_range()."""
        pass
  
    def set_ylabelr(self, s, x, y):
        #lbl = self.ax.set_ylabel(rf'{s}', y=y)
        lbl = self.ax.set_ylabel(s)
        self.ax.yaxis.set_label_coords(x, y)

    def limit_span(self, lim:tuple):
        return lim[1]-lim[0]

    def _on_xlim_changed(self, ax):
        global time_range_g
        self.set_span_label_px(73,18)
        if self.x_is_time:
            self.time_range = ax.get_xlim()
            if not self._guard:
                time_range_g = self.time_range

            self.span_text.set_text(f"Span={self.limit_span(self.time_range):.5f} s")
            if self.xrange_callback:
                self.xrange_callback(self.time_range)

    def sync_btn_clicked(self, event):
        global time_range_g
        if self.x_is_time:
            self._guard = True
            # toggle between time_range and original or keep zoomed if the time range is changed
            self._is_zoomed = (not self._is_zoomed) or time_range_g != self.time_range

            self.ax.set_xlim(*(time_range_g if self._is_zoomed else self._orig_xlim))
            
            self.fig.canvas.draw_idle()
            self._guard = False
        else:
            self.sync_btn_flip()

    def add_series_stats(self, no:int, y_all: np.ndarray, symbol:str=r'^{\!}'):    # fixed-height + invisible glyph \u200b
        """
        Register the full y array corresponding to self.t so stats can be computed
        and updated when x-limits change. Call this from the subclass draw() after plotting.
        """
        self._stats_y = np.asarray(y_all)
        # initial update
        self._update_stats(no, symbol)
        # force redraw
        try:
            self.fig.canvas.draw_idle()
        except Exception:
            self.fig.canvas.draw()

    def _update_stats(self, no, symbol):
        """
        Compute average and amplitude for the *visible* segment (current xlim).
        Updates avg_text and amp_text.
        """
        if self._stats_y is None:
            # nothing registered
            return

        # get visible x-range in data coordinates
        x0, x1 = self.ax.get_xlim()
        # find indices in self.t
        i0 = np.searchsorted(self.t, x0, 'left')
        i1 = np.searchsorted(self.t, x1, 'right')
        if i1 <= i0:
            return

        y_seg = self._stats_y[i0:i1]

        # compute average (mean) and amplitude (peak = (max-min)/2)
        avg = float(np.mean(y_seg))
        app = np.ptp(y_seg)
        meas_s = f"${symbol}$ avg : {avg:.8g}   pp : {app:.8g}" # hair spaces

        # update text objects
        if no==1:
            if self.meas1_text is not None:
                self.meas1_text.set_text(meas_s)
        elif no==2:
            if self.meas2_text is not None:
                self.meas2_text.set_text(meas_s)

        # redraw canvas
        try:
            self.fig.canvas.draw_idle()
        except Exception:
            self.fig.canvas.draw()
    #-------------------------------------------------
    def ZeroPaddingFFT(self, _time_range, y_all, zero_pad_factor=10):
        t0, t1 = _time_range
        i0 = np.searchsorted(self.t, t0, 'left')
        i1 = np.searchsorted(self.t, t1, 'right')
        t_seg = self.t[i0:i1]
        y_seg = y_all[i0:i1]
        y_seg = y_seg - np.mean(y_seg)

        dt = np.mean(np.diff(t_seg))      # Mean time step
        fs = 1 / dt                   # Sampling frequency
        #dt = (t_seg[1] - t_seg[0]) #*0.01

        N = len(y_seg)
        if N < 3:
            raise ValueError("Not enough points")

        zero_padded = np.pad(y_seg, (0, zero_pad_factor * N))  # 10× zero-padding
        fft_vals = np.abs(rfft(zero_padded))
        freqs = rfftfreq(len(zero_padded), d=dt)

        # Find precise frequency of maximum amplitude
        peak_idx = np.argmax(fft_vals)
        peak_freq = freqs[peak_idx]

        return peak_freq, freqs, 0

    #———————————————————————————————————————————————————————————————————
    def funda_periodogram(self, _time_range, y_all, method='periodogram',
                          window='hann', zero_pad_factor=4):
        t0, t1 = _time_range
        i0 = np.searchsorted(self.t, t0, 'left')
        i1 = np.searchsorted(self.t, t1, 'right')
        t_seg = self.t[i0:i1]
        y_seg = y_all[i0:i1]
        y_seg= y_seg - np.mean(y_seg)

        N = len(y_seg)
        if N < 3:
            raise ValueError("Not enough points")

        # sampling
        dt = t_seg[100] - t_seg[0]
        fs = 100.0 / dt

        # FFT length for zero‐padding
        nfft = zero_pad_factor * N

        # periodogram with zero‑pad
        freqs, Pxx = periodogram(y_seg, fs=fs, window=window, detrend='constant', nfft=nfft)

        # ignore DC
        Pxx[0] = 0

        # pick peak
        idx = np.argmax(Pxx)

        # quadratic peak interpolation
        def parabolic(P, i):
            α, β, γ = P[i-1], P[i], P[i+1]
            return 0.5*(α - γ) / (α - 2*β + γ)
        δ = parabolic(Pxx, idx) if 1 <= idx < len(Pxx)-1 else 0

        # refined fundamental
        df = freqs[1] - freqs[0]
        f0 = freqs[idx] + δ*df

        return f0, freqs, Pxx

    #———————————————————————————————————————————————————————————————————
    def calc_spectrum(self, _time_range: Tuple[float,float], y_range: np.ndarray,
                         n_peaks: int = 10,
                         zero_pad_factor: int = 8,
                         do_inst_freq: bool = True) -> Dict[str,Any]:
        """
        Robust frequency estimator for a visible segment.
        Returns a dict with keys:
          - 'peaks': list of (freq_Hz, amplitude) sorted by amplitude desc
          - 'fundamental': freq_Hz of largest peak
          - 'secondary': freq_Hz of second peak (or None)
          - 'beat_freq': abs diff between first two peaks (Hz) or None
          - 'inst_freqs': (optional) dict of median instantaneous freq per peak (Hz)
          - 'spectrum': (freqs, mags) arrays used (useful for plotting)
          - 'method': 'fft+parabolic' or 'hilbert'
        Notes:
          - Expects self.t in seconds; if time_unit='ms' the function will convert.
          - Requires scipy.signal.
        """

        def _parabolic_refine(mag, k):
            """
            Parabolic interpolation of a peak at index k in magnitude array mag.
            Returns fraction offset (delta) relative to bin k and interpolated mag.
            """
            if k <= 0 or k >= len(mag) - 1:
                return 0.0, mag[k]
            alpha = mag[k-1]
            beta  = mag[k]
            gamma = mag[k+1]
            denom = (alpha - 2*beta + gamma)
            if denom == 0:
                return 0.0, beta
            delta = 0.5 * (alpha - gamma) / denom
            # interpolated magnitude (optional)
            mag_ip = beta - 0.25*(alpha - gamma)*delta
            return delta, mag_ip


        t = self.t
        t0, t1 = _time_range
        # 1) slice indices
        i0 = np.searchsorted(t, t0, side='left')
        i1 = np.searchsorted(t, t1, side='right')
        if i1 - i0 < 6:
            raise ValueError("Not enough points in selected range to estimate frequency")

        t_seg = t[i0:i1]
        y_seg = np.asarray(y_range[i0:i1])

        # basic sampling info
        dt = np.mean(np.diff(t_seg))
        if not np.allclose(np.diff(t_seg), dt, rtol=1e-3, atol=1e-12):
            # non-uniform sampling: resample onto uniform grid (simple)
            # build uniform time grid with same dt
            N = len(t_seg)
            t_uniform = t_seg[0] + np.arange(N) * dt
            y_seg = np.interp(t_uniform, t_seg, y_seg)
            t_seg = t_uniform

        N = len(y_seg)
        if N < 8:
            raise ValueError("Segment too short after resampling")

        # 2) detrend + window
        y_detr = signal.detrend(y_seg - np.mean(y_seg))
        window = signal.windows.hann(N, sym=False)
        y_win = y_detr * window

        # 3) zero-pad FFT for interpolation
        Nfft = int(2**(np.ceil(np.log2(N)))*zero_pad_factor)
        if Nfft <= N:
            Nfft = 2**(int(np.ceil(np.log2(N))) + 3)

        Y = np.fft.rfft(y_win, n=Nfft)
        mags = np.abs(Y)
        freqs = np.fft.rfftfreq(Nfft, dt)

        # remove DC
        mags[0] = 0.0

        # 4) find peaks in spectrum
        # dynamic threshold: prominence relative to max
        max_mag = np.max(mags)
        if max_mag <= 0:
            return {'peaks': [], 'funda': None, 'secon': None, 'beat_freq': None}

        # tune the prominence and distance
        prominence = max_mag * 1e-5   # peaks must be at least 0.005% of the largest peak in magnitude
        # minimum distance in bins to consider separate peaks: at least (Nfft/N) * 0.5 ~ 0.5*zero_pad_factor
        min_bin_dist = max(1, int(0.5 * zero_pad_factor))
        #min_bin_dist = max(1, int(8 * zero_pad_factor))
        peaks_idx, props = signal.find_peaks(mags, prominence=prominence, distance=min_bin_dist)
        if peaks_idx.size == 0:
            # fallback: take global max
            peak = int(np.argmax(mags))
            peaks_idx = np.array([peak])

        # 5) refine peaks by parabolic interpolation
        peak_list = []
        df = freqs[1] - freqs[0]
        for k in peaks_idx:
            delta, mag_ip = _parabolic_refine(mags, k)
            freq_refined = freqs[k] + delta * df
            peak_list.append((freq_refined, mag_ip, k + delta))

        # sort by magnitude descending
        peak_list.sort(key=lambda x: x[1], reverse=True)
        peak_list = peak_list[:n_peaks]
        # build output peaks list (Hz)
        peaks_out = [(f, a) for (f, a, _) in peak_list]

        fundamental = peaks_out[0][0] if len(peaks_out) >= 1 else None
        secondary   = peaks_out[1][0] if len(peaks_out) >= 2 else None
        beat_freq_hz = abs(fundamental - secondary) if (fundamental and secondary) else None

        result = {
            'peaks': peaks_out,
            'funda': fundamental,
            'secon': secondary,
            'beat_freq': beat_freq_hz,
            'spectrum': (freqs, mags),
            'method': 'fft+parabolic'
        }

        # 6) optional: instantaneous frequency estimate around each detected peak via Hilbert
        if do_inst_freq and len(peaks_out) >= 1:
            inst = {}
            fs = 1.0 / dt
            nyq = fs/2.0
            for idx, (f_pk, amp) in enumerate(peaks_out[:2]):  # compute for first two peaks only
                bw = max(0.5, 0.05*f_pk)   # Hz bandwidth, choose at least 0.5 Hz or 5% of freq
                low = max(0.001, (f_pk - bw)/nyq)
                high = min(0.999, (f_pk + bw)/nyq)
                if low >= high:
                    inst[f"peak{idx+1}"] = None
                    continue
                b,a = signal.butter(4, [low, high], btype='bandpass')
                try:
                    y_bp = signal.filtfilt(b, a, y_seg)
                except Exception:
                    y_bp = signal.lfilter(b, a, y_seg)  # fallback
                analytic = signal.hilbert(y_bp)
                inst_phase = np.unwrap(np.angle(analytic))
                inst_freq = (np.diff(inst_phase) / (2*np.pi)) * fs  # in Hz
                # use median to avoid transients
                valid = inst_freq[np.isfinite(inst_freq)]
                if valid.size == 0:
                    inst[f"peak{idx+1}"] = None
                else:
                    inst[f"peak{idx+1}"] = float(np.median(valid))
            result['inst_freqs'] = inst
            result['method'] += '+hilbert'

        return result

    #———————————————————————————————————————————————————————————————————
    def meas_box_clicked(self, artist, event):
        try:
            bbox = artist.get_window_extent(self.fig.canvas.get_renderer())

            x_click = event.x
            x_mid = 0.5 * (bbox.x0 + bbox.x1)

            nums = re.findall(r':\s*([^\s]+)',artist.get_text())
            #with out:
            #    print(nums)
            # choose left or right
            text = nums[0] if x_click < x_mid else nums[1]
            with js_out:
                js_out.clear_output(wait=True)
                display(Javascript(f"navigator.clipboard.writeText({text!r});"))
            #display(Javascript('alert("Copied to clipboard");'))
            #display(Javascript(f"navigator.clipboard.writeText('{text}').then(() => console.log('Copied OK'), err => console.log('Copy failed', err));"))
            #print("Copied to clipboard:", text)
        except Exception as e:
            with out:
                msg = "Clipboard copy failed: " + str(e)
                display(Javascript(f'alert("{msg}");'))

    def on_click_fig(self, event):
        if event.inaxes is None:
            return
        
        for artist in [self.meas1_text, self.meas2_text]:
            if artist is None:
                continue
            contains, _ = artist.contains(event)
            if contains:
                #with out:
                #   print(artist)
                self.meas_box_clicked(artist, event)
                return
                
        if hasattr(self, "fig_clicked"): # derived class method
            self.fig_clicked(event)

    #———————————————————————————————————————————————————————————————————
    def add_button_fixed_px(self, right_px, top_px, w_px, h_px):
        fig_w_px, fig_h_px = self.fig.get_size_inches() * self.fig.dpi
        left = (fig_w_px - right_px - w_px) / fig_w_px
        bottom = (fig_h_px - top_px - h_px) / fig_h_px
        self.btn_ax = self.fig.add_axes([left, bottom, w_px / fig_w_px, h_px / fig_h_px])

    def set_span_label_px(self, left_px, top_px):
        fig_w_px, fig_h_px = self.fig.get_size_inches() * self.fig.dpi
        x = left_px / fig_w_px
        y = (top_px) / fig_h_px  # move downward
        self.span_text.set_position((x, y))        

    def connect_callbacks(self):
        self.ax.callbacks.connect('xlim_changed', self._on_xlim_changed)
        self.add_button_fixed_px(right_px=25, top_px=5, w_px=50, h_px=22 )
        #self.btn_ax = self.fig.add_axes([0.930, 0.939, 0.05, 0.043])  #[left, bottom, width, height]
        self.btn_sync = mButton(self.btn_ax, 'Sync', hovercolor='0.975')
        self.btn_sync.on_clicked(self.sync_btn_clicked)
        self.freq1_text = self.fig.text(0.90, 0.941, "", ha='right', va='center', fontsize=9, color='black')
        self.freq2_text = self.fig.text(0.90, 0.973, "", ha='right', va='center', fontsize=9, color='black')
        #self.span_text  = self.fig.text(0.19, 0.941, "", ha='left',  va='center', fontsize=9, color='black')
        self.span_text = self.fig.text(0, 0, '', fontsize=8, ha='left', va='top', color='black')
        self.ax.set_title(self.ax.get_title(), fontsize=11)
        # create the two small stat text boxes just above the x axis (bottom-center)
        # position uses axes coords: x=0.5 center, y=0.02 = slightly above bottom
        if self.lines:
            if self.meas1_text is None:
                self.meas1_text = self.ax.text(
                    0.53, 0.030, "", transform=self.ax.transAxes,
                    ha='center', va='center', fontsize=8, color='black', zorder=100,
                    bbox=dict(facecolor='#e8f2ff', lw=0.5, edgecolor='#4a90e2', alpha=0.82, boxstyle='round, pad=0.22') #dict(facecolor='white', alpha=0.75, edgecolor='none', pad=2)
                )

            if self.lines==2:
                if self.meas2_text is None:
                    self.meas2_text = self.ax.text(
                        0.79, 0.030, "", transform=self.ax.transAxes,
                        ha='center', va='center', fontsize=8, color='black', zorder=100,
                        bbox=dict(facecolor='#f2ffe6', lw=0.5, edgecolor='#84e64a', alpha=0.82, boxstyle='round, pad=0.22') #dict(facecolor='white', alpha=0.75, edgecolor='none', pad=2)
                    )

        self.cid_click = self.fig.canvas.mpl_connect('button_press_event', self.on_click_fig)
            
    #———————————————————————————————————————————————————————————————————
    def draw(self):
        """
        Subclasses must override this to do the actual plotting:
            - call self.ax.plot() or self.ax.scatter(), etc.
            - set labels, title, legend, etc.
        """
        raise NotImplementedError

    #———————————————————————————————————————————————————————————————————
    def show(self, figsize=(11,5), show=True):
        # store any common size
        self.figsize = figsize
        # create fig/ax
        self.create_figure()
        # subclass draws its data
        self.draw()
        # apply common axis styling
        self.configure_axes()
        # allow subclasses to nudge margins
        self.adjust_layout()
        # wire up callbacks
        self.connect_callbacks()
        # finalize
        if show:
            plt.show()


#———————————————————————————————————————————————————————————————————
class PlotSpectrum:
    def on_click_fig(self, event):
        if event.dblclick is False:
            return  # ignore single clicks entirely
        self.marks = (self.marks + 1) % 3
        for dot, lbl, freq in self.peak_artists:
            if self.marks > 0:
                lbl.set_text(f"{freq:.3f} Hz" if self.marks==1 else f"{freq*2*np.pi:.3f}") # radian figures are displayed without units
            dot.set_visible(self.marks > 0)
            lbl.set_visible(self.marks > 0)

        self.help_lbl.set_visible(False)
        self.fig.canvas.draw_idle()

    def set_title(self):
        self.ax.set_title(f"{self.title}\u2003-\u2003(time window: {self.time_range[0]:.4f} - {self.time_range[1]:.4f} s)", fontsize=11)

    def sync(self, result, _time_range):
        self.time_range = _time_range
        #print('PlotSpectrum.sync: ', _time_range)
        self.plot(result)
        self.fig.canvas.draw_idle()

    def __init__(self, result, time_range, title, varname, figsize, fmin=None, fmax=None, log=True, mark_peaks=True):
        """
        Plot the spectrum contained in result['spectrum'].

        Parameters
        ----------
        result : dict
            The dictionary returned by your FFT function. Must contain:
                result['spectrum'] = (freqs, mags)
                result['peaks'] = [(freq, mag), ...]   # optional
        title : the title of the plot
        varname : the variable of the spectrum
        fmin, fmax : float or None
            Frequency range to display (Hz). If None, show entire range.
        log : bool
            If True, use logarithmic y-scale.
        mark_peaks : bool
            If True, mark detected peaks.
        """
        self.time_range = time_range
        self.title = title
        self.figsize = figsize
        self.fmin = fmin
        self.fmax = fmax
        self.log = log
        self.mark_peaks = mark_peaks
        self.fig = None
        self.ax = None
        self.marks = 1 # 0 = hide, 1=Hz, 2 = rad/s
        self.plot(result)
        if mark_peaks:
            cid = self.fig.canvas.mpl_connect('button_press_event', self.on_click_fig)
        plt.show()
   
    def plot(self, result):
        # unpack spectrum
        freqs, mags = result['spectrum']

        # frequency mask (if limits provided)
        if  self.fmin is not None or self.fmax is not None:
            mask = np.ones_like(freqs, dtype=bool)
            if  self.fmin is not None:
                mask &= freqs >=  self.fmin
            if  self.fmax is not None:
                mask &= freqs <=  self.fmax
            freqs_p = freqs[mask]
            mags_p = mags[mask]
        else:
            freqs_p = freqs
            mags_p = mags

        # plotting
        if  self.fig is None:
            self.fig, self.ax = plt.subplots(figsize= self.figsize)
            self.fig.subplots_adjust(left=0.067, right=0.977, top=0.94, bottom=0.10, hspace=0.1)
            # hide header/footer, enable toolbar
            self.fig.canvas.header_visible = False
            self.fig.canvas.footer_visible = False
            self.fig.canvas.toolbar_visible = True
        else:
            self.ax.cla()

        # common styling
        self.ax.tick_params(axis='both', labelcolor='#303030', labelsize=9)
        self.ax.grid(True, which='both', color='lightgray', linestyle='--', linewidth=0.3)
        self.ax.locator_params(axis='x', nbins=20)
        #self.ax.locator_params(axis='y', nbins=20)
        self.ax.xaxis.set_minor_locator(AutoMinorLocator(2))
        #self.ax.yaxis.set_minor_locator(AutoMinorLocator(2))
        # labels
        self.ax.set_ylabel("Magnitude" + (' (log)' if self.log else ''))
        self.ax.yaxis.set_label_coords(-0.055, 0.45)
        self.ax.set_xlabel("Frequency (Hz)")
        self.help_lbl = self.ax.text(0.006, 0.007, "Double-click to display peak frequencies in Hz or rad/s",
            transform=self.ax.transAxes, ha='left', va='bottom', fontsize=7, color='#444444')

        self.set_title()
        self.ax.set_xlim(self.fmin if self.fmin else 0, self.fmax if self.fmax else 1000)

        if  self.log:
            self.ax.semilogy(freqs_p, mags_p, lw=glw, color='blue', zorder=5)
        else:
            self.ax.plot(freqs_p, mags_p, lw=glw, color='blue', zorder=5)

        self.peak_artists = []
        # mark peaks
        if  self.mark_peaks and 'peaks' in result and result['peaks']:
            for f, a in result['peaks']:
                if  self.fmin is None or f >=  self.fmin:
                    if  self.fmax is None or f <=  self.fmax:
                        if self.log:
                            dot = self.ax.semilogy(f, a, 'r.', zorder=1)[0]
                        else:
                            dot = self.ax.plot(f, a, 'r.', zorder=1)[0]

                        lbl = self.ax.text(f, a, f"{f:.3f} Hz", fontsize=8, color='brown', ha='center', va='bottom')
                        if  self.mark_peaks:
                            self.peak_artists.append((dot, lbl, f))


# ——————————————————————————————————————————————————————————————————————
class PlotTheta(PlotBaseTimeSeries):
    def __init__(self, t, theta_vals, xrange_callback: Callable|None=None):
        super().__init__(t, 1, self.on_xrange_changed)
        self.user_xrange_callback = xrange_callback
        self.theta_vals = theta_vals
        self.om_n1 = 0
        self.ax2 = None
        self.freq_res = None
        self.spectrum_plot = None
        self.ax2_visible = True

    def fig_clicked(self, event):
        if event.dblclick is False:
            return  # ignore single clicks entirely
        if self.ax2 is None:
            return
        self.ax2_visible = not self.ax2_visible
        for line in self.ax2.get_lines():
            line.set_visible(self.ax2_visible)
        self.help_lbl.set_visible(False)
        self.fig.canvas.draw_idle()

    def measure_freq(self, _xrange):
        global omega_n1, omega_n2
        #f0 = self.calc_spectrum(xrange, self.theta_vals)
        #f0, freqs, Pxx = self.ZeroPaddingFFT(xrange, self.theta_vals)
        self.freq_res = self.calc_spectrum(_xrange, self.theta_vals, 20)
        #f0, freqs, Pxx = self.funda_periodogram(_xrange, self.theta_vals)
        f1 = self.freq_res['funda']
        f2 = self.freq_res['secon']
        if f1:
            omega_n1 = f1*2*np.pi
            self.freq1_text.set_text(f"f₁ = {f1:8.4f} Hz  {omega_n1:8.4f} rad/s".replace(' ','\u2007')) # hair and thin spaces are used
        if f2:
            omega_n2 = f2*2*np.pi
            self.freq2_text.set_text(f"f₂ = {f2:8.4f} Hz  {omega_n2:8.4f} rad/s".replace(' ','\u2007')) # hair and thin spaces are used

        if f1 and f2 and min(f1,f2)*100 > max(f1,f2):
            self.om_n1 = abs(omega_n1 - omega_n2)/2
            # should be less than 50 cycles for visualization
            if self.om_n1*(self.t[-1]-self.t[0])<314: 
                top = np.max(self.theta_vals)
                th_center = (top + min(self.theta_vals))/2
    
                dif_mid = abs(self.theta_vals[0] - th_center)
                dif_top = top - th_center - dif_mid
    
                if self.t[0]==0 and dif_mid < dif_top:
                    sin_ref = np.sin(self.om_n1 * self.t)
                else:
                    sin_ref = np.cos(self.om_n1 * self.t)
    
                #print(f"center: {th_center:.5f}")
                #print(f"vals[0]: {self.theta_vals[0]:.5f}")
                #print(f"dif mid: {dif_mid:.5f}")
                #print(f"dif top: {dif_top:.5f}")
    
                if self.ax2 == None:
                    self.ax2 = self.ax1.twinx()
                    self.ax2.plot(self.t, sin_ref, color='saddlebrown', lw=0.4, label=f'{abs(f2-f1):.4f} Hz', zorder=4)
                    self.ax2.legend(loc='lower right', fontsize=8)
                    #self.ax2.set_ylabel(r'$beat$', color='saddlebrown')
                    self.ax2.set_yticks([])
                    self.help_lbl = self.ax.text(0.006, 0.007, "Double-click to turn on and off the beat envelop",
                        transform=self.ax.transAxes, ha='left', va='bottom', fontsize=7, color='#444444')

    def set_callback_spectrum(self, spectrum_plot):
        self.spectrum_plot = spectrum_plot

    def on_xrange_changed(self, xrange):
        #self.time_range = xrange
        self.measure_freq(xrange)
        self.add_series_stats(1, self.theta_vals)
        self.fig.canvas.draw_idle()
        if self.user_xrange_callback:
            self.user_xrange_callback(xrange)
        #print('on_xrange_changed')
        #with out:
        #    print('sanino on_xrange_changed')

        if self.spectrum_plot:
            #print('going to call self.spectrum_plot.sync with params ', xrange)
            self.spectrum_plot.sync(self.freq_res, xrange)

    def adjust_layout(self):
        self.fig.tight_layout()
        #self.fig.subplots_adjust(left=0.075, right=0.98, top=0.94, bottom=0.10, hspace=0.1) #!!!!!!

    def draw(self):
        # Primary axis
        self.ax1 = self.ax
        self.ax1.plot( self.t, self.theta_vals, color='blue', lw=glw, label=r'$\theta$', zorder=5)
        self.set_ylabelr(r'$\theta\ (\mathrm{rad})$', x=-0.055, y=0.46)
        #self.ax1.set_ylabel(r'$\theta\ (\mathrm{rad})$', y=0.46)

        self.ax1.tick_params(axis='y', labelcolor='blue')
        self.ax1.locator_params(axis='y', nbins=20)

        # Secondary axis, if requested
        if self.om_n1:
            sin_ref = np.cos(float(self.om_n1) * self.t)
            self.ax2 = self.ax1.twinx()
            self.ax2.plot(self.t, sin_ref, color='saddlebrown', lw=0.3, label=r'$beat$', zorder=1)
            self.ax2.legend(loc='lower right', fontsize=8)

            #ax2.set_ylabel(r'$beat$', color='saddlebrown')
            #self.ax2.tick_params(axis='y', labelcolor='saddlebrown')
            self.ax2.set_yticks([])
            self.add_series_stats(1, self.theta_vals)
            with out:
                print('muharrem')

        # common labels & title
        self.ax1.set_title(r'$\theta(t)$', x=0.52)
        self.ax1.legend(loc='upper right', fontsize=8)

    def show(self, figsize=(11,5)):
        super().show(figsize=figsize)
        self.measure_freq(self.time_range)
        # display average and peak to peak
        self.add_series_stats(1, self.theta_vals)
        #with out:
        #    print('sanino show')


# ——————————————————————————————————————————————————————————————————————
class PlotPhiPhase(PlotBaseTimeSeries):
    def __init__(self, t, phi_phases, xrange_callback: Callable|None=None):
        super().__init__(t, 1, self.on_xrange_changed)
        self.phi_phases = phi_phases
        self.user_xrange_callback = xrange_callback

    #def set_phase_ylim(self, margin=0.01):
    #    med = np.median(self.phi_phases)
    #    if med < np.pi*0.5:
    #        ymax = np.max(np.abs(self.phi_phases)) + margin
    #        self.ax.set_ylim(-ymax, ymax)
    #    else:
    #        # keep conventional phase view
    #        self.ax.set_ylim(-180, 180)    

    def adjust_layout(self):
        self.fig.tight_layout()
        #self.fig.subplots_adjust(left=0.075, right=0.98, top=0.94, bottom=0.10, hspace=0.1)

    def set_avg_pp(self):
        self.add_series_stats(1, self.phi_phases, r'\boldsymbol{\phi_0}')

    def on_xrange_changed(self, xrange):
        #self.time_range = xrange
        self.set_avg_pp()
        self.fig.canvas.draw_idle()
        if self.user_xrange_callback:
            self.user_xrange_callback(xrange)

    def draw(self):
        #self.set_phase_ylim()
        self.ax.plot(self.t, self.phi_phases, linewidth=0.6, color='blue', label=r'$\phi - \omega t (rad)$')
        self.set_ylabelr('Phase (deg)', x=-0.055, y=0.46);
        self.ax.set_title(r'Phase of $\phi$ with respect to $B$ field', x=0.52);

    def show(self, figsize=(11,5)):
        super().show(figsize=figsize)
        self.set_avg_pp()


# ——————————————————————————————————————————————————————————————————————
class Plot_phd_psd(PlotBaseTimeSeries):
    def __init__(self, t, phd_vals, psd_vals, xrange_callback: Callable|None=None):
        super().__init__(t, 2, self.on_xrange_changed)
        self.phd_vals = phd_vals
        self.psd_vals = psd_vals
        self.user_xrange_callback = xrange_callback

    def adjust_layout(self):
        self.fig.tight_layout()
        #self.fig.subplots_adjust(left=0.075, right=0.98, top=0.94, bottom=0.10, hspace=0.1)

    def set_avg_pp(self):
        self.add_series_stats(1,self.phd_vals, r'\boldsymbol{\dot\phi}')
        self.add_series_stats(2,self.psd_vals, r'\boldsymbol{\dot\psi}')

    def on_xrange_changed(self, xrange):
        #self.time_range = xrange
        self.set_avg_pp()
        self.fig.canvas.draw_idle()
        if self.user_xrange_callback:
            self.user_xrange_callback(xrange)

    def draw(self):
        self.ax.plot(self.t,  self.phd_vals, linewidth=0.6, color='blue', label=r'$\dot\phi$')
        self.ax.plot(self.t, -self.psd_vals, linewidth=0.6, color='green', label=r'$-\dot\psi$')
        bottom, top = self.ax.get_ylim()
        # Clamp both ends to [-5000, 5000]
        self.ax.set_ylim(max(bottom, -5000), min(top, 5000))
        self.ax.set_xlabel('Time (sec)');
        self.set_ylabelr(r'Velocity (rad/s)', x=-0.055, y=0.46);
        self.ax.set_title(r'$\dot{\phi}$, $\dot{\psi}$ (Angular velocity of $\phi$ and $\psi$)', x=0.52);
        self.ax.legend(fontsize=8);

    def show(self, figsize=(11,5)):
        super().show(figsize=figsize)
        self.set_avg_pp()


# ——————————————————————————————————————————————————————————————————————
class PlotNu_z(PlotBaseTimeSeries):
    def __init__(self, t, vz_vals_lab, vz_vals_bod, xrange_callback: Callable|None=None):
        super().__init__(t, 2, self.on_xrange_changed)
        self.vz_vals_lab = vz_vals_lab
        self.vz_vals_bod = vz_vals_bod
        self.user_xrange_callback = xrange_callback

    def adjust_layout(self):
        self.fig.tight_layout()

    def set_avg_pp(self):
        self.add_series_stats(1, self.vz_vals_lab, r'\boldsymbol{\dot{\nu}_{z\;LA}}')
        self.add_series_stats(2, self.vz_vals_bod, r'\boldsymbol{\dot{\nu}_{z\;BO}}')

    def on_xrange_changed(self, xrange):
        self.set_avg_pp()
        self.fig.canvas.draw_idle()
        if self.user_xrange_callback:
            self.user_xrange_callback(xrange)

    def draw(self):
        self.ax.plot(self.t, self.vz_vals_lab, linewidth=0.6, color='mediumblue', label=r'$^{LAB}\nu_z$')
        self.ax.plot(self.t, self.vz_vals_bod, linewidth=0.6, color='green',  label=r'$^{BODY}\nu_z$')
        self.set_ylabelr('Velocity (rad/s)', x=-0.055, y=0.46);
        self.ax.set_title(r'Angular velocity Z comp. $(\nu_z)$ in lab and in body frames', x=0.52);
        self.ax.legend(fontsize=8);

    def show(self, figsize=(11,5)):
        super().show(figsize=figsize)
        self.set_avg_pp()
        self.fig.canvas.draw_idle()


# ——————————————————————————————————————————————————————————————————————
class PlotNu_XY(PlotBaseTimeSeries):
    def __init__(self, t, vx_vals, vy_vals, xrange_callback: Callable|None=None):
        #super().__init__(t, xrange_callback)
        super().__init__(t, 2, self.on_xrange_changed)
        self.user_xrange_callback = xrange_callback
        self.vx_vals = vx_vals
        self.vy_vals = vy_vals
        self.user_xrange_callback = xrange_callback

    def adjust_layout(self):
        self.fig.tight_layout()

    def set_avg_pp(self):
        self.add_series_stats(1, self.vx_vals, r'\boldsymbol{\dot{\nu}_{x}}')
        self.add_series_stats(2, self.vy_vals, r'\boldsymbol{\dot{\nu}_{y}}')

    def on_xrange_changed(self, xrange):
        self.set_avg_pp()
        #self.measure_freq(xrange)
        self.fig.canvas.draw_idle()
        if self.user_xrange_callback:
            self.user_xrange_callback(xrange)

    def draw(self):
        self.ax.plot(self.t, self.vx_vals, linewidth=0.6, color='firebrick', label=r'$\nu_x$')
        self.ax.plot(self.t, self.vy_vals, linewidth=0.6, color='mediumblue', label=r'$\nu_y$')
        self.set_ylabelr('Velocity (rad/s)', x=-0.055, y=0.46);
        self.ax.set_title(r'Angular velocity X, Y comp. $(\nu_x, \nu_y)$ in body frame ', x=0.52);
        self.ax.legend(fontsize=8);

    def measure_freq(self, _xrange):
        global omega_zero_nu_x
        #f0 = self.calc_spectrum(xrange, self.theta_vals)
        #f0, freqs, Pxx = self.ZeroPaddingFFT(xrange, self.theta_vals)
        f0, freqs, Pxx = self.funda_periodogram(_xrange, self.vx_vals)
        omega_zero_nu_x = f0*2*np.pi
        self.freq1_text.set_text(f"f₀ = {f0:.6g} Hz  {omega_zero_nu_x:.6g} rad/s")

    def show(self, figsize=(11,5)):
        super().show(figsize=figsize)
        self.set_avg_pp()
        self.measure_freq(self.time_range)


# ——————————————————————————————————————————————————————————————————————
class PlotDissipation(PlotBaseTimeSeries):
    def __init__(self, t, th_vals, ph_vals, ps_vals, th_dots, ph_dots, ps_dots, xi, xrange_callback: Callable|None=None):
        super().__init__(t, 1, self.on_xrange_changed)
        self.user_xrange_callback = xrange_callback

        omega_body = np.zeros((len(t), 3))
        dissipation = np.zeros(len(t))

        for i, (th, ph, ps, dth, dph, dps) in enumerate(zip(
                th_vals, ph_vals, ps_vals, th_dots, ph_dots, ps_dots)):
            # ZXZ→body omega components
            A = np.array([
                [ np.cos(ps),      np.sin(th)*np.sin(ps),  0       ],
                [-np.sin(ps),      np.sin(th)*np.cos(ps),  0       ],
                [ 0,               np.cos(th),             1       ]
            ])
            omega_vec = A.dot([dth, dph, dps])
            omega_body[i] = omega_vec
            dissipation[i] = xi * np.dot(omega_vec, omega_vec)

        self.dissipation = dissipation

    def set_avg_pp(self):
        self.add_series_stats(1, self.dissipation, r'\boldsymbol{D}')

    def adjust_layout(self):
        self.fig.tight_layout()

    def on_xrange_changed(self, xrange):
        self.set_avg_pp()
        #self.measure_freq(xrange)
        self.fig.canvas.draw_idle()
        if self.user_xrange_callback:
            self.user_xrange_callback(xrange)

    def draw(self):
        self.ax.plot(self.t, self.dissipation, linewidth=0.6, color='mediumblue', label=r'$D$')
        self.set_ylabelr('Dissipation Power $P(t)$', x=-0.055, y=0.46)
        self.ax.set_title('Instantaneous Viscous Dissipation');
 
    def show(self, figsize=(11,5)):
        super().show(figsize=figsize)
        self.set_avg_pp()


# ——————————————————————————————————————————————————————————————————————
class PlotKinetEn(PlotBaseTimeSeries):
    """
    Given simulation outputs, compute and plot the instantaneous kinetic energy using angular velocity vector and moment of inertia tensor.
    """
    def __init__(self, t, th_vals, ph_vals, ps_vals, th_dots, ph_dots, ps_dots, I1, I2, I3, xrange_callback: Callable|None=None):
        global T_vals
        super().__init__(t, 1, self.on_xrange_changed)
        if T_vals is None:
            T_vals = compute_T(I1, I2, I3, th_vals, ph_vals, ps_vals, ph_dots, th_dots, ps_dots)
        self.T_vals = T_vals
        self.user_xrange_callback = xrange_callback

    def adjust_layout(self):
        self.fig.tight_layout()
        #self.fig.subplots_adjust(left=0.075, right=0.98, top=0.94, bottom=0.10, hspace=0.1)

    def set_avg_pp(self):
        self.add_series_stats(1, self.T_vals, r'\boldsymbol{T}')

    def on_xrange_changed(self, xrange):
        self.set_avg_pp()
        self.fig.canvas.draw_idle()
        if self.user_xrange_callback:
            self.user_xrange_callback(xrange)

    def draw(self):
        #self.ax.yaxis.set_major_formatter(FormatStrFormatter('%.3e'))
        self.ax.ticklabel_format(axis='y', style='sci', scilimits=(-3, 3), useMathText=True)
        self.ax.plot(self.t, self.T_vals, linewidth=0.6, color='darkred')
        self.set_ylabelr(r'Kinetic Energy $(J)$', x=-0.055, y=0.46)
        self.ax.set_title('Instantaneous Kinetic Energy');

    def show(self, figsize=(11,5)):
        super().show(figsize=figsize)
        self.set_avg_pp()


# ——————————————————————————————————————————————————————————————————————
class PlotPotenEn(PlotBaseTimeSeries):
    """
    Given simulation outputs, compute and plot the instantaneous potential energy using magnetic moment and field vectors.
    """
    def __init__(self, t, th_vals, ph_vals, omega_, mB_, gamma_, xrange_callback: Callable|None=None):
        global V_vals
        super().__init__(t, 1, self.on_xrange_changed)
        if V_vals is None:
            V_vals = compute_V(omega_, mB_, gamma_, t, th_vals, ph_vals)
        self.V_vals = V_vals
        self.user_xrange_callback = xrange_callback

    def adjust_layout(self):
        self.fig.tight_layout()
        #self.fig.subplots_adjust(left=0.075, right=0.98, top=0.94, bottom=0.10, hspace=0.1)

    def set_avg_pp(self):
        self.add_series_stats(1, self.V_vals, r'\boldsymbol{V}')
    
    def on_xrange_changed(self, xrange):
        self.set_avg_pp()
        self.fig.canvas.draw_idle()
        if self.user_xrange_callback:
            self.user_xrange_callback(xrange)
    
    def draw(self):
        self.ax.ticklabel_format(axis='y', style='sci', scilimits=(-3, 3), useMathText=True)
        self.ax.plot(self.t, self.V_vals, linewidth=0.6, color='darkred')
        self.set_ylabelr(r'Potential Energy $(J)$', x=-0.055, y=0.46)
        self.ax.set_title('Instantaneous Magnetic Potential Energy', x=0.52);

    def show(self, figsize=(11,5)):
        super().show(figsize=figsize)
        self.set_avg_pp()


# ——————————————————————————————————————————————————————————————————————
class PlotTotalEn(PlotBaseTimeSeries):
    def __init__(self, t, H_vals, xrange_callback: Callable|None=None):
        super().__init__(t, 1, xrange_callback)
        self.H_vals = H_vals  #total energy
        self.user_xrange_callback = xrange_callback

    def adjust_layout(self):
        self.fig.tight_layout()
        #self.fig.subplots_adjust(left=0.075, right=0.98, top=0.94, bottom=0.10, hspace=0.1)

    def set_avg_pp(self):
        self.add_series_stats(1, self.H_vals, r'\boldsymbol{V}')
    
    def on_xrange_changed(self, xrange):
        self.set_avg_pp()
        self.fig.canvas.draw_idle()
        if self.user_xrange_callback:
            self.user_xrange_callback(xrange)
    
    def draw(self):
        self.ax.ticklabel_format(axis='y', style='sci', scilimits=(-3, 3), useMathText=True)
        self.ax.plot(self.t, self.H_vals, linewidth=0.6, color='darkred')
        self.set_ylabelr(r'Total Energy $(J)$', x=-0.055, y=0.46)
        self.ax.set_title('Instantaneous Total Energy', x=0.52);

    def show(self, figsize=(11,5)):
        super().show(figsize=figsize)
        self.set_avg_pp()


# ——————————————————————————————————————————————————————————————————————
class PlotPhasePortrait(PlotBaseTimeSeries):
    def __init__(self, t_vals, x_vals, y_vals, x_label, y_label, params=None):
        self.t_vals = t_vals
        self.x_vals = x_vals
        self.y_vals = y_vals
        self.xlabel = x_label
        self.ylabel = y_label
        self.lc = None
        self.arrow_artists = None
        if params:
            self.set_params(params)
        else:
            self.arrow_step = 50
            self.color_rate = 1.0
            self.arrow_flag = True
            self.color_flag = True      # colored by time
            self.base_color = '#C00000'     # fallback fixed color
 
        super().__init__(None, 0, None)
        self.x_is_time = False
    
    def adjust_layout(self):
        self.fig.subplots_adjust(left=0.12, right=0.97, top=0.95, bottom=0.070 )
        #self.fig.tight_layout()

    def set_params(self, params):
        try:
            self.arrow_flag, self.arrow_step, self.color_flag, self.color_rate, self.base_color = params
        except:
            pass

    def get_params(self):
        return self.arrow_flag, self.arrow_step, self.color_flag, self.color_rate, self.base_color
        
    # ---------------------------------------------
    def _update_phase_collection(self, xs, ys, ts):
        # remove previous collection
        if self.lc is not None and self.lc.axes is not None:
            self.lc.remove()

        if self.arrow_artists:
            for a in self.arrow_artists:
                a.remove()
                
        pts = np.column_stack([xs, ys])
        segs = np.stack([pts[:-1], pts[1:]], axis=1)

        # ---------- LineCollection ----------
        if self.color_flag:
            self.lc = LineCollection(segs, cmap='turbo', linewidths=0.6, alpha=0.9)
            # Cyclic colors (wrap-around)
            
            #N = len(ts)
            #color_vals = np.arange(N // (20/max(self.color_rate, 1e-2))) 
            #period = (ts[-1] - ts[0]) / max(self.color_rate, 1e-6)
            T = (ts[-1] - ts[0])/max(self.color_rate, 1e-1)
            color_vals = ((ts - ts[0]) / T) % 1.000000000001
            
            self.lc.set_array(color_vals)
            self.lc.set_clim(0.0, 1.0)
                
            #period = (ts[-1] - ts[0]) / max(self.color_rate, 1e-6)
            #color_vals = (ts - ts[0]) % period
            #self.lc.set_array(color_vals)
            #self.lc.set_clim(0.0, period)
            # ---- Arrowheads (time direction, color-matched) ----
            norm = self.lc.norm
            cmap = self.lc.cmap
        else:
            self.lc = LineCollection(segs, colors=self.base_color, linewidths=0.6, alpha=0.9)
            norm = cmap = None

        self.ax.add_collection(self.lc)
            
        # ---- Arrowheads (time direction) ----
        self.arrow_artists = []
        if self.arrow_flag:
            for i in range(0, len(xs) - 1, self.arrow_step):
                c = cmap(norm(color_vals[i])) if self.color_flag else self.base_color
                a = self.ax.annotate('',
                    xy=(xs[i+1], ys[i+1]),
                    xytext=(xs[i], ys[i]),
                    arrowprops=dict( arrowstyle='->', color=c, lw=0.8, shrinkA=0, shrinkB=0),
                    zorder=5
                )
                self.arrow_artists.append(a)
    
    # ---------------------------------------------
    def update_plot_range(self, t_range=None):
        if self._is_zoomed:
            if t_range is not None:
                self.time_range = t_range
            t0, t1 = self.time_range
            i0 = np.searchsorted(self.t_vals, t0, side='left')
            i1 = np.searchsorted(self.t_vals, t1, side='right')
            if i1 - i0 < 2:
                return
            xs = self.x_vals[i0:i1]
            ys = self.y_vals[i0:i1]
            ts = self.t_vals[i0:i1-1]   # length = len(xs)-1
        else:
            self.time_range = (self.t_vals[0], self.t_vals[-1])
            i0 = 0
            i1 = len(self.x_vals)
            xs = self.x_vals
            ys = self.y_vals
            ts = self.t_vals[:-1]        # full-data segments

        self._update_phase_collection(xs, ys, ts)
        
        #self.ax.relim()
        #self.ax.autoscale_view(tight=True)
        self.set_ax_limits(xs, ys)
        t0, t1 = self.time_range
        self.span_text.set_text(f"Span={t1-t0:.5f} s ({t0:.4f} – {t1:.4f})")
        self.fig.canvas.draw_idle()

    def sync_btn_flip(self):
        global time_range_g
        self._is_zoomed = (not self._is_zoomed) or time_range_g != self.time_range
        self.update_plot_range(time_range_g)
        self_safe_guard = True
        self.time_slider.value = self.time_range
        self_safe_guard = False
    
    def set_ax_limits(self, xs, ys, margin=0.02):
        x_max = xs.max()
        x_min = xs.min()
        x_mar = (x_max-x_min)*margin
        y_max = ys.max()
        y_min = ys.min()
        y_mar = (y_max-y_min)*margin
        
        self.ax.set_xlim(x_min-x_mar, x_max+x_mar)
        self.ax.set_ylim(y_min-y_mar, y_max+y_mar)
        
    # -------------------------------------------------
    def draw(self):
        self._update_phase_collection(self.x_vals, self.y_vals,self.t_vals[:-1])
        # set limits with a margin 2%
        self.set_ax_limits(self.x_vals, self.y_vals)

        #self.ax.autoscale(enable=True, tight=True)
        #self.ax.margins(0.02)
        self.ax.set_xlabel(self.xlabel)
        self.set_ylabelr(self.ylabel, x=-0.055, y=0.46)
        self.ax.ticklabel_format(axis='y', style='sci', scilimits=(-3, 3), useMathText=True)
        self.ax.set_title(f'Phase Portrait ({self.xlabel}, {self.ylabel})', x=0.52);

    # -------------------------------------------------
    def update(self, t_vals, x_vals, y_vals, x_label, y_label):
        self.t_vals = t_vals
        self.x_vals = x_vals
        self.y_vals = y_vals
        self.xlabel = x_label
        self.ylabel = y_label
        #self._is_zoomed = False
        self._guard = False
        if self._is_zoomed:
            #print('update',self.time_slider.value)
            self.update_plot_range(self.time_slider.value)        
            self.ax.set_xlabel(self.xlabel)
            self.set_ylabelr(self.ylabel, x=-0.055, y=0.46)
            self.ax.ticklabel_format(axis='y', style='sci', scilimits=(-3, 3), useMathText=True)
            self.ax.set_title(f'Phase Portrait ({self.xlabel}, {self.ylabel})', x=0.52);
            
        else:
            #self.lc = None
            #self.arrow_artists = None
            #self._update_phase_collection(self.x_vals, self.y_vals,self.t_vals[:-1])
            self.draw()

    def setup_ui(self):
    # example controls (replace/add your real widgets)
        self.btn_reset = Button(description='Reset Parameters', layout=Layout(width='150px',margin='10px 0 0 0'))
        self.cb_arrows = Checkbox(value=self.arrow_flag, description='Time Arrows', style={'description_width': '0px'}, layout=Layout(width='140px', margin='10px 0 0 0', padding='0 0'))
        self.cb_color  = Checkbox(value= self.color_flag, description='Varying Colors', style={'description_width': '0px'}, layout=Layout(width='140px', margin='0 0 0 0px', padding='0 0'))
        self.arrow_step_box = BoundedIntText(value=self.arrow_step, min=2,  max=10000, step=1, description='Arrow step', layout=Layout(width='146px'), style={'description_width': '80px'} )
        
        self.color_rate_slider = FloatSlider(
            value=self.color_rate,
            min=0.50,
            max=100.0,
            step=0.01,
            readout=True,
            readout_format='.2f',
            continuous_update=True,
            layout=Layout(width='152px', margin='0', padding='0'),
            style={'description_width': '0px'}
        )
  
        # wire up their callbacks to methods if needed, e.g.:
        # btn_auto.on_click(lambda _: self.auto_zoom())
        # cb_arrows.observe(lambda ch: self.toggle_arrows(ch['new']), names='value')
    
        self.time_slider = FloatRangeSlider(
            value=(self.t_vals[0], self.t_vals[-1]),
            min=self.t_vals[0],
            max=self.t_vals[-1],
            step=(self.t_vals[-1] - self.t_vals[0]) / 2000,
            orientation='vertical',
            readout=True,
            readout_format='.3f',
            layout=Layout(
                #min_height='536px',
                #align_self='stretch', 
                width='90px', min_width='90px', margin='4px 0 0 4px')
        )

        self.color_picker = ColorPicker(
            concise=True,
            description='Base color',
            value=self.base_color,
            layout=Layout(width='140px', margin='4px 0 0 4px')
        )
        
        self.cb_arrows.observe(self.on_arrow_toggle, names='value')
        self.arrow_step_box.observe(self.on_arrow_step_change, names='value')
        self.btn_reset.on_click(self.on_params_reset)
        self.cb_arrows.value = self.arrow_flag
        self.cb_color.observe(self.on_color_toggle, names='value')
        self.cb_color.value = self.color_flag

        self.color_rate_slider.observe(self.on_color_rate_change, names='value')
        self.time_slider.observe(self.on_time_slider_change, names='value')
        self.color_picker.observe(self.on_base_color_changed, names='value')
    #------------------------------------------------
    def on_arrow_step_change(self, change):
        if not self._guard:
            self.arrow_step = change['new']
            self.update_plot_range()   # or just redraw arrows

    def on_arrow_toggle(self, change):
        if self._guard:
            return
        self.arrow_flag = change['new']
        self.arrow_step_box.disabled = not self.arrow_flag
        self.update_plot_range()

    def on_color_toggle(self, change):
        if self._guard:
            return
        self.color_flag = change['new']
        self.color_picker.disabled = self.color_flag
        self.color_rate_slider.disabled = not self.color_flag
        self.update_plot_range()

    def on_color_rate_change(self, change):
        if self._guard:
            return
        self.color_rate = change['new']
        self.update_plot_range()

    def on_base_color_changed(self, change):
        if self._guard:
            return
        self.base_color = change['new']

        if not self.color_flag:   # monochrome mode
            self.update_plot_range()
    
    def on_params_reset(self, _=None):
        self._guard = True
        self.cb_arrows.value = self.arrow_flag = True
        self.cb_color.value =self.color_flag = True
        self.arrow_step_box.value = self.arrow_step = 50
        self.color_rate_slider.value = self.color_rate = 1.0
        self.color_picker.value = self.base_color = '#C00000'
        self._guard = False
        self.update_plot_range()
    
    def on_time_slider_change(self, change):
        if self._guard:
            return
        t_range = change['new']
        #print(t_range)
        self._is_zoomed = True
        self.update_plot_range(t_range)
    
    def show(self, figsize=(6.5, 6.5)):
        super().show(figsize=figsize, show=False)
        with io.capture_output():
            plt.show()
            
        self.setup_ui()

        fig_w_px, fig_h_px = (self.fig.get_size_inches() * self.fig.dpi).astype(int)
        
        self.fig.canvas.layout.height = f'{fig_h_px}px'
        self.fig.canvas.layout = Layout(flex='1 1 auto', min_width='0px' )             # 🔑 allows shrinking
        
        self.time_slider.layout.height = f"{fig_h_px}px"

        controls = VBox(
            [phase_portr_dd,
             self.cb_arrows,
             self.arrow_step_box,
             self.cb_color, 
             self.color_picker,
             iHTML("<b>Color rate</b>", layout=Layout(margin='4px 0 0 4px')),
             self.color_rate_slider,
             self.btn_reset],
            layout=Layout(width='164px', justify_content='flex-start', align_items='flex-start', margin='3px 0 0 2px')
        )
        
        plot_and_slider = HBox([self.fig.canvas, self.time_slider],
            layout=Layout(align_items='stretch', gap='2px', margin='0', overflow='hidden'))
        container = HBox([controls, plot_and_slider], layout=Layout(align_items='flex-start') )
       
        #display(container)
        return container
        
    # -------------------------------------------------
    
    #def show(self, figsize=(7,7), show=False):
    #   super().show(figsize=figsize)
    #   display(HBox([self.fig.canvas], layout=Layout(width='1107px', align_items='center', margin='0 0 10px 0')))

# ——————————————————————————————————————————————————————————————————————
# global data
polar_slider_sel = {
    "th": None,
    "ph": None,
    "maxy": 0
}

def plot_sphere2(t_vals, theta_vals, phi_vals, px_size=800):
    
    def spherical_xyz():
        nonlocal r, theta, phi, fw
        with fw.batch_update():
            fw.data[0].x = r * np.sin(theta) * np.cos(phi)
            fw.data[0].y = r * np.sin(theta) * np.sin(phi)
            fw.data[0].z = r * np.cos(theta)

    def refresh_sphere_on_click(x, y, z):
        nonlocal theta, phi
        if not polar_slider_sel["maxy"]:
            return
        theta = polar_slider_sel["th"]
        phi   = polar_slider_sel["ph"]
        spherical_xyz()
  
    # Plotly figure
    fig = go.Figure(
        data=[go.Scatter3d(x=[], y=[], z=[], mode='lines', line=dict(color='blue', width=1))],
        layout=go.Layout(
            title=dict(
                text=r"Spherical Plot of (𝜃, 𝜑)",
                font=dict(size=16, color='black'),
                x=0.5, y=0.99,
                xanchor='center', yanchor='top'),
            width=px_size,
            height=px_size,
            margin=dict(l=0, r=0, b=0, t=0, pad=0),
            scene=dict(
                xaxis=dict(visible=False),
                yaxis=dict(visible=False),
                zaxis=dict(visible=False),
                aspectmode="data"
            ),
            showlegend=False
        )
    )

    fig.update_layout(
        updatemenus=[
            dict(
                type="buttons",
                direction="right",
                showactive=True,
                x=0.005,
                y=1.0,            # 🔑 keep ≤ 1
                xanchor="left",
                yanchor="top",
                pad=dict(r=4, t=4, b=0, l=0),
                #bordercolor="#777",
                #borderwidth=1,
                font=dict(size=16, color='black'),
                #font=dict(size=18),
                bgcolor="rgba(180,180,180,0.75)",
                buttons=[
                    dict(
                        label="⧉",
                        method="relayout",
                        args=["scene.camera.projection.type", "orthographic"]
                    ),
                    dict(
                        label="◯",
                        method="relayout",
                        args=["scene.camera.projection.type", "perspective"]
                    ),
                ]
            )
        ]
    )
    
    fw = go.FigureWidget(fig)
    
    '''
    fw = go.FigureWidget(
        fig,
        config={
            "modeBarButtonsToAdd": [
                {
                    "name": "⧉",
                    "title": "Orthographic projection",
                    "click": """
                        function(gd) {
                            Plotly.relayout(gd, {
                                'scene.camera.projection.type': 'orthographic'
                            });
                        }
                    """
                },
                {
                    "name": "◯",
                    "title": "Perspective projection",
                    "click": """
                        function(gd) {
                            Plotly.relayout(gd, {
                                'scene.camera.projection.type': 'perspective'
                            });
                        }
                    """
                }
            ]
        }
    )
    '''
    def right_row(widget):
        return HBox([widget], layout=Layout(justify_content='flex-end', width='174px', height='40px', margin='4px 0 4px 0'))

    color_picker = ColorPicker(
        description='Color',
        value="blue",
        concise=True,
        #layout=Layout(width='158px', margin='0')
        layout=Layout(width='100%', min_width='0px', margin='0')
    )

    dpi_dd = Dropdown(
        options=[
            ("100", 100),
            ("150", 150),
            ("200", 200),
            ("300", 300),
            ("600", 600),
        ],
        value=300,
        description='DPI',
        layout=Layout(width='142px', margin='0')
    )
    
    btn_save = Button(
        description='Save PNG',
        icon='save',
        layout=Layout(width='90px', margin='0')
    )    

    def on_color_changed(change):
        base_color = change['new']
        fw.data[0].line.color = base_color

    def on_save_clicked(_):
        px = 8 * dpi_dd.value
        fig_static = go.Figure(fw.to_dict())
        fig_static.write_image(f"spherical_plot_{id_hash}.png", width=px, height=px)
        
        #fw.write_image(
        #    f"spherical_plot_{id_hash}.png",
        #    format="png",
        #    width=800,
        #    height=800,
        #    scale=dpi_dd.value
        #)        
    
    color_picker.observe(on_color_changed, names='value')
    btn_save.on_click(on_save_clicked)
    
    r = 2.0  # constant radius
    theta = theta_vals
    phi = phi_vals
    spherical_xyz()

    fw.data[0].on_click(refresh_sphere_on_click)
    # center with HBox — this works inside Output/Accordion/etc.


    controls = VBox(
        [
            right_row(color_picker),
            right_row(dpi_dd),
            right_row(btn_save),
        ],
        layout=Layout(
            align_items='flex-start',   # left align all children
            padding='0 6px 0 0',
            margin='0 0 0 -196px'
        )
    )

    '''
    controls = VBox(
        [
            color_picker,
            dpi_dd,
            btn_save
            #right_row(color_picker),
            #right_row(dpi_dd),
            #right_row(btn_save)
        ],
        layout=Layout(
            #width='210px',
            width='170px',
            #align_items='flex-end',
            margin='0 0 0 -180px',
            padding='0 5px 2px 0'
        )
    )
    '''
    box = HBox([controls, fw], layout=Layout(justify_content='center', width='1107px', padding='10px 0 10px 0'))
    display(box)

    #display(HBox([ fig.canvas], layout=Layout(width='1107px', align_items='center', margin='0 0 10px 0')))
    #fig.show()


# ——————————————————————————————————————————————————————————————————————
class PolarPlotController:
    def __init__(self, t_vals, theta_vals, phi_vals, width='1107px'):
        self.t_vals = t_vals
        self.theta_vals = theta_vals
        self.phi_vals = phi_vals
        
        self.theta_sel = None
        self.phi_sel = None
        self.maxy = 0
        self._listener = None
        
        self._build_plot()
        self._build_widgets()
        self._wire_callbacks()

        display(
            VBox([ HBox([self.btn_auto, self.fig.canvas], layout=Layout(align_items='flex-start')),
                   self.time_slider ],
                layout=Layout(width=width, align_items='center', margin='0 0 10px 0')
            )
        )

    # ------------------------------------------------------------
    def _build_plot(self):
        self.fig, self.ax = plt.subplots(
            figsize=(8, 8),
            subplot_kw={'projection': 'polar'}
        )

        self.fig.canvas.header_visible = False
        self.fig.canvas.footer_visible = False
        self.fig.canvas.toolbar_visible = True

        self.line, = self.ax.plot(
            self.phi_vals,
            self.theta_vals,
            linewidth=0.6,
            color='blue'
        )

        self.ax.tick_params(axis='both', labelcolor='grey')
        self.ax.grid(True, which='major', color='lightgray',
                     linestyle='--', linewidth=0.5)

        sp = self.ax.spines['polar']
        sp.set_color('gray')
        sp.set_linewidth(0.5)

        self.ax.set_theta_zero_location('S')
        self.ax.set_title('Polar Plot: θ vs φ')

        self.fig.tight_layout()
        self.fig.canvas.layout.margin = '0 32px 0 0'
        self.initial_max = self.ax.get_ylim()[1]

        with io.capture_output():
            plt.show()

    # ------------------------------------------------------------
    def _build_widgets(self):
        t_beg = self.t_vals[0]
        t_end = self.t_vals[-1]

        self.time_slider = FloatRangeSlider(
            value=[0, t_end],
            min=t_beg,
            max=t_end,
            step=(t_end-t_beg)/10000,
            description='Time',
            continuous_update=False,
            layout={'width': '95%', 'margin': '0'},
            style={'description_width': '40px'},
            readout_format='.3f'
        )

        self.btn_auto = Button(
            description='',
            icon='search',
            layout=Layout(width='34px', height='26px', margin='5px 0 0 0'),
        )

    # ------------------------------------------------------------
    def _wire_callbacks(self):
        self.time_slider.observe(self._on_slider_change, names='value')
        self.btn_auto.on_click(self._auto_ylim)

    # ------------------------------------------------------------
    def set_listener(self, fn):
        self._listener = fn
    # ------------------------------------------------------------
    def _on_slider_change(self, change):
        global polar_slider_sel
        
        t_min, t_max = change['new']

        i0 = np.searchsorted(self.t_vals, t_min, side='left')
        i1 = np.searchsorted(self.t_vals, t_max, side='right')

        self.theta_sel = self.theta_vals[i0:i1]
        if self.theta_sel.size == 0:
            return

        self.phi_sel = self.phi_vals[i0:i1]
        self.maxy = math.ceil(self.theta_sel.max() * 40) / 40

        polar_slider_sel["th"]=self.theta_sel
        polar_slider_sel["ph"]=self.phi_sel
        polar_slider_sel["maxy"]=self.maxy
        
        if self.maxy > self.ax.get_ylim()[1]:
            self.ax.set_ylim(0, self.maxy)

        self.line.set_data(self.phi_sel, self.theta_sel)
        self.fig.canvas.draw_idle()
        if self._listeners:
            self._listeners(self.theta_sel, self.phi_sel)

    # ------------------------------------------------------------
    def _auto_ylim(self, _=None):
        if self.theta_sel is None or not self.theta_sel.size:
            return

        top = self.maxy if self.ax.get_ylim()[1] == self.initial_max else self.initial_max
        self.ax.set_ylim(0, top)
        self.fig.canvas.draw_idle()

# ——————————————————————————————————————————————————————————————————————
def plot_polar(t_vals, theta_vals, phi_vals, figsize=(7,7)):
    #class AutoYlimTool(ToolBase):
    #    """Tool to reset the radial limit to [0, max(theta_sel)]."""
    #    default_keymap = ''   # no key bound
    #    description   = 'Auto‐scale r‐axis to data max'
    #    image         = 'home'  # one of the built‐in toolbar icons (e.g. home, pan, zoom)

    #    def trigger(self, sender, event, data=None):
    #        # Your snippet:
    #        if theta_sel.size:
    #            ax.set_ylim(0, theta_sel.max())
    #            fig.canvas.draw_idle()

    #fig = Figure(figsize=(6, 6), dpi=100)
    #ax = fig.add_subplot(1, 1, 1, projection='polar')
    theta_sel = None
    maxy = 0
    fig, ax = plt.subplots(figsize=figsize, subplot_kw={'projection': 'polar'})
    fig.canvas.header_visible = False
    fig.canvas.footer_visible = False
    fig.canvas.toolbar_visible = True

    # Register the tool and add it to the navigation toolbar
    #tm = fig.canvas.manager.toolmanager
    #tm.add_tool('auto_ylim', AutoYlimTool)
    #fig.canvas.manager.toolbar.add_tool('auto_ylim', 'navigation')  # put it next to pan/zoom

    # Plot the full curve and keep the Line2D reference
    line, = ax.plot(phi_vals, theta_vals, linewidth=0.6, color='blue', antialiased=True)
    ax.tick_params(axis='both', labelcolor='grey')
    ax.grid(True, which='major', color='lightgray', linestyle='--', linewidth=0.5, antialiased=True)

    for spine in ax.spines.values():
        spine.set_antialiased(True)

    # make the contour cirle (x-axis) thin
    sp = ax.spines['polar']
    sp.set_color('gray')
    sp.set_linewidth(0.5)

    ax.set_theta_zero_location('S')
    ax.set_title('Polar Plot: θ vs φ')
    fig.tight_layout()
    fig.canvas.layout.margin = '0 50px 0 0'
    initial_max = ax.get_ylim()[1]

    with io.capture_output():
        plt.show()

    # Create a time‐range slider spanning [0, t_vals[-1]]
    t_beg = t_vals[0]
    t_end = t_vals[-1]
    time_slider = FloatRangeSlider(
        value=[t_beg, t_end],
        min=t_beg,
        max=t_end,
        step=((t_end-t_beg) / 20000),       # adjust granularity if needed
        description='Time',
        continuous_update=False,  # callback on release
        layout={'width':'95%', 'margin':'0'}, style={'description_width':'40px'},
        readout_format='.4f'
    )

    #time_slider = SelectionRangeSlider(
    #    options=[(f"{t:.4f}", t) for t in t_vals[::max(1, len(t_vals)//10000)]],
    #    value=(t_beg, t_end),
    #    description='Time',
    #    continuous_update=False,
    #    layout=Layout(width='95%', height='28px'),
    #    style={'description_width': '40px'}
    #)

    
    # Define the callback that updates the line and redraws the figure
    def on_slider_change(change):
        global polar_slider_sel
        nonlocal theta_sel, maxy

        t_min, t_max = change['new']

        i0 = np.searchsorted(t_vals, t_min, side='left')
        i1 = np.searchsorted(t_vals, t_max, side='right')
        # Now slice by indices
        theta_sel = theta_vals[i0:i1]
        if len(theta_sel)==0:
            return
        phi_sel   = phi_vals[i0:i1]
        maxy = math.ceil(theta_sel.max() * 40) / 40

        polar_slider_sel["th"]=theta_sel
        polar_slider_sel["ph"]=phi_sel
        polar_slider_sel["maxy"]=maxy
        
        if maxy > ax.get_ylim()[1]:
            ax.set_ylim(0, maxy)

        # Update the data on the polar line
        line.set_data(phi_sel, theta_sel)

        # Adjust radial limits so the slice always fits
        #if theta_sel.size:
        #    ax.set_ylim(0, theta_sel.max())
        #else:
        #    ax.set_ylim(0, 0)

        fig.canvas.draw_idle()

    # Define the “auto‐ylim” function
    def auto_ylim(_=None):
        nonlocal theta_sel
        if theta_sel is not None and theta_sel.size:
            ax.set_ylim(0, maxy if ax.get_ylim()[1] == initial_max else initial_max)
            fig.canvas.draw_idle()

    # Create a normal ipywidgets button
    btn_auto = Button(
        description='',
        icon='search',
        layout=Layout(width='34px', height='26px', margin='5px 0 0 0'),
    )
    btn_auto.on_click(auto_ylim)

    # Attach the observer to the slider’s “value” trait
    time_slider.observe(on_slider_change, names='value')
    display(VBox([HBox([btn_auto, fig.canvas], layout=Layout(align_items='flex-start')), time_slider],
                 layout=Layout(width='1107px', align_items='center', overflow='hidden', margin='0 0 16px 0')))
    return None


# ——————————————————————————————————————————————————————————————————————
plot_portraits = None
portrait_params = None

def plot_sim_results(t_vals, th_vals, ph_vals, ps_vals, th_dots, ph_dots, ps_dots, params, flag):
    global plot_flg, plots_out, plot_portraits, portrait_params, T_vals, V_vals

    def get_T():
        global T_vals
        if T_vals is None:
            T_vals = compute_T(I1, I2, I3, th_vals, ph_vals, ps_vals, ph_dots, th_dots, ps_dots)
        return T_vals
    
    def get_V():
        global V_vals
        if V_vals is None:
            V_vals = compute_V(om, mB, gam, t_vals, th_vals, ph_vals)
        return V_vals
    
    def get_H():
        T = get_T(); V = get_V()
        if T is None or V is None:
            return None
        return T + V
    
    portraits_vals = [
        (lambda: th_vals, lambda: th_dots, r'$\theta$', r'$\dot\theta$'),
        (lambda: ph_vals, lambda: ph_dots, r'$\phi-\omega t$', r'$\dot\theta$'),
        (lambda: ps_vals, lambda: ps_dots, r'$\psi$', r'$\dot\psi$'),
        (lambda: th_vals, get_T,   r'$\theta$', r'$T$'),
        (lambda: th_vals, get_V,   r'$\theta$', r'$V$'),
        (lambda: th_vals, get_H,   r'$\theta$', r'$H$'),
        ]

    def get_portrait_vals(n):
        get_x, get_y, x_label, y_label = portraits_vals[n]
        return get_x(), get_y(), x_label, y_label
    
    def build_plot1():
        plot = PlotTheta(t_vals, th_vals, set_ani_beg); plot.show()
        #freq_theta = plot1.freq_res.copy()
        plot_spec = PlotSpectrum(plot.freq_res, plot.time_range, 'Frequency Spectrum of $\\theta$', '$\\theta$', figsize=(11,5), fmin=0, fmax=max(310, round(om*2.8/np.pi,-2)), log=True)
        plot.set_callback_spectrum(plot_spec)

    def build_plot2():
        #phi_phases = np.degrees(ph_vals - t_vals * om ) % 360.0 # - np.pi
        #phi_phases = (np.degrees(ph_vals - t_vals * om )+180.0 % 360.0) - 180.0   #this is wrong
        #phi_phases = (np.degrees(ph_vals - t_vals * om )+180.0 % 360.0) - 180.0    #this is wrong but working with parallel alignment 4
        
        #phi_phases = (ph_vals - t_vals * om + np.pi) % (np.pi*2) - np.pi  #this is correct

        phi_phases = np.unwrap(ph_vals - t_vals * om)     # removes +/-2π jumps

        plot = PlotPhiPhase(t_vals, np.degrees(phi_phases)); plot.show()

    def build_plot3():
        plot = Plot_phd_psd(t_vals, ph_dots, ps_dots, None); plot.show()

    def build_plot4():
        sin_ps_vals = np.sin(ps_vals)
        cos_ps_vals = np.cos(ps_vals)
        nu_x_vals_bod = np.sin(th_vals) * sin_ps_vals * ph_dots + cos_ps_vals * th_dots
        nu_y_vals_bod = np.sin(th_vals) * cos_ps_vals * ph_dots - sin_ps_vals * th_dots
        plot = PlotNu_XY(t_vals, nu_x_vals_bod, nu_y_vals_bod, None); plot.show()

    def build_plot5():
        nu_z_vals_bod = np.cos(th_vals) * ph_dots + ps_dots
        nu_z_vals_lab = np.cos(th_vals) * ps_dots + ph_dots
        plot = PlotNu_z(t_vals, nu_z_vals_lab, nu_z_vals_bod, None); plot.show()

    def build_plot6():
        plot = PlotKinetEn(t_vals, th_vals, ph_vals, ps_vals, th_dots, ph_dots, ps_dots, I1, I2, I3, None); plot.show()

    def build_plot7():
        plot = PlotPotenEn(t_vals, th_vals, ph_vals, om, mB, gam, None); plot.show()

    def build_plot8():
        plot = PlotTotalEn(t_vals, get_H(), None); plot.show()

    def build_plot9():
        plot = PlotDissipation(t_vals, th_vals, ph_vals, ps_vals, th_dots, ph_dots, ps_dots, xi, None); plot.show();
    
    def build_plot10():
        plot_polar(t_vals, th_vals, ph_vals, (7,7));

    def build_plot11():
        plot_sphere2(t_vals, th_vals, ph_vals, 700);

    def build_plot12():
        global T_vals, V_vals, plot_portraits
        option = phase_portr_dd.value

        x_vals, y_vals, x_label, y_label = get_portrait_vals(option-1)

        if phase_portr_dd.value == 2: # phi -omega*t / phi_dot
            #x_vals = x_vals - om * t_vals
            #x_vals = (x_vals + np.pi) % (2*np.pi) - np.pi
            
            x_vals = np.unwrap(x_vals - om * t_vals) #this works but fails in negatives
            #x_vals = (x_vals - t_vals * om + np.pi) % (np.pi*2) - np.pi
                
        if plot_portraits is None:
            plot_portraits = PlotPhasePortrait(t_vals, x_vals, y_vals, x_label, y_label, portrait_params); 
            ui = plot_portraits.show(figsize=(6.5, 6.5))
            display(ui)
        else:
            plot_portraits.update(t_vals, x_vals, y_vals, x_label, y_label)
    # ——————————————————————————————————————————————————————————————————————

    build_plots = [build_plot1, build_plot2, build_plot3, build_plot4, build_plot5, build_plot6, build_plot7, build_plot8, build_plot9, build_plot10, build_plot11, build_plot12]

    I1, I2, I3, mB, om, gam, xi =  params

    def build_plot_wrap(i):
        if plot_flg[i] is False:
            if i !=11 or plot_portraits is None:
                plots_out[i].clear_output() #gpt suggested
            with plots_out[i]:
              build_plots[i]()
              #print('build_plots', i)
            plot_flg[i] = True
        plots_out[i].layout.display = ""

    # ——————————————————————————————————————————————————————————————————————
    if flag < 0:
        if plot_portraits:
            portrait_params = plot_portraits.get_params()
            plot_portraits = None
            
        for i in range(len(plot_flg)):
            plot_flg[i] = False
        plt.close('all')
        for out in plots_out:
            out.clear_output()

        if ck_th.value:
            build_plot_wrap(0)
        if ck_php.value:
            build_plot_wrap(1)
        if ck_phd_psd.value:
            build_plot_wrap(2)
        if ck_nuXY.value:
            build_plot_wrap(3)
        if ck_nuZ.value:
            build_plot_wrap(4)
        if ck_kinE.value:
            build_plot_wrap(5)
        if ck_magE.value:
            build_plot_wrap(6)
        if ck_totE.value:
            build_plot_wrap(7)
        if ck_diss.value and xi:
            build_plot_wrap(8)
        if ck_polar.value:
            build_plot_wrap(9)
        if ck_spher.value:
            build_plot_wrap(10)
        if phase_portr_dd.value != 0:
            build_plot_wrap(11)
    elif flag < N_PLOTS+1:
            build_plot_wrap(flag)
    
#————————————————————— Plotting procedures ends ———————————————————————





    
#——————————————————————————————————————————————————————————————————————
def reordered_vector_latex(name, vec):
    """
    Display `name = vec` as a pmatrix, reordering each entry
    so (ω*t) factors come before γ-factors, preserving that order in LaTeX,
    and omitting the leading `1*` when the coefficient is +1.
    """
    rows = []
    for i in range(vec.rows):
        e = vec[i, 0]
        # 1) extract coeff and factors
        coeff, factors = e.as_coeff_mul()

        # 2) sort factors by (ω*t) first, then γ, then others
        def key(f):
            syms = f.free_symbols
            if t in syms or omega in syms: return 0
            if gamma in syms:              return 1
            return 2
        ordered = sorted(factors, key=key)
        # 3) rebuild without the +1 coefficient
        if coeff == 1:
            e_ord = sp.Mul(*ordered, evaluate=False)
        else:
            e_ord = sp.Mul(coeff, *ordered, evaluate=False)
        # 4) render to LaTeX without reordering
        s = latex(e_ord, order='none')
        s = s.replace(r'\left(t \right)', '') #\left(t \right)
        rows.append(s)
        #print (s)
        #rows.append(latex(e_ord, order='none'))

    # 5) assemble pmatrix
    body = r" \\ ".join(rows)
    if name:
        return rf"{name} = \begin{{pmatrix}}{body}\end{{pmatrix}}"
    else:
        return rf"\begin{{pmatrix}}{body}\end{{pmatrix}}"

#——————————————————————————————————————————————————————————————————————
def gamma_min_mb2(x):
    """
    Empirical γ_min approximation via a 5rd-order polynomial:
        γ_min(x) = p1·x**5 + p2·x**4 + p3·x**3 + p4·x**2 + p5·x + p6

    Coefficients (with 95% confidence bounds):
      p1 =  0.4388  (0.2729, 0.6047)
      p2 = -2.9481  (-3.6718, -2.2243)
      p3 =  7.7000  (6.4535, 8.9465)
      p4 = -10.3082 (-11.3678, -9.2487)
      p5 =  6.9811  (6.5365, 7.4257)
      p6 = -1.5784  (-1.6521, -1.5047)
    """
    p1, p2, p3, p4, p5, p6 = 0.4388, -2.9481, 7.7000, -10.3082, 6.9811, -1.5784
    return (p1 * x**5 +
            p2 * x**4 +
            p3 * x**3 +
            p4 * x**2 +
            p5 * x +
            p6)

#——————————————————————————————————————————————————————————————————————
'''
def notify_when_ui_responsive():
    js = """
    (function() {
        function whenIdle() {
            // Ask browser: "Call me when you're not busy"
            requestIdleCallback(() => {
                // Double-check after one more animation frame
                requestAnimationFrame(() => {
                    alert("UI is now responsive");
                });
            });
        }
        whenIdle();
    })();
    """
    with js_out:
        display(Javascript(js))
'''
'''
from IPython import get_ipython

def notify_when_notebook_idle():
    ip = get_ipython()

    def callback():
        # This runs AFTER all pending UI tasks complete
        with js_out:
            display(Javascript("""
                requestAnimationFrame(() => {
                    requestIdleCallback(() => {
                        alert("UI is now responsive");
                    });
                });
            """))

        # remove callback so it runs only once
        ip.events.unregister('post_execute', callback)

    ip.events.register('post_execute', callback)
'''
###########################################################################################
###########################################################################################
# start from here
#import matplotlib as mpl
global em
top_title = Latex(r"\[\large \text{Solution, simulation and analysis of the angular dynamics of a free body with a magnetic moment subject to a rotating magnetic field}\]")

out = Output()
display(out)
    
SETUP_DONE_FLAG = False
if SETUP_DONE_FLAG not in globals():
    # make sure the RESULTS and RESULTS/TEMP folders exist
    os.makedirs('RESULTS', exist_ok=True)
    os.makedirs('RESULTS/TEMP', exist_ok=True)
    with out:
        xyz_buttons_css()
        css_shade = """
<style>
.my-disabled {
  opacity: 0.55;                 /* dim the whole widget */
  filter: grayscale(20%);        /* slightly gray it */
  pointer-events: none;          /* disable mouse events if you want it non-interactive */
  transition: opacity 0.15s;
}

/* if you want disable only interaction but not child widgets removal: keep pointer-events: auto on children */
.my-disabled * { pointer-events: none; }
</style>
"""
        display(HTML(css_shade))
        display(top_title)
        display(Latex(r"\(\text{Deriving equations of motion, please wait}\ldots\)"))
        try:
            em = EOMDiag()
            globals()[SETUP_DONE_FLAG] = True
            out.clear_output()  # clear_output(wait=True)
        except Exception as e:
            print(f"Exception: {e}")
            display(str(e))

#============================================================================
# global variables
psd_changing = False
eps_gamma_suppress = False
show_nu = True

calc_psd_from_cos_theta = 1 # how psd is set:  0 = user defined, 1 = -omega, 2 = -cos(theta)*omega
current_rec_key = None
om_B = 100*np.pi  # user defined parameter
I1_v = 1e-7       # user defined parameter
I2_v = 1e-7       # user defined parameter
I3_v = 1e-7       # user defined parameter
mB_v = 3.0e-3     # user defined parameter
psd_ratio = 1
ph_v = np.pi      # phase of the theta angle
phd_v = om_B
psd_v = -om_B * psd_ratio
th_eq = th_v = 1e-9
th_offs = 0
thd_v = 0         # normally zero but can be used for non_zero theta angle velocities for non equilibrium tests
ps_v = 0          #phase angle of psi should have no effect
#thd_v = np.pi
gam_v = gam_min = gam_eps = 0
xi_v = 0           # damping factors (xi effective on velocity vector nu)
ps_acc = 0
t0_v = 0

#om_nat = 0.0
omega_n1 = 0.0
omega_n2 = 0.0
sim_time = 10
sim_steps = 2500
link = True
id_hash = '0'
csv_filename = None
mp4_filename = None
csv_files = []
mp4_files = []
time_range_g = None
body_shape = 'Spheroid'
body_scale = 1
box_polar_plot = None
T_vals = V_vals = None  # calculated instantaneous kinetic T(t) and potential V(t) energies as np.array
stop_event = threading.Event()
counter = 0
t_vals = y_vals = None  # simulation result
sim_save_dlg = None
anim_save_dlg = None
csv_dialog = None
# eigenvalue analysis global variables
eigvals_all = None
eigvecs_all = None
sweep_vals = None
plot_flg = [False] * (N_PLOTS+1)
#freq_theta = None
eq_phi_phase = -em.xi * em.omega * sp.sin(em.th)/(em.mB*sp.cos(em.gamma))
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 200

# ——————————————————————————————————————————————————————————————————————
def write_csv():
    global csv_filename, t_vals, y_vals, id_hash
    if t_vals is None or y_vals is None:
        return False
    try:
        th_vals, ph_vals, ps_vals = y_vals[0], y_vals[1], y_vals[2]
        th_dots, ph_dots, ps_dots = y_vals[3], y_vals[4], y_vals[5]

        # export CSV
        csv_filename = f'RESULTS/sim-{id_hash}-{time.strftime("%Y%m%d_%H%M%S")}.csv'
        # make sure the RESULTS folder exists
        os.makedirs('RESULTS', exist_ok=True)

        with open(csv_filename, 'w', newline='') as f:
            phys_params = physics_params_dict()
            f.write("# physics_params = ")
            f.write(json.dumps(phys_params))
            f.write("\n")
            
            w = csv.writer(f);
            w.writerow(['t','theta','phi','psi','theta_dot','phi_dot','psi_dot'])
            for i in range(len(t_vals)):
                w.writerow([t_vals[i],
                            th_vals[i], ph_vals[i], ps_vals[i],   # theta, phi, psi
                            th_dots[i], ph_dots[i], ps_dots[i]])  # theta_dot, phi_dot, psi_dot
        return True
    except Exception as e:
        lb_result_sim.value = str(e)
        return False
# ——————————————————————————————————————————————————————————————————————

# Assume simulate_theta, plot_theta, plot_phid_psid, plot_polar, gamma_min_mb2 are imported
def run_sim(om_, I1_, I2_, I3_, mB_, gam_, th_ , ph_, ps_, thd_, phd_, psd_, xi_, ps_acc, solver_params, steps_per_sec, t_start, simlength) -> bool:
    global csv_filename, box_polar_plot, t_vals, y_vals, T_vals, V_vals #, freq_theta
    T_vals = V_vals = None
    xi = (I1_ + I2_) * xi_ * 0.5
    params = (I1_, I2_, I3_, mB_, om_, gam_, xi)
    init = (th_, ph_, ps_)
    omega_body = (thd_, phd_, psd_)
    start = time.perf_counter()
    t_vals, y_vals = simulate_euler(params, init, omega_body, t_start, t_end=t_start+simlength, ps_acc=ps_acc, solver_params=solver_params, n_steps=int(steps_per_sec*simlength))
    if t_vals is None:
        return 0
    duration = time.perf_counter() - start
    plot_sim_results(t_vals, y_vals[0], y_vals[1], y_vals[2], y_vals[3], y_vals[4], y_vals[5], params, -1)
    return duration

def on_toggle_plot_view(change, i):
    if change['new'] and t_vals is not None:
        params = I1_v, I2_v, I3_v, mB_v, om_B, gam_v, xi_v*(I1_v+I2_v)*0.5
        plot_sim_results(t_vals, y_vals[0], y_vals[1], y_vals[2], y_vals[3], y_vals[4], y_vals[5], params, i)
    else:
        plots_out[i].layout.display = "None"

# ——————————————————————————————————————————————————————————————————————
def show_nu_vector():
    #caller = inspect.stack()[1]
    #print(f"Called from {caller.function}() line {caller.lineno}")    
    the = w_th.valRad
    phi = w_ph.valRad
    psi = w_ps.valRad
    thd = w_thd.valRad
    phd = w_phd.valRad
    psd = w_psd.valRad
    #nu_x =  np.sin(phi)*np.sin(the)*psd
    #nu_y = -np.cos(phi)*np.sin(the)*psd
    #nu_z =  np.cos(the)*psd + phd

    subs = {em.th:the, em.ph:phi, em.ps:psi, em.th_d:thd, em.ph_d:phd, em.ps_d:psd}

    # velocity vector in world frame
    #nu0_wo = em.nu_wo.subs(subs)
    # velocity vector in body frame
    #nu0_bo = em.nu_bo.subs(subs)

    # velocity vector in world frame
    nuw_x, nuw_y, nuw_z = map(float, em.nu_wo.subs(subs))
    # velocity vector in body frame
    nub_x, nub_y, nub_z = map(float, em.nu_bo.subs(subs))

    #from a matlab formula
    #comsol_vy = -phd*np.sin(the)  # Vy = -OM*sin(PHI)
    #comsol_vz = psd+phd*np.cos(the)  # Vz =  OM*(k+cos(PHI))

    if DEGREE_MODE:
        rev = 0.5/np.pi
        # velocity vector in world frame
        nuw_x *= rev
        nuw_y *= rev
        nuw_z *= rev
        nub_x *= rev
        nub_y *= rev
        nub_z *= rev

    params_status1.setM(
        f"<span style=\"font-family: 'Cambria Math', Cambria; font-weight: 500\">"
        f"ν<sub>0 bo</sub></span> = [ {nub_x:.8f}, {nub_y:.8f}, {nub_z:.8f} ]</span> {velunit()}"
        )

    params_status2.setM(
        f"<span style=\"font-family: 'Cambria Math', Cambria; font-weight: 500\">"
        f"ν<sub>0 wo</sub></span> = [ {nuw_x:0.8f}, {nuw_y:0.8f}, {nuw_z:0.8f} ] {velunit()}"
        )

# ——————————————————————————————————————————————————————————————————————
def safe_clear_psd_flag(cleanup_fn):
    io_loop.add_callback(cleanup_fn)

# ——————————————————————————————————————————————————————————————————————
def clear_eps_gamma_suppress():
    global eps_gamma_suppress
    eps_gamma_suppress = False

def clear_psd_flag():
    global psd_changing
    psd_changing = False

#——————————————————————————————————————————————————————————————————————
def calc_th_eq_simple_sub():
    global th_eq, phd_d, psd_v, gam_v, mB_v, I1_v, I2_v, I3_v
    #print(f"calc_th_eq_simple: psd_v = {psd_v}  Called by function: {inspect.stack()[1].function} , {inspect.stack()[2].function}")
    IR = (I1_v + I2_v)/2
    if IR == I3_v and psd_v == -phd_v:
        return sp.atan2(mB_v*sp.cos(gam_v), IR * np.abs(phd_v*psd_v) - mB_v * sp.sin(gam_v))
    elif phd_v!= 0:
        K = psd_v / phd_v
        OM2 = phd_v**2
        JC = mB_v * sp.cos(gam_v)/OM2
        JS = mB_v * sp.sin(gam_v)/OM2
        x, ir, ia, om2, mb, jc, js, k = sp.symbols('x ir ia vv mb jc js k', real=True)

        #PHI = vpasolve(TC*cos(x)+TS*sin(x) + OM^2*sin(x)*(k*IA + cos(x)*(IA-IR))==0)

        #expr = vv * sp.sin(x) * (i3 + sp.cos(x)*(i1-i3)) - mb*(sp.sin(g) * sp.sin(x) + sp.cos(g) * sp.cos(x))

        expr = jc*sp.cos(x) + js*sp.sin(x) + (k*ia + (ia-ir)*sp.cos(x))*sp.sin(x)

        res = sp.nsolve(expr.subs({ir: IR, ia: I3_v, jc: JC, js: JS, k: K}), 0.1)
        #with out:
        #    print('theta_eq = ', res)
        return res

def calc_th_eq_simple():
    global th_eq
    w_th_eq.valRad = th_eq = calc_th_eq_simple_sub()

#——————————————————————————————————————————————————————————————————————
# calculate equilibrium theta while keeping relation  phi_dot = omega, psi_dot = -omega*cos(theta)
def calc_theta_for_psi_cos_omega_sub():
    x, ir, o, mb, g = sp.symbols('x i1 o mb g', real=True)
    expr = o**2 * sp.sin(x) * sp.cos(x) * ir - mb*(sp.sin(g) * sp.sin(x) + sp.cos(g) * sp.cos(x)) #expr = ir *o**2 * sin(2*x)/2 - mb*cos(g - x)
    try:
        return sp.nsolve(expr.subs({ir:(I1_v + I2_v)/2, o:om_B, mb:mB_v, g:gam_v}), 0.1)
    except Exception as e:
        print (e)
    return 1e-20

#——————————————————————————————————————————————————————————————————————
# modifies th_eq, psd_v, w_psd_ratio and their widgets
def calc_theta_for_psi_cos_omega():
    global th_eq, psd_ratio, psd_v
    th_eq = calc_theta_for_psi_cos_omega_sub()
    w_th_eq.valRad = th_eq
    w_psd_ratio.valRad = psd_ratio = sp.cos(th_eq)
    psd_v = -om_B * psd_ratio
    w_psd.valRad = psd_v

#——————————————————————————————————————————————————————————————————————
def calc_th_eq_sub():
    global calc_psd_from_cos_theta, om_B, I1_v, I2_v, mB_v, gam_v, xi_v, th_eq, ph_v, xi_v, psd_v
    IR = (I1_v + I2_v)/2
    if calc_psd_from_cos_theta==2:
        if xi_v == 0:
            return calc_theta_for_psi_cos_omega_sub(), np.pi
        else:
            return calc_theta_phi_with_xi_cos_sub(om_B, IR, mB_v, gam_v, xi_v)
    else:
        if xi_v == 0:
            return calc_th_eq_simple_sub(), np.pi
        else:
            return calc_theta_phi_with_xi_sub(om_B, IR, I3_v, mB_v, gam_v, xi_v, psd_v)

def calc_th_eq():
    global calc_psd_from_cos_theta
    if calc_psd_from_cos_theta==2:
        if xi_v == 0:
            calc_theta_for_psi_cos_omega()
        else:
            calc_theta_phi_with_xi_cos()
    else:
        if xi_v == 0:
            calc_th_eq_simple()
        else:
            calc_theta_phi_with_xi()

#——————————————————————————————————————————————————————————————————————
def calc_gamma_min():
   global gam_min, om_B, I1_v, I2_v, I3_v, mB_v

   angle = mB_v / ((I1_v + I2_v) * om_B**2) if om_B else 0
   if angle < 1:
       gam_min = sp.asin(angle)
   else:
       gam_min = float('nan')

_ft_lay = Layout(width='186px', margin='0 5px 0 0')
_ft_lay2= Layout(width='160px', margin='0 5px 0 0')
_ft_sty  = {'description_width': '25px'}

#——————————————————————————————————————————————————————————————————————
def calc_theta_phi_with_xi_cos_sub(omega, IR, mB, gamma, xi):
    #print('IR=', IR)
    th = sp.symbols('th', real=True)
    phi0 = sp.pi + sp.asin( xi * IR * omega * sp.sin(th) / (mB * sp.cos(gamma)) )

    # ψ̇0(theta)
    psi_dot =-omega * sp.cos(th)    # your first-order model

    # Steady-state θ equation:
    eq = -IR * sp.sin(th) * omega * psi_dot + mB * (-sp.sin(gamma) * sp.sin(th) + sp.cos(gamma) * sp.cos(phi0) * sp.cos(th))
    # Convert to a numerical function
    f_th = sp.lambdify(th, eq, 'mpmath')

    try:
        # Provide an initial guess
        th_guess = 0.2
        theta_sol = findroot(f_th, th_guess)

        # Compute phi0
        phi0_func = sp.lambdify(th, phi0, 'mpmath')
        phi0_sol = phi0_func(theta_sol)
        if isinstance(theta_sol, mpc) or isinstance(phi0_sol, mpc):
            raise ValueError("Complex solution encountered")
    except Exception as e:
        print(f'calc_theta_phi_with_xi_sub : {str(e)}')
        return 1e-15, 0
    #print("θ (rad) =", float(theta_sol))
    #print("φ0 (rad) =", float(phi0_sol))
    return float(theta_sol), float(phi0_sol)

#——————————————————————————————————————————————————————————————————————
def calc_theta_phi_with_xi_sub(omega, IR, I3, mB, gamma, xi, psi_dot):
    th = sp.symbols('th', real=True)
    #not valid if psid != omega*cos(theta)
    #phi0 = sp.pi + sp.asin( xi * I1 * omega * sp.sin(th) / (mB * sp.cos(gamma)) )


    #mB * sin(phi0) * cos(gamma) = xi * I1 * sin(th) * phd_v
    #phi0 = sp.pi + asin(xi * I1 * sin(th) * phd_v / (cos(gamma)* mB))

    phi0 = sp.pi + asin(omega * xi * IR * sp.sin(th) / (mB * cos(gamma)))

    # Steady-state θ equation:
    #eq = mB * (-sp.sin(gamma) * sp.sin(th) + sp.cos(gamma) * sp.cos(phi0) * sp.cos(th)) - I1 * sp.sin(th) * omega * psi_dot

    # Steady-state θ equation (axisymmetric)
    eq = mB * (-sp.sin(gamma) * sp.sin(th) + sp.cos(gamma) * sp.cos(phi0) * sp.cos(th)) - (IR * psi_dot + (I3 - IR) * (sp.cos(th) * omega + psi_dot)) * sp.sin(th) * omega

    # Convert to a numerical function
    f_th = sp.lambdify(th, eq, 'mpmath')

    try:
        # Provide an initial guess
        th_guess = 0.2
        theta_sol = findroot(f_th, th_guess)
        # Compute phi0
        phi0_func = sp.lambdify(th, phi0, 'mpmath')
        phi0_sol = phi0_func(theta_sol)
        if isinstance(theta_sol, mpc) or isinstance(phi0_sol, mpc):
            raise ValueError("Complex solution encountered")
    except Exception as e:
        print(f'calc_theta_phi_with_xi_sub : {str(e)}')
        return 1e-15, 0

    #print("θ (rad) =", float(theta_sol))
    #print("φ0 (rad) =", float(phi0_sol))
    return float(theta_sol), float(phi0_sol)

#——————————————————————————————————————————————————————————————————————
# modifies th_eq, ph_vm, psd_v, w_psd_ratio and their widgets
def calc_theta_phi_with_xi_cos():
    global gam_v, om_B, I1_v, I2_v, I3_v, mB_v, gam_v, xi_v, th_eq, ph_v
    th_eq, ph_v = calc_theta_phi_with_xi_cos_sub(om_B, (I1_v+I2_v)/2, mB_v, gam_v, xi_v)
    w_th_eq.valRad = th_eq
    w_ph.valRad = ph_v
    w_psd_ratio.valRad = psd_ratio = sp.cos(th_eq)
    psd_v = -om_B * psd_ratio
    w_psd.valRad = psd_v

#——————————————————————————————————————————————————————————————————————
def calc_theta_phi_with_xi():
    global om_B, I1_v, I2_v, I3_v, mB_v, gam_v, psd_v, xi_v, th_eq, ph_v

    th_eq, ph_v = calc_theta_phi_with_xi_sub(om_B, (I1_v+I2_v)/2, I3_v, mB_v, gam_v, xi_v, psd_v)
    w_th_eq.valRad = th_eq
    w_ph.valRad = ph_v

#——————————————————————————————————————————————————————————————————————
#def calc_omega_nat():
#    # another empirical formula
#    if gam_v > gam_min and I1_v > 0:
#        return float(sp.sqrt(sp.sin(gam_v - gam_min) * mB_v / I1_v))
#    else:
#        return 0
#    #w_om_nat.valRad = om_nat

#——————————————————————————————————————————————————————————————————————
#def set_psd_ratio():
#    psd_ratio = sp.cos(th_eq)
#    psd_v = -om_B * psd_ratio
#    w_psd.valRad = psd_v

#——————————————————————————————————————————————————————————————————————
# this routine provides a counter accleration to compansate the drap caused by damping.
# it is based on axisymeetic solutions, however approximates this in the case of small non-axistmmetry
def on_set_ext_psdd(button):
    global psd_changing, th_eq, xi_v
    if psd_changing:
            return
    if xi_v!=0 and abs(th_eq)>1e-15:
        #thd = thd_v = w_thd.valRad
        phd = phd_v = w_phd.valRad
        psd = psd_v = w_psd.valRad
        th = th_eq = w_th_eq.valRad
        ph_v = w_ph.valRad
        phi_phase = -ph_v
        IR = (I1_v+I2_v)/2 # accept small non-axisymmetries
        xi = xi_v * IR
        #print(phi_phase, thd, phd, psd, th, IR, I3_v, mB_v, gam_v, xi)

        sin_th = sin(th)
        cos_th = cos(th)

        #exclude the term 1 and term4 because thd is zero under equilibrium
        #term1 = -I3_v * cos(th) * psd * thd / sin_th
        #term4 = (-Ir * sin_th + 2 * IR/sin_th + I3_v * sin_th - I3_v / sin_th) * phd * thd
        term2 = -mB_v * np.sin(phi_phase) * cos(gam_v) * cos_th / sin_th
        term3 = xi * (-IR * (cos_th * phd + psd) / I3_v + cos_th * phd)
        #print(term2,term3)
        RHS = term2 + term3  #(term1 and term4 excluded)
        ps_acc = -RHS / IR
        if abs(ps_acc) < 1e-10:
            ps_acc = 0

        w_ps_acc.valRad = ps_acc
    else:
        w_ps_acc.valRad = ps_acc = 0

# ——————————————————————————————————————————————————————————————————————
def update_initials(change):
    global psd_changing, om_B, I1_v, I2_v, I3_v, mB_v, gam_v, gam_min, gam_eps, phd_v, psd_v, xi_v, psd_ratio, th_offs, calc_psd_from_cos_theta

    om_B = w_om.valRad
    I1_v = w_I1.value
    I2_v = w_I2.value
    I3_v = w_I3.value
    mB_v = w_mB.value
    xi_v = w_xi.value
    gam_eps = w_gam_eps.valRad

    #if I3_v >= 2 * I1_v:
    #    params_status.set("For a homogenious filled body, I_{3} should be less than 2 I_{1}", color="darkred")
    #else:
    params_status1.set("")
    params_status2.set("")
    if w_link.value:
        #gam_v = w_gam.valRad
        psd_changing = True
        calc_gamma_min()
        if math.isnan(gam_min):
            return

        gam_v = gam_min + gam_eps
        #print(f"update_initials 0: gam_min = {gam_min}")

        #print(f"om_nat = {om_nat}")
        #if gam_v==0 or True:a
        #    gam_v = gam_min + gam_eps
        #    w_gam.valRad = gam_v
            #print(f"update_initials 2: gam_min = {gam_min}, gam_v = {gam_v}")
        w = change['owner']
        if w is w_om:
            w_phd.valRad = phd_v = om_B

        if calc_psd_from_cos_theta==2:
            if xi_v == 0:
                calc_theta_for_psi_cos_omega()
            else:
                calc_theta_phi_with_xi_cos()
        else:
            w_psd.valRad = psd_v = -om_B * psd_ratio
            if xi_v == 0:
                calc_th_eq_simple()
            else:
                calc_theta_phi_with_xi()

        #w_thd.valRad  = 0
        #print("update_initials: th_offs = 0")
        #reset th_offs because phd_v and psd_v are calculated  for theta = theta_equilibrium
        #otherwise we can  phd_v and psd_v keeping th_offs as is and nu_body invariant from theta change
        w_th_eps.valRad = th_offs = 0
        #w_phd.valRad = phd_v
        #w_psd.valRad = psd_v
        w_gam.valRad = gam_v
        #calc_omega_nat()
        show_nu_vector()
        safe_clear_psd_flag(clear_psd_flag)

# ——————————————————————————————————————————————————————————————————————
def update_th_eq(change):
    global th_v, th_eq, th_offs
    if w_link.value:
        th_eq = w_th_eq.valRad
        th_offs = w_th_eps.valRad
        th_v = th_eq + th_offs
        w_th.valRad = th_v

        if w_xi.value != 0:
            on_set_phi_zero(None)

        if th_eq >= np.pi/2:
            #print('red')
            w_th_eq.style.background_color = '#fee2e2'
        else:
            w_th_eq.style.background_color = '#ffffff'

        #print(f"update_th_eq: {th_v}")

# ——————————————————————————————————————————————————————————————————————
# fired when th_v is updated
def update_th(change):
    global th_v, th_eq, th_offs
    if w_link.value:
        th_v = w_th.valRad
        th_offs = w_th_eps.valRad = th_v  - w_th_eq.valRad
        if th_v >= np.pi/2:
            #print('red')
            w_th.style.background_color = '#fee2e2'
        else:
            w_th.style.background_color = '#ffffff'

# ——————————————————————————————————————————————————————————————————————
# called when gamma entry is changed
def update_gamma_v(change):
    global psd_changing, eps_gamma_suppress, th_eq, th_v, th_offs, gam_v, gam_min, gam_eps
    if w_link.value:
        if psd_changing or math.isnan(gam_min):
            #print(f"update_gamma_v: gam_min = {gam_min}")
            return
        #print(f"update_gamma_v: psd_changing = {psd_changing}")
        psd_changing = True
        om_B = w_om.valRad
        gam_v = w_gam.valRad
        if not eps_gamma_suppress:
            w_gam_eps.valRad = gam_eps = gam_v - gam_min
        else:
            gam_eps = w_gam_eps.valRad
        #print(f"w_gam_eps.valRad = gam_v - gam_min: {w_gam_eps.valRad}")
        calc_th_eq()
        th_offs = w_th_eps.valRad
        th_v = th_eq + th_offs
        w_th.valRad = th_v
        safe_clear_psd_flag(clear_psd_flag)
    else:
        gam_v = w_gam.valRad
        if not eps_gamma_suppress:
            w_gam_eps.valRad = gam_eps = gam_v - gam_min
    show_nu_vector()

# ——————————————————————————————————————————————————————————————————————
# called when gamma_epsilon entry is changed
def update_gamma_eps(change):
    global psd_changing, eps_gamma_suppress, gam_eps, gam_min, gam_v
    if psd_changing or math.isnan(gam_min):
        return
    gam_eps = w_gam_eps.valRad
    eps_gamma_suppress = True
    w_gam.valRad = gam_v = gam_min + gam_eps
    safe_clear_psd_flag(clear_eps_gamma_suppress)

# ——————————————————————————————————————————————————————————————————————
# recalculate phi_dot and psi_dot in order  body angular velocity nu remain
# same of nu of theta_equilibrium
def remap_rates(old_theta, phi, old_phi_dot, old_psi_dot, new_theta):
    """
    Remap (φ̇,ψ̇) so that world‐frame ω = (νx,νy,νz) stays identical when θ jumps.
    Inputs:
      old_theta, phi           : before‐jump θ, φ
      old_phi_dot, old_psi_dot : before‐jump φ̇, ψ̇
      new_theta                : after‐jump θ
    Returns:
      (phi_dot, psi_dot)
    """
    # Reconstruct νx, νy, νz at the instant (dotθ=0)
    nu_x =  np.sin(old_theta) * np.sin(phi)*old_psi_dot
    nu_y = -np.sin(old_theta) * np.cos(phi)*old_psi_dot
    nu_z =  np.cos(old_theta) * old_psi_dot + old_phi_dot
    sin_nt = np.sin(new_theta)
    #print('1.new_theta: ',new_theta)
    if np.isclose(sin_nt, 0.0):
        raise ValueError("new_theta too close to 0 or π → singular")

    # Pick stable inversion for ψ̇
    if abs(np.sin(phi)) > 1e-6:
        psi_dot = nu_x / (sin_nt * np.sin(phi))
    else:
        # fall back when sin(phi)=0
        psi_dot = -nu_y / (sin_nt * np.cos(phi))

    # φ̇ follows from νz = cosθ·ψ̇ + φ̇
    phi_dot = nu_z - np.cos(new_theta) * psi_dot
    return phi_dot, psi_dot


# ——————————————————————————————————————————————————————————————————————
# recalculate phi_dot and psi_dot in order  angular velocity in comsol initial conditon remains same
# same of nu of theta_equilibrium
def remap_rates_comsol(theta_old, phi, phd_old, psd_old, theta_new):
    s_old = np.sin(theta_old)
    s_new = np.sin(theta_new)
    #print('2.theta_new: ', theta_new)
    if np.isclose(s_new, 0.0):
        raise ValueError("theta_new is too close to 0 or π → sin(theta_new)=0")

    phd_new = phd_old * (s_old / s_new)

    psd_new = (psd_old + phd_old * np.cos(theta_old) - phd_new * np.cos(theta_new))

    return phd_new, psd_new
# ——————————————————————————————————————————————————————————————————————

_saved_phd = 0
_saved_psd = 0

def update_phd_psd_from_eps_theta(change):
    global psd_changing, _saved_phd, _saved_psd, th_v, th_eq, th_offs
    if psd_changing:
        return
    psd_changing = True
    th_offs = w_th_eps.valRad
    th_eq = w_th_eq.valRad
    th_v = th_eq + th_offs

    if w_link.value:
        old_eps = change.get('old', 0)
        if old_eps == 0:
            _saved_phd = w_phd.valRad
            _saved_psd = w_psd.valRad

        if _saved_phd:
            w_phd.valRad, w_psd.valRad = remap_rates(th_eq, w_ph.valRad, _saved_phd, _saved_psd, th_v)
            w_phd.valRad, w_psd.valRad = remap_rates_comsol(th_eq, w_ph.valRad, _saved_phd, _saved_psd, th_v)
            w_th.valRad = th_v
            #print("update_phd_psd_from_eps_theta")
            show_nu_vector()
    else:
       w_th.valRad = th_v

    safe_clear_psd_flag(clear_psd_flag)

# ——————————————————————————————————————————————————————————————————————
def update_psd_from_ratio(change):
    global psd_changing, psd_ratio, psd_v, th_v, th_eq, calc_psd_from_cos_theta

    if w_link.value is False or psd_changing is True or calc_psd_from_cos_theta==2:
        return
    #print("update_psd_from_ratio")
    psd_changing = True
    calc_psd_from_cos_theta = 0
    set_psi_omega_cos_theta_btn_color()

    psd_ratio = w_psd_ratio.value
    #print("update_psd_from_ratio", psd_ratio)
    if np.isclose( w_th_eq.valRad,  w_th_eq.valRad):
        #print("update_psd_from_ratio")
        #psd_ratio = round(psd_ratio,3)
        #print(f"update_psd_from_ratio: {psd_ratio}")
        psd_v = -om_B * psd_ratio
        # Only emit one notification on w_psd when we leave this block
        #with w_psd.hold_trait_notifications():
        w_psd.valRad = psd_v
        #print("update_psd_from_ratio:psd=", psd_v)
        show_nu_vector()
    safe_clear_psd_flag(clear_psd_flag)

# ——————————————————————————————————————————————————————————————————————
def update_ph(change):
    global show_nu
    if show_nu:
         show_nu_vector()

def update_thd(change):
    global show_nu
    if show_nu:
         show_nu_vector()

def update_phd(change):
    global show_nu
    if show_nu:
         show_nu_vector()
# ——————————————————————————————————————————————————————————————————————
def update_psd(change):
    global psd_changing, psd_ratio, psd_v, calc_psd_from_cos_theta
    if w_link.value:
        if psd_changing is False:
            psd_changing = True
            #print("update_psd 1")
            calc_psd_from_cos_theta = 0
            set_psi_omega_cos_theta_btn_color()
            psd_v = w_psd.valRad
            psd_ratio = (-psd_v / phd_v) if phd_v else 1
            # Only emit one notification on w_psd_ratio when we leave this block
            #with w_psd_ratio.hold_trait_notifications():
            w_psd_ratio.value = psd_ratio

            # only when psd_changing is False
            psd_v = w_psd.valRad
            #print(f"update_psd: {psd_v}")
            if xi_v == 0:
                calc_th_eq_simple()
            else:
                calc_theta_phi_with_xi()
            update_th_eq(None)
            #print("update_psd")
            show_nu_vector()
            safe_clear_psd_flag(clear_psd_flag)
        else:
            #elif calc_psd_from_cos_theta is False:
            #psd_v = w_psd.valRad
            #print(f"update_psd 2: psd_changing={psd_changing}")
            #psd_ratio = (-psd_v / phd_v) if phd_v else 1
            #w_psd_ratio.value = psd_ratio
            pass

#    a version allowing update even when psd_changing is true
#    psd_v = w_psd.valRad
#    print(f"update_psd: {psd_v}")
#    calc_th_eq_simple()
#    update_th_eq(None)
#    w_th_eq.valRad = th_eq

# ——————————————————————————————————————————————————————————————————————
def set_psi_omega_cos_theta_btn_color():
    global calc_psd_from_cos_theta
    bt_psi_om_cos_theta.button_style = 'warning' if calc_psd_from_cos_theta==2 else 'primary'
    bt_psi_eq_om.button_style = 'warning' if calc_psd_from_cos_theta==1 else 'primary'
    w_psd.disabled = w_psd_ratio.disabled = calc_psd_from_cos_theta==2 and w_link.value

# ——————————————————————————————————————————————————————————————————————
def on_set_psi_omega_cos_theta(button):
    global psd_changing, calc_psd_from_cos_theta, xi_v
    psd_changing = True
    calc_psd_from_cos_theta = 2
    set_psi_omega_cos_theta_btn_color()
    if xi_v == 0:
        calc_theta_for_psi_cos_omega()
    else:
        calc_theta_phi_with_xi_cos()
    safe_clear_psd_flag(clear_psd_flag)
    show_nu_vector()
    #print(f"on_set_psi_omega_cos_theta: th_v={th_v} psid = {psd_v}")

# ——————————————————————————————————————————————————————————————————————
def on_set_psi_eq_omega(button):
    global psd_changing, calc_psd_from_cos_theta, om_B, phd_v, psd_v, psd_ratio
    psd_changing = True
    calc_psd_from_cos_theta = 1
    set_psi_omega_cos_theta_btn_color()
    w_phd.valRad = phd_v =  om_B
    w_psd.valRad = psd_v = -om_B
    w_psd_ratio.value = psd_ratio = 1.0
    if w_link.value:
        if xi_v == 0:
            calc_th_eq_simple()
        else:
            calc_theta_phi_with_xi()

    safe_clear_psd_flag(clear_psd_flag)
    show_nu_vector()
    #print(f"on_set_psi_eq_omega: {psd_v}")

# ——————————————————————————————————————————————————————————————————————
def radius_to_height_from_Iratio(Ir, Ia):
    """
    Compute R/H for a solid cylinder given the ratio Ir/Ia,
    where:
      Ir = I_radial = (1/12) m (3 R^2 + H^2)
      Ia = I_axial  = (1/2) m R^2

    Formula derived: (H/R)^2 = 6*(Ir/Ia) - 3, so
                     R/H     = 1 / sqrt(6*(Ir/Ia) - 3).

    Returns:
        R_over_H (float): the ratio R/H.

    Raises:
        ValueError: if (6*Ir/Ia - 3) <= 0 (no real solution).
    """
    ratio = Ir / Ia
    inside = 6 * ratio - 3
    if inside <= 0:
        raise ValueError(
            f"Invalid ratio Ir/Ia={ratio:.6g}: 6*(Ir/Ia) - 3 must be positive."
        )
    H_over_R = math.sqrt(inside)
    return 1.0 / H_over_R

def on_w_link_change(_):
    w_psd.disabled = w_psd_ratio.disabled = calc_psd_from_cos_theta==2 and w_link.value

# ——————————————————————————————————————————————————————————————————————
def scene_defaults():
    return "\'#40F070\', \'#4070FF\', 4, \'green\', \'#FF4040\', 1.3, 0.8, 24, 24, 1,1,1,0,1,(6,6)"

#=======================================================================================================================================#

w_om,      box_om      = LabeledFloatB.create(r'\(\omega_B\)',                 om_B,      1, -1e5, 1e5, None,
    'Angular velocity of the rotating field.')
w_I1,       box_I1       = LabeledFloat2(r'\(I_1\)',                           I1_v,      'kg m²', 1e-9, 0, 1e3,
    r'I₁ of a diagonal moment of intertia of the body (othogonal to the magnetic axis).')
w_I2,       box_I2       = LabeledFloat2(r'\(I_2\)',                           I2_v,      'kg m²', 1e-9, 0, 1e3,
    r'I₂} of a diagonal moment of intertia of the body (othogonal to the magnetic axis).')
w_I3,       box_I3       = LabeledFloat2(r'\(I_3\)',                           I3_v,      'kg m²', 1e-9, 0, 1e3,
    r'I₃ of a diagonal moment of intertia of the body (aligned with the magnetic axis).')
w_mB,      box_mB      = LabeledFloat2(r'\(mB\)',                              mB_v,      'Nm', 1e-9, 0, 1e3,
    'Product of strength of body magnetic moment and the rotating field.')
w_gam,     box_gam     = LabeledFloatA.create(r'\(\gamma\)',                   gam_v,      0.001,  -np.pi, np.pi,
    ('Elevation angle of the rotating field from the xy plane. This value is automatically calculated as a sum of an offset '
    'and the minimum \u03B3 angle for obtaining a solution with constant 𝜃.'))
w_gam_eps, box_eps_gam = LabeledFloatA.create(r'\(\gamma_{\Large \epsilon}\)', gam_eps, 0.0001, -np.pi/2, np.pi/2,
    'Manual offset value to set angle \u03B3')
w_th_eq,   box_th_eq   = LabeledFloatA.create(r'\(\theta_{\large eq}\)',       th_eq,     0.001,  -np.pi, np.pi,
    'Automatically calculated initial theta angle to keep 𝜃 constant.')
w_th_eps,  box_th_eps  = LabeledFloatA.create(r'\(\theta_{\Large \epsilon}\)', th_offs,   0.001,  -np.pi, np.pi,
    'An offset can be added to the angle 𝜃.')
w_th,      box_th0     = LabeledFloatA.create(r'\(\theta_{0}\)',                th_v,     0.001,  -np.pi, np.pi,
    'Angle 𝜃 initial value')
w_ph,      box_ph      = LabeledFloatA.create(r'\(\phi_{0}\)',                  ph_v,     0.0001,   -1e6, 1e6,
    'Angle 𝛷 initial value. It is typically close to 𝜋 as it corresponds to the phase lag of the motion.')
w_ps,      box_ps      = LabeledFloatA.create(r'\(\psi_{0}\)',                  ps_v,     0.001,   -1e6, 1e6,
    'Angle 𝛹 initial value.')
w_t_start, box_t_start = LabeledFloat2(r'\(t_0\)',                              t0_v,  's', 0.001,   0, 1e6,
    'Used to select state variable from a previous run and to define simulation start time (the offset of the time variable)')
w_thd,     box_thd     = LabeledFloatB.create(r'\(\dot\theta\)',                thd_v,     0.01,   -1e6, 1e6, None,
    'Angular velocity on rotation 𝜃')
w_phd,     box_phd     = LabeledFloatB.create(r'\(\dot\phi\)',                  om_B,      0.01,   -1e6, 1e6, None,
    'Angle 𝛷 initial velocity. For a solution with constant 𝜃, this is typically equal to velocity of the rotating field.')
w_psd,     box_psd     = LabeledFloatB.create(r'\(\dot\psi\)',                 -om_B,      0.1,   -1e6, 1e6, None,
    'Angle 𝛹 initial velocity. This can be set to -𝛷 dot to obtain a motion of the body which dont drag with respect to the lab frame.')
w_xi,     box_xi     = LabeledFloat2(r'\(\xi\)',                                xi_v,      '1/s',  0.01, -1e6,  1e6,
    'Angular viscous damping factor expressed as angular acceleration. It is converted to torque by multiplying by I₁ component of the m.o.i.')

w_link               = Checkbox(value=link, description='Link parameters for symmetry', layout=Layout(margin='0'), style={'description_width': '0'}, tooltip='')

w_ps_acc, box_ps_acc = LabeledFloatB.create(r'\(\ddot\psi\)',                  ps_acc,   0.1, -1e6, 1e6, ACC_UNITS, 'Acceleration applied to rotation  𝛹')

w_psd_ratio = FloatSlider(
    value = psd_ratio,
    min = -1.5, max=1.5, step=1e-16,
    readout=True,
    readout_format='.6f',
    description='',  # <-- drop it
    style={'description_width': '0'},
    layout=Layout(
        width='235px',
        margin='0 0 0 0',    # no extra outside space
        padding='0p'    # no extra inside space
    )
)

w_psd_ratio_om = HBox([Label(value='−𝜓/𝝎',layout=Layout(width='36px', margin='0'), tooltip='𝜅 = −𝜓/𝝎'), w_psd_ratio],layout=Layout(margin='0'))

bt_psi_om_cos_theta = Button(
    description='̇𝜓 = − 𝝎cos 𝜃',
    button_style='primary',      # 'primary', 'info', 'warning', 'danger' or ''
    tooltip='Set body angular velocity around its dipole axis equal to − 𝝎 cos 𝜃',
    layout=Layout(width='80px', height='22px', margin='0px 0 0 28px', padding='0 0'))
bt_psi_om_cos_theta.add_class('btn-center')

bt_psi_eq_om = Button(
    description='𝜑̇, −𝜓̇ = 𝜔',
    button_style='warning', #'success','primary', 'info', 'warning', 'danger' or ''
    tooltip='Set body angular velocity around its dipole axis equal to −𝝎',
    layout=Layout(width='80px', height='22px', margin='0px 0 0 5px', padding='0 0'))
bt_psi_eq_om.add_class('btn-center')

params_status1 = StatusLabel()
params_status2 = StatusLabel()

bt_psi_om_cos_theta.on_click(on_set_psi_omega_cos_theta)

bt_psi_eq_om.on_click(on_set_psi_eq_omega)

w_th_eq.observe(update_th_eq, names='value')
w_th.observe(update_th, names='value')
w_th_eps.observe(update_phd_psd_from_eps_theta, names='value')
for w in (w_om, w_I1, w_I2, w_I3, w_mB):
    w.observe(update_initials, names='value')
w_gam.observe(update_gamma_v, names='value')
w_gam_eps.observe(update_gamma_eps, names='value')
w_psd_ratio.observe(update_psd_from_ratio, names='value')
w_thd.observe(update_thd, names='value')
w_phd.observe(update_phd, names='value')
w_psd.observe(update_psd, names='value')
w_ph.observe(update_ph, names='value')

def on_set_phi_zero(button):
    global psd_changing, calc_psd_from_cos_theta, xi_v
    if psd_changing:
        return
    psd_changing = True
    xi_v = w_xi.value
    if xi_v == 0:
        w_ph.valRad = ph_v = np.pi
    calc_th_eq()
    safe_clear_psd_flag(clear_psd_flag)

def update_xi(change):
    if w_link.value:
        on_set_phi_zero(None)

# —————————————————————————————————————————————————————————————————————————————————————————————
def find_nearest_index_sorted(target_time: float) -> int:
    """
    Return index of t_vals closest to target_time.
    Assumes t_vals is sorted ascending.
    """
    #if target_time < t_vals[0] or target_time > t_vals[-1]:
    #    return None
    
    # insertion index
    i = np.searchsorted(t_vals, target_time)

    if i == 0:
        return 0
    if i >= len(t_vals):
        return len(t_vals) - 1
    # choose closer of neighbors
    return i if (t_vals[i] - target_time) < (target_time - t_vals[i - 1]) else i - 1

def on_load_instance(button):
    global psd_changing, show_nu, t_vals, y_vals, th_eq, th_offs, th_v, ph_v, ps_v, thd_v, phd_v, psd_v
    if t_vals is None or y_vals is None:
        return
    show_nu = False
    psd_changing = True
    w_link.value = False
    if button is bt_load_time_instance:
        idx = find_nearest_index_sorted(w_t_start.value)
        w_t_start.value = t_vals[idx] 
    else:
        idx = -1
        w_t_start.value = 0
        
    w_th_eps.valRad = th_offs = 0
    w_th_eq.valRad = th_eq = y_vals[0][idx]
    w_th.valRad = th_v = th_eq
    
    w_ph.valRad = ph_v = y_vals[1][idx]
    w_ps.valRad = ps_v = y_vals[2][idx]
    if ck_mod360_phi_ps.value:
        ph_v = ph_v % (np.pi*2)
        ps_v = ps_v % (np.pi*2)
    w_ph.valRad = ph_v
    w_ps.valRad = ps_v

    w_thd.valRad = thd_v = y_vals[3][idx]
    w_phd.valRad = phd_v = y_vals[4][idx]
    w_psd.valRad = psd_v = y_vals[5][idx]
    
    calc_psd_from_cos_theta = 0 # user defined
    set_psi_omega_cos_theta_btn_color()
    safe_clear_psd_flag(clear_psd_flag)
    show_nu_vector()
    show_nu = True
# —————————————————————————————————————————————————————————————————————————————————————————————
def act_mod360_phi_psi(x):
    on_load_instance(x)

# —————————————————————————————————————————————————————————————————————————————————————————————
bt_set_phi_zero = Button(
    description= 'φ₀ = 𝑓 ( ξ, ...)',
    button_style='primary',
    tooltip='Set initial φ angle using Eq.15 in section C.',
    layout=Layout(width='78px', height='22px', margin='2px 0 0 33px', padding='0 0'))
bt_set_phi_zero.add_class('btn-center')
# —————————————————————————————————————————————————————————————————————————————————————————————
bt_set_ext_psdd = Button(
    description= 'Set ext 𝜓˙˙',
    button_style='primary',
    tooltip=r'Set external accleration to keep the psd at it initial value.',
    layout=Layout(width='78px', height='22px', margin='2px 0 0 5px', padding='0 0'))
bt_set_ext_psdd.add_class('btn-center')
# —————————————————————————————————————————————————————————————————————————————————————————————
bt_load_time_instance = Button(
    description= 'Set IC from 𝑡₀',
    button_style='primary',
    tooltip=r'Set initial condition from state variables at time point 𝑡₀.',
    layout=Layout(width='90px', height='22px', margin='2px 0 0 0', padding='0 0'))
bt_load_time_instance.add_class('btn-center')

bt_load_from_end = Button(
    description= '…end',
    button_style='primary',
    tooltip=r'Set initial condition from state variables at the end of present simulation.',
    layout=Layout(width='40px', height='22px', margin='2px 0 0 4px', padding='0 0'))
bt_load_from_end.add_class('btn-center')

ck_mod360_phi_ps = Checkbox(
    value=True,
    description= '％',
    style={'description_width': '0px'},
    layout=Layout(width='45px', margin='0 0 0 4px', padding='0 0'),
    tooltip='Remove full turns from φ and 𝜓 values'
    )

# —————————————————————————————————————————————————————————————————————————————————————————————
w_xi.observe(update_xi, names='value')
w_link.observe(on_w_link_change)

bt_set_phi_zero.on_click(on_set_phi_zero)
bt_set_ext_psdd.on_click(on_set_ext_psdd)
bt_load_time_instance.on_click(on_load_instance)
bt_load_from_end.on_click(on_load_instance)
ck_mod360_phi_ps.observe(act_mod360_phi_psi, names='value')

# ——————————————————————————————————————————————————————————————————————————————————————
# ——————————————————————————————— define the Run section ———————————————————————————————
w_solver = Dropdown(
    options=['LSODA', 'DOP853','RK45', 'Radau'],
    value='LSODA',
    description='Solver',
    layout=Layout(width='150px', margin='0 7px 0 0'),
    style={'description_width': '57px'} )
w_solv_max_step, box_maxstep = LabeledFloat('Max Step',  2e-1,  'ms', 1e-2, 1e-5, 1,
        "Recommended maximum step is 0.5 ms or smaller. May be set to lower values for fast rotation velocities like 50 rev/s.")
w_solv_rel_tol, box_rel_tol = LabeledFloatK('R. Tol',  1e-10,  None,  1e-18, 1e-18, 1e-6,
        "1e-10 is OK, smaller values than this may not offer more accuracy.")
w_solv_abs_tol, box_abs_tol = LabeledFloatK('A. Tol',  1e-12,  None, 1e-18, 1e-18, 1e-6,
        "1e-12 is OK, smaller values than this may not offer more accuracy.")
w_simtime, box_simtime   = LabeledFloat('Sim Time',  sim_time,  's', 1, 0.001, 3600,
        "The time length of the physical process.")
w_step_per_sec, box_simsteps = LabeledFloat('Steps/s',  sim_steps,  '',  200, 1000, 1e5,
        "Values in 2000-10000 are OK. Large values slows down results for long simulations.")

_saved_anim_length = 0

def act_interpolate(change):
    global _saved_anim_length
    w_ani_len.disabled = not change['new']
    if w_ani_len.disabled:
        _saved_anim_length = w_ani_len.value
        update_anim_length(None)
    else:
        w_ani_len.value = _saved_anim_length

def update_anim_length(change):
    if w_interpolate.value or w_ani_fps.value == 0:
        return
    w_ani_len.disabled = False
    w_ani_len.value = (w_step_per_sec.value * w_ani_spa.value) / w_ani_fps.value
    w_ani_len.disabled = True

w_interpolate = Checkbox(value=True, description='Interpolate', layout=Layout(margin='0',width='106px'), style={'description_width': '0'})
w_interpolate.observe(act_interpolate, names='value')

btn_simulate = Button(
    description='Simulate',
    button_style='success',      # 'primary', 'info', 'warning', 'danger' or ''
    tooltip='Click to run simulation',
    icon='play',                 # font-awesome icon name
    layout=Layout(width='82px', height="26px", margin='0 0 0 2px', padding='0')
)

btn_load_sim = Button(
    description='Load Sim',
    button_style='info',      # 'primary', 'info', 'warning', 'danger' or ''
    tooltip='Load a saved simulation from CSV file',
    icon='folder-open',
    layout=Layout(width='82px', height="26px", margin='0 0 0 5px', padding='0')
)

# ——————————————————————————————— define the Animate section ———————————————————————————————
w_ani_beg, box_ani_beg   = LabeledFloat('Start',   0,  's',    0.1, 0, 4000,
    "The time in the physical process the animation will start")
w_ani_spa, box_ani_spa   = LabeledFloat('Span',    0,  's',    0.01, 0, 1000,
    "The time span of the physical process that animation will cover.")
w_ani_fps, box_ani_fps   = LabeledFloat('FPS',     30,  'fps', 1, 1,   2000,
    "20-30 FPS is OK for normal speed playback. Set higher rates if the video can be watched at slow motion rates.")
w_ani_len, box_ani_len   = LabeledFloat('Length',  30,  's',  1, 1, 600,
    "30 seconds is typical. Set longer duration if the covered physical process is more than 1 seconds.")
w_scene_opts =  Text(description='Options',
    placeholder='Enter a body specs (shape, color top, color bottom , sector count, light factor, dark factor, div Long, div latt, B vector, m vector, text antialiasing',
    value= scene_defaults(),
    layout=Layout(width='535px', margin='3px 5px 0 0'), style={'description_width': '58px'})

for widget in (w_step_per_sec, w_ani_spa, w_ani_fps):
    widget.observe(update_anim_length, names='value')

btn_animate = Button(
    description='Animate',
    button_style='success',
    tooltip='Click to create animation',
    icon='play',
    layout=Layout(width='82px', height="26px", margin='0 0 0 2px', padding='0')
)

btn_load_anim = Button(
    description='Load Anim',
    button_style='info',
    tooltip='Click to load an animation',
    icon='folder-open',
    layout=Layout(width='82px', height="26px", margin='0 0 0 5px', padding='0')
)

btn_del_anim = Button(
    description='Delete',
    button_style='danger',
    tooltip='Delete an animation from the project including the named mp4 file.',
    icon='',
    layout=Layout(width='45px', height="26px", margin='0 0 0 6px', padding='0')
)

btn_preview = Button(
    description='Scene',
    button_style='info',
    tooltip='Setup the animation scene',
    icon='picture-o',
    layout=Layout(width='82px', height="26px", margin='0 0 0 2px', padding='0')
)

btn_del_sim = Button(
    description='Delete',
    button_style='danger',
    tooltip='Delete a simulation result from project and the named csv file.',
    icon='',
    layout=Layout(width='45px', height="26px", margin='0 0 0 6px', padding='0')
)

lb_sts = Label(value='Status', layout=Layout(width='50px',  margin='0 0 0 20px'))
lb_result_sim = Label(value='', layout=Layout(width='535px', margin='0 2px 0 0'))
lb_result_ani = Label(value='', layout=Layout(width='535px', margin='0 2px 0 0'))
box_result_ani= HBox([lb_sts, lb_result_ani])

progress_ani = IntProgress(
    value=0, min=0, max=100,
    description='Progress',
    bar_style='info',
    layout=Layout(width='535px', height='18px', margin='5px 0 0 0',display='none'),  # hide initially
    style={'description_width': '57px'} )

# 1) helper to inline a PNG as data-URI
def img_data_uri(path):
    data = open(path, "rb").read()
    b64  = base64.b64encode(data).decode("ascii")
    return f"data:image/png;base64,{b64}"

# 2) your three images
xy_uri = img_data_uri("view_xy.png")
yz_uri = img_data_uri("view_yz.png")
xz_uri = img_data_uri("view_xz.png")

'''
def display_custom_video2(mp4_filename, width=600):
    html = f"""
    <video width="{width}" controls
           style="outline:none; border:none; /* no bg on the video itself */">
      <source src="{mp4_filename}" type="video/mp4">
      Your browser does not support the video tag.
    </video>

    <style>
      /* Enclosure around the controls (play button, scrub, etc.) */
      video::-webkit-media-controls-enclosure {{
        background: rgba(0,0,0,1) !important;
      }}
      /* The bottom control-panel strip */
      video::-webkit-media-controls-panel {{
        background: rgba(0,0,0,1) !important;
        min-height: 24px !important;
      }}
    </style>
    """
    display(HTML(html))

def display_custom_video(mp4_filename, width=600):
    html = f"""
    <video class="cleanvid" width="{width}" controls>
      <source src="{mp4_filename}" type="video/mp4">
      Your browser does not support the video tag.
    </video>
    <style>
      /* The overall container around the controls */
      .cleanvid::-webkit-media-controls-enclosure {{
        background-color: black !important;
      }}
      /* The panel that holds the buttons, progress bar, etc. */
      .cleanvid::-webkit-media-controls-panel {{
        background-color: black !important;
        min-height: 24px !important;
      }}
      /* The “background” gradient strip behind the progress bar */
      .cleanvid::-webkit-media-controls-background {{
        background: black !important;
      }}
      /* Remove the semi-transparent scrubber overlay on hover */
      .cleanvid::-webkit-media-controls-overlay-play-button {{
        display: none !important;
      }}
    </style>
    """
    display(HTML(html))
'''
# —————————————————————————————————————————————————————————————————————————————————————————————
# Create video animation
def setup_player(exec_time):
    with video_out:
        clear_output(wait=True)
        html = """
<!-- single toggle button; initially shows ▶️ -->
<button id="playPauseBtn"
        onclick="
            if (v.paused || v.ended) {{
                v.play();
                this.textContent = '⏸️';
            }} else {{
                v.pause();
                this.textContent = '⏯️';
            }}
        "
        style="font-size: 20px; padding: 2px 8px; height: 24px; line-height: 0;
            vertical-align: middle; border: 1px solid blue; border-radius: 5px; background-color:#0078D7;"
>▶️</button>

<button onclick="
    v.pause();
    v.currentTime = 0;
    document.getElementById('playPauseBtn').textContent = '▶️';"
    style="font-size: 20px; padding: 2px 8px; height: 24px; line-height: 0;
        vertical-align: middle; border: 1px solid blue; border-radius: 5px; background-color:#0078D7;"
>⏹️</button>
&nbsp;<input type="range" id="seekBar" value="0" min="0" step="0.005" style="width: 400px; vertical-align: middle;">
&nbsp;<span id="timeDisplay" style="font-size: 16px">00:00.000</span>&nbsp <!-- ;Speed: -->
<select onchange="v.playbackRate=this.value">
  <option value="0.0625">1/16 ×</option>
  <option value="0.125">1/8 ×</option>
  <option value="0.25">1/4 ×</option>
  <option value="0.5">1/2 ×</option>
  <option value="1.0" selected>1.0 ×</option>
  <option value="1.5">1.5 ×</option>
  <option value="2.0">2.0 ×</option>
  <option value="4.0">4.0 ×</option>
</select>

&nbsp;<span style="font-weight:bold; margin-right:8px;">{filename}</span>
<br>
<video id="v1">
  <source src="{fp}" type="video/mp4">
  Your browser does not support video.
</video>

<script>
function fmtTime(sec) {{
    const m = String(Math.floor(sec/60)).padStart(2,'0');
    let s_ms = (sec % 60).toFixed(3);
    if (s_ms.length < 6) s_ms = s_ms.padStart(6, '0');
    return m + ':' + s_ms;
}}

var v = document.getElementById('v1');
var disp = document.getElementById('timeDisplay');
var btn = document.getElementById('playPauseBtn');

// update time display on every timeupdate
v.addEventListener('timeupdate', function() {{
    disp.textContent = fmtTime(v.currentTime);
}});

// when video ends, reset button back to ▶️
v.addEventListener('ended', function() {{
    btn.textContent = '▶️';
}});
//---------------------------------------
var seekBar = document.getElementById('seekBar');

// Update slider max when video metadata is loaded
v.addEventListener('loadedmetadata', function() {{ seekBar.max = v.duration; }});

// Update slider as video plays
v.addEventListener('timeupdate', function() {{ seekBar.value = v.currentTime; disp.textContent = fmtTime(v.currentTime); }});

// Seek video when slider is changed
seekBar.addEventListener('input', function() {{ v.currentTime = seekBar.value; }});
</script>
""".format(fp=mp4_filename, filename=os.path.basename(mp4_filename))
        display(HTML(html))

        if exec_time:
            lb_result_ani.value = f'Complete ({exec_time:0.2f} secs) {mp4_filename}.'
        else:
            lb_result_ani.value = f'MP4 file {mp4_filename} is ready for playback.'
# —————————————————————————————————————————————————————————————————————————————————————————————
def generate_animation(button):
    global  mp4_filename, csv_filename, t_vals, y_vals
    anim_res = None
    exec_time = 0

    def worker():
        nonlocal exec_time, anim_res
        exec_time = 0
        #——————————————————————————————————————
        def generate_anim():
            global  mp4_filename, csv_filename, t_vals, y_vals

            def show_animation_progress(frame, totframes):
                progress_ani.value = int((frame/totframes)*100)

            progress_ani.value = 0
            params = get_current_params()
            s = f"({w_scene_opts.value},{w_I1.value/w_I3.value},{w_I2.value/w_I3.value})"
            #print("csv_filename=",csv_filename)
            try:
                scene_specs = ast.literal_eval(s)
                mp4_filename, exec_time = AnimateSim(None,
                    "", #"Rigid Body Dynamics Animation", # title
                    (t_vals, y_vals),
                    csv_filename,                    # path_to_sim_csv
                    f'ani-{id_hash}',                # mp4 file prefix (default 'ani')
                    params,
                    scene_specs,
                    vid_Length = w_ani_len.value if w_interpolate.value else -1,
                    fps=w_ani_fps.value,
                    t_start=w_ani_beg.value,
                    t_span = w_ani_spa.value,
                    callback=show_animation_progress,
                    interval=10
                    )
            except Exception as e:
                error_details = traceback.format_exc()
                print(f"Exception: {error_details}")
                traceback.print_exc()  # prints the full traceback to the console
                return str(e)
            #finally:
            #    anim_done_event.set()

            #print(mp4_filename, exec_time)
            return None

        #——————————————————————————————————
        anim_res = generate_anim()

    #——————————————————————————————————————
    def ui_update():
        nonlocal anim_res, exec_time # , pc
        #if not anim_done_event.is_set():
        #    return
        #print("************* anim done **************")
        #pc.stop()
        if anim_res:  #failed
            lb_result_ani.value = f"Animation failed: {anim_res}"
        else:
            if mp4_filename:
                setup_player(exec_time)

            if current_rec_key:
                if not mp4_files:
                    update_file_list(current_rec_key, 'mp4_files', mp4_files, mp4_filename, 'a') # add
                else:
                    anim_save_dlg.show()
                #update_record_subkey(current_rec_key, 'mp4_filename', mp4_filename)
            else:
                pass
        progress_ani.layout.display = 'none'
        box_result_ani.layout.display = 'flex'
        btn_animate.disabled = False
        btn_animate.button_style = 'success'

    #——————————————————————————————————————
    if not csv_filename and t_vals is None: # nothing to animate
        lb_result_ani.value = 'PLease load or generate a simulation result first.'
        return

    lb_result_ani.value = ''
    if w_ani_spa.value==0 and 0 < w_simtime.value - w_ani_beg.value <= 2:
        w_ani_spa.value = w_simtime.value - w_ani_beg.value

    if w_ani_spa.value==0 or w_ani_beg.value + w_ani_spa.value > w_simtime.value:
        lb_result_ani.value = "Animation duration is either zero or it pass the simulation time."
        return

    btn_animate.disabled = True
    btn_animate.button_style = 'warning'
    progress_ani.layout.display = 'flex'
    box_result_ani.layout.display = 'none'

    if mp4_filename and mp4_filename.startswith("RESULTS/TEMP/") and os.path.exists(mp4_filename):
        try:
            os.remove(mp4_filename)
        except FileNotFoundError:
            pass  # already gone

    mp4_filename = None

    #non threaded version
    worker()
    ui_update()

    #thread version
    #anim_done_event.clear()
    #threading.Thread(target=worker, daemon=True).start()
    #pc = PeriodicCallback(ui_update, 250)
    #pc.start()

# ————————————————————————————————————————————————————————————————————————
def get_tags_with_ffmpeg_python(mp4_file):
    probe = ffmpeg.probe(mp4_file)
    tags = probe.get('format', {}).get('tags', {})
    return tags.get('csv'), tags.get('begin'), tags.get('span')

# ——————————————————————————————————————————————————————————————————————
def on_csv_selected(filename):
    global csv_filename
    csv_filename = filename
    if load_simulation_results2():
        accor_plots.selected_index = 0
    else:
        show_nu_vector()
        setup_scene(None)

# ——————————————————————————————————————————————————————————————————————
def on_mp4_selected(filename):
    global mp4_filename
    mp4_filename = filename
    load_animation2()

# ——————————————————————————————————————————————————————————————————————
def on_csv_delete(filename):
    global csv_files, csv_filename
    if csv_filename == filename:
        plots_out.clear_output()
    update_file_list(current_rec_key, 'csv_files', csv_files, filename, 'd')

# ——————————————————————————————————————————————————————————————————————
def on_mp4_delete(filename):
    global mp4_files, mp4_filename
    update_file_list(current_rec_key, 'mp4_files', mp4_files, filename, 'd')

# ——————————————————————————————————————————————————————————————————————
def on_csv_clean():
    global csv_files
    update_file_list(current_rec_key, 'csv_files', csv_files, None, 'c')
    return csv_files

# ——————————————————————————————————————————————————————————————————————
def on_mp4_clean():
    global mp4_files
    update_file_list(current_rec_key, 'mp4_files', mp4_files, None, 'c')
    return mp4_files

# ————————————————————————————————————————————————————————————————————————
def load_animation(button):
    global mp4_files, mp4_filename
    if mp4_files:
        if len(mp4_files)==1:
            mp4_filename = mp4_files[0]
        else:
            mp4_dialog.show_select(mp4_files)
            return
    load_animation2()

# ————————————————————————————————————————————————————————————————————————
def update_csv_count():
    w_csv_count.value = f"<span style=\" font-weight: 500\"> {len(csv_files) or (1 if csv_filename else 0)} item(s)</span>"

# ————————————————————————————————————————————————————————————————————————
def update_mp4_count():
    w_mp4_count.value = f"<span style=\" font-weight: 500\"> {len(mp4_files) or (1 if mp4_filename else 0)} item(s)</span>"

# ————————————————————————————————————————————————————————————————————————
def load_animation2():
    global mp4_filename
    if mp4_filename and os.path.exists(mp4_filename):
        lb_result_ani.value ="Loading animation..."
        try:
            csv, beg, spa = get_tags_with_ffmpeg_python(mp4_filename)
            if beg:
                w_ani_beg.value = float(beg)
            if spa:
                w_ani_spa.value = float(spa)
            setup_player(0)
        except Exception as e:
            lb_result_ani.value = f'Error occured reading {mp4_file}: {e.message}.'
            return
    else:
        lb_result_ani.value ="No animation available."

# ————————————————————————————————————————————————————————————————————————
def setup_scene(button):
    if button == btn_preview:
        accor_plots.selected_index = None

    accor_scene.selected_index = 0

    params = get_current_params()

    try:
        tt = ast.literal_eval(w_scene_opts.value)
        if not (isinstance(tt, tuple) and len(tt) == 15):
            raise ValueError
        mcolors.to_rgba(tt[0])
        mcolors.to_rgba(tt[1])
        mcolors.to_rgba(tt[3])
        mcolors.to_rgba(tt[4])
        lb_result_ani.value = ""

    except Exception as e:
        w_scene_opts.value = scene_defaults()
        lb_result_ani.value =f"Scene configurion is invalid ({e}), setting to defaults. Adjust if needed and click the Scene button."

    x = w_I1.value/w_I3.value if w_I3.value else 50
    s = f"({w_scene_opts.value},{w_I1.value/w_I3.value},{w_I2.value/w_I3.value})"
    try:
        scene_specs = ast.literal_eval(s)
        # Vid_Length is how long you want the video to be, the longer the video the slower the FM is animated (slow motion).
        # fps is frames per second, recommended 30 or more
        # t_start is where in the simulation output time step you want to start at, for instance start at the begging t_start=0, this should be < total sim time in csv
        # t_span is the real time interval you want to sample from the CSV. If you want to see 1 second of FM Body movement in real time, it will be spread out over the "Vid_Length" time you chose.
        mp4_filename, exec_time = AnimateSim(
            video_out, # present only for displaying scene setup
            '', # title
            None,
            csv_filename,
            None,
            params,
            scene_specs,
            vid_Length=w_ani_len.value,
            fps=w_ani_fps.value,
            t_start=w_ani_beg.value,
            t_span = w_ani_spa.value,
            callback=None,
            interval=0
            )
    except Exception as e:
        lb_result_ani.value = f"Scene creation failed: {str(e)}"
        error_details = traceback.format_exc()
        print(f"Exception: {error_details}")
        traceback.print_exc()  # prints the full traceback to the console
        #video_out.clear_output()

# ——————————————————————————————————————————————————————————————————————
def load_simulation_results(button):
    global csv_files, csv_filename, t_vals, y_vals
    t_vals = y_vals = None
    loaded = False
    # select csv_filename from csv_files
    if csv_files:
        if len(csv_files)==1:
            csv_filename = csv_files[0]
        else:
            csv_dialog.show_select(csv_files)
            return
    #print ("csv_filename=",csv_filename)
    if csv_filename:
        loaded = load_simulation_results2()
    else:
        lb_result_sim.value = 'No simulation results available.'
        plt.close('all')
        for out in plots_out:
            out.clear_output()        

    update_csv_count()
    if loaded:
        accor_plots.selected_index = 0
    #else:
    #    show_nu_vector()

# ——————————————————————————————————————————————————————————————————————
def load_simulation_results2() -> bool:
    global psd_changing, csv_filename, box_polar_plot, t_vals, y_vals, T_vals, V_vals
    global I1_v, I2_v, I3_v, mB_v, om_B, gam_v, xi_v
    
    T_vals = V_vals = None
    if not csv_filename or not  os.path.exists(csv_filename):
        lb_result_sim.value = f"Simulation file  {csv_filename} cannot be found."
        return False
    try:
        sim_params = None
        with open(csv_filename) as f:
            first_line = f.readline()
            if first_line.startswith("# physics_params ="):
                try:   
                    sim_params = json.loads(first_line.split("=", 1)[1])        
                except:
                    pass

        df = pd.read_csv(csv_filename, comment='#').apply(pd.to_numeric, errors='coerce')                
        t_vals  = df['t'].to_numpy()
        th_vals = df['theta'].to_numpy()
        ph_vals = df['phi'].to_numpy()
        ps_vals = df['psi'].to_numpy()
        th_dots = df['theta_dot'].to_numpy()
        ph_dots = df['phi_dot'].to_numpy()
        ps_dots = df['psi_dot'].to_numpy()

        if sim_params:
            psd_changing = True
            get_physics_params(sim_params)
            psd_changing = False
        
        #sim_beg =   t_vals[0]
        sim_end =   t_vals[-1]
        w_simtime.value = sim_end
        #params = get_current_params()
        params = I1_v, I2_v, I3_v, mB_v, om_B, gam_v, xi_v*(I1_v+I2_v)*0.5

        y_vals = th_vals, ph_vals, ps_vals, th_dots, ph_dots, ps_dots
        
        plot_sim_results(t_vals, th_vals, ph_vals, ps_vals, th_dots, ph_dots, ps_dots, params, -1)
        #===============================
        show_nu_vector()
        lb_result_sim.value = f"Simulation file {csv_filename} is available."
        return True
        #w_ani_beg.value = sim_beg
        #w_ani_spa.value = sim_end - sim_beg
        #display_physics_params() !test! keep commented
        #setup_scene(button)
    except Exception as e:
        lb_result_sim.value = f"Error on reading CSV file: {str(e)}"
        return False


# ——————————————————————————————————————————————————————————————————————
def show_delete_csv_dlg(button):
    global csv_filename
    lb_result_sim.value = ''
    if csv_files:
        csv_dialog.show_delete(csv_files)
    elif  csv_filename:
        update_file_list(current_rec_key, 'csv_files', csv_files, csv_filename, 'd')

# ——————————————————————————————————————————————————————————————————————
def show_delete_mp4_dlg(button):
    global mp4_filename
    lb_result_ani.value = ''
    if mp4_files:
        mp4_dialog.show_delete(mp4_files)
    elif mp4_filename:
        update_file_list(current_rec_key, 'mp4_files', mp4_files, mp4_filename, 'd')

# ——————————————————————————————————————————————————————————————————————
btn_load_sim.on_click(load_simulation_results)
btn_animate.on_click(generate_animation)
btn_load_anim.on_click(load_animation)
btn_preview.on_click(setup_scene)
btn_del_anim.on_click(show_delete_mp4_dlg)
btn_del_sim.on_click(show_delete_csv_dlg)

#=======================================================================================

# 1) File where we store parameter sets
STORE_FILE = "params_store.json"

# ——————————————————————————————————————————————————————————————————————
def load_store():
    if os.path.exists(STORE_FILE):
        with open(STORE_FILE, 'r') as f:
            return json.load(f)
    return {}

# ——————————————————————————————————————————————————————————————————————
def save_store(data):
    with open(STORE_FILE, 'w') as f:
        json.dump(data, f, indent=2)

# ——————————————————————————————————————————————————————————————————————
PIN_CHAR = '*'
SEP = '\u2013'

def update_restore_options(_=None):
    store = load_store()
    pinned = []
    rest   = []

    for k in store.keys():
        # split into [datepart] — [notes]
        if SEP in k:
            _, note = k.split(SEP, 1)
            note = note.strip()
        else:
            note = ''

        # test for leading or trailing asterisk
        if note.startswith(PIN_CHAR) or note.endswith(PIN_CHAR):
            pinned.append(k)
        else:
            rest.append(k)

    # sort each group as you like (here reverse date order)
    pinned = sorted(pinned, reverse=True)
    rest   = sorted(rest,   reverse=True)

    # combine: pinned keys always on top
    restore_dropdown.options = pinned + rest

#def on_list_dblclick(event):
#    print("on_list_dblclick")
#    on_restore(event)

def get_physics_params(rec):
    global om_B, I1_v, I2_v, I3_v, mB_v, gam_v, gam_min, gam_eps, xi_v, th_eq, th_offs, th_v, ph_v, ps_v, thd_v, phd_v, psd_v, psd_ratio, ps_acc
    w_om.valRad =	     om_B    = rec.get('om',           0)
    w_I1.value =	     I1_v    = rec.get('I1',           1e-8)
    w_I2.value =	     I2_v    = rec.get('I2',           1e-8)
    w_I3.value =	     I3_v    = rec.get('I3',           1e-8)
    w_mB.value =	     mB_v    = rec.get('mB',           0)
    w_gam.valRad =	     gam_v   = rec.get('gam',          0)
    w_gam_eps.valRad =   gam_eps = rec.get('eps_gam',      0)
    w_th_eq.valRad =	 th_eq   = rec.get('th_eq',        1e-9)
    w_th_eps.valRad =    th_offs = rec.get('th_eps',       0)
    w_th.valRad =	     th_v    = rec.get('th',           1e-9)
    w_ph.valRad =	     ph_v    = rec.get('ph',           np.pi)
    w_ps.valRad =	     ps_v    = rec.get('ps',           0)
    w_thd.valRad =	     thd_v   = rec.get('thd',          0)
    w_phd.valRad =	     phd_v   = rec.get('phd',          0)
    w_psd.valRad =	     psd_v   = rec.get('psd',          0)
    w_xi.value =	     xi_v    = rec.get('xi',           0)
    w_ps_acc.valRad =    ps_acc  = rec.get('ps_acc',       0)
    w_psd_ratio.value =  psd_ratio = rec.get('psd_ratio',  1)

# ——————————————————————————————————————————————————————————————————————
def physics_params_dict():
    return {
        'om':            w_om.valRad,
        'I1':            w_I1.value,
        'I2':            w_I2.value,
        'I3':            w_I3.value,
        'mB':            w_mB.value,
        'gam':           w_gam.valRad,
        'eps_gam':       w_gam_eps.valRad,
        'th_eq':         w_th_eq.valRad,
        'th_eps':        w_th_eps.valRad,
        'th':            w_th.valRad,
        'ph':            w_ph.valRad,
        'ps':            w_ps.valRad,
        'thd':           w_thd.valRad,
        'phd':           w_phd.valRad,
        'psd':           w_psd.valRad,
        'xi':            w_xi.value,
        'ps_acc':        w_ps_acc.valRad,
        'psd_ratio':     w_psd_ratio.value,
        'psd_from_cos_theta': calc_psd_from_cos_theta,
    }
    
# ——————————————————————————————————————————————————————————————————————
def load_parameters(rec):
    global psd_changing, id_hash, csv_files, mp4_files, csv_filename, mp4_filename, calc_psd_from_cos_theta, current_rec_key, DEGREE_MODE, deg_mode_action_suppress, portrait_params
    w_link.value = False
    psd_changing = True
    id_hash =                rec.get('id_hash','0')
    calc_psd_from_cos_theta= rec.get('psd_from_cos_theta', calc_psd_from_cos_theta)
    set_psi_omega_cos_theta_btn_color()

    get_physics_params(rec)
    
    #w_om_nat.valRad =         rec.get('om_nat',       0)
    w_solver.value =         rec.get('solv_method',  'LSODA')
    w_solv_max_step.value =  rec.get('solv_max_stp', w_solv_max_step.value)*1e3
    w_solv_rel_tol.value =   rec.get('solv_rel_tol', w_solv_rel_tol.value)
    w_solv_abs_tol.value =   rec.get('solv_abs_tol', w_solv_abs_tol.value)
    w_step_per_sec.value =   rec.get('sim_steps',    w_step_per_sec.value)
    w_simtime.value =        rec.get('sim_time',     w_simtime.value)
    w_scene_opts.value =     rec.get('body_colors',  w_scene_opts.value)
    w_ani_beg.value =        rec.get('anim_start',   w_ani_beg.value)
    w_ani_spa.value =        rec.get('anim_span',    w_ani_spa.value)
    w_interpolate.value =    rec.get('anim_interpol',w_interpolate.value)
    SetScale                (rec.get('body_scale',   1))
    SetShape                (rec.get('body_shape',   GetShape()))
    SetViewAngles           (rec.get('view_angles',  (10,-60)))
    csv_files =              rec.get('csv_files',    [])
    mp4_files =              rec.get('mp4_files',    [])

    sweep_param_dd.value =   rec.get('sweep_name','gamma')
    beg_range_box.value =    rec.get('sweep_beg',-10)
    end_range_box.value =    rec.get('sweep_end', 10)
    step_box.value =         rec.get('sweep_steps', 101)
    plots_view =             rec.get('plot_view',None)
    phase_portr_dd.value =   rec.get('phase_portr',0)
    portrait_params =        rec.get('portr_params', None)

    if plots_view is not None and len(plots_view) ==len(ck_plots):
        for ck, val in zip(ck_plots, plots_view):
            ck.value = bool(int(val))
  
    w_link.value =	         rec.get('link',         w_link.value)
    #deg_mode_action_suppress = True
    DEGREE_MODE =            rec.get('degree_mode',  DEGREE_MODE)
    #deg_mode_action_suppress = False

    # backward compatibility -----
    if not csv_files:
        csv_filename =       rec.get('csv_filename', '')
    if not mp4_files:
        mp4_filename =       rec.get('mp4_filename', '')
    #-----------------------------

    psd_changing = False
    update_id_hash()

# ——————————————————————————————————————————————————————————————————————
def clear_eigen_results():
    eigen_resL.clear_output()
    eigen_resR.clear_output()
    eig_range_scale(0)
    eigen_sts_res.value = 'Ready'

def on_restore(button):
    global DEGREE_MODE, deg_mode_action_suppress, psd_changing, current_rec_key, csv_files, csv_filename, t_vals, y_vals
    lb_result_sim.value = ''
    key = restore_dropdown.value
    if not key:
        return
    csv_files=[]
    csv_filename = mp4_filename = ''
    t_vals = y_vals = None
    #plots_out.clear_output()
    clear_eigen_results()
    rec = load_store().get(key, {})
    load_parameters(rec)

    current_rec_key = key

    s = key.split("\u2013\u2009")
    if len(s)==1:
        s = key.split("\u2014 ")
       
    if len(s)>1:
        note_w.value = s[1]

    with out:
        clear_output()

    deg_mode_action_suppress = True
    unit_btn.value = 'Deg' if DEGREE_MODE else 'Rad'
    set_RadDeg_mode(DEGREE_MODE, False)
    deg_mode_action_suppress = False

    update_csv_count()
    update_mp4_count()
    lb_result_sim.value ='Solution parameters restored.'

    if csv_files:
        if len(csv_files)==1:
            csv_filename = csv_files[0]
        else:
            csv_dialog.show_select(csv_files)
            return
    elif csv_filename:
        csv_files.append(csv_filename)
        update_csv_count()

    #print ("csv_filename=",csv_filename)
    if csv_filename:
        lb_result_sim.value ='Loading simulation results...'
        if load_simulation_results2():
            accor_plots.selected_index = 0
        else:
            show_nu_vector()
        setup_scene(button)
    else:
        lb_result_sim.value = 'Ready'

    #print("pony 2")

# ——————————————————————————————————————————————————————————————————————
def get_current_params():
    global csv_files, mp4_files, id_hash, DEGREE_MODE, portrait_params
    return {
        'id_hash':       id_hash,
        'om':            w_om.valRad,
        'I1':            w_I1.value,
        'I2':            w_I2.value,
        'I3':            w_I3.value,
        'mB':            w_mB.value,
        'gam':           w_gam.valRad,
        'eps_gam':       w_gam_eps.valRad,
        'th_eq':         w_th_eq.valRad,
        'th_eps':        w_th_eps.valRad,
        'th':            w_th.valRad,
        'ph':            w_ph.valRad,
        'ps':            w_ps.valRad,
        'thd':           w_thd.valRad,
        'phd':           w_phd.valRad,
        'psd':           w_psd.valRad,
        'xi':            w_xi.value,
        'ps_acc':        w_ps_acc.valRad,
        'psd_ratio':     w_psd_ratio.value,
        'psd_from_cos_theta': calc_psd_from_cos_theta,
        #'om_nat':        w_om_nat.valRad,
        'link':          w_link.value,
        'solv_method':   w_solver.value,
        'solv_max_stp':  w_solv_max_step.value*1e-3,
        'solv_rel_tol':  w_solv_rel_tol.value,
        'solv_abs_tol':  w_solv_abs_tol.value,
        'sim_steps':     w_step_per_sec.value,
        'sim_time':      w_simtime.value,
        'body_colors':   w_scene_opts.value,
        'body_scale':    GetScale(),
        'body_shape':    GetShape(),
        'view_angles':   GetViewAngles(),
        'anim_start':    w_ani_beg.value,
        'anim_span':     w_ani_spa.value,
        'anim_interpol': w_interpolate.value,
        'csv_files':     csv_files,
        'mp4_files':     mp4_files,
        'degree_mode':   DEGREE_MODE,
        'sweep_name':    sweep_param_dd.value,
        'sweep_beg':     beg_range_box.value,
        'sweep_end':     end_range_box.value,
        'sweep_steps':   step_box.value,
        'sweep_cos':     calc_psd_from_cos_theta,
        'plot_view':     [int(ck.value) for ck in ck_plots],
        'phase_portr':   phase_portr_dd.value,
        'portr_params':  portrait_params
    }

# ——————————————————————————————————————————————————————————————————————
# save current simulation parameters by optionally remove old entries:
def on_save(_):
    global current_rec_key, id_hash, csv_files, mp4_files, mp4_filename, csv_filename, plot_portraits, portrait_params
    store = load_store()
    note = note_w.value or 'no note'
    if current_rec_key: # in case the project is loaded
        rec_note = current_rec_key.split(" - ", 1)[-1]
        if rec_note == note:
            rec = store.get(current_rec_key, None)
            #rec = store.setdefault(current_rec_key, {})
            if rec:
                # Pull out any existing lists you want to preserve
                csv_files = rec.get('csv_files', [])
                mp4_files = rec.get('mp4_files', [])
        else:
            current_rec_key = None

    if not current_rec_key:
        ts   = time.strftime("%Y%m%d-%H%M")
        current_rec_key  = f"{ts}\u2009\u2013\u2009{note}"
        # now insert the new one
        b = abs(hash(current_rec_key)).to_bytes(8, 'big')
        id_hash = base64.b32encode(b).decode('utf-8')[:10]  #.replace('=', '')

    if not csv_filename and y_vals is not None:
        csv_flag = write_csv()
    else:
        csv_flag= True

    if csv_flag:
        if csv_filename and csv_filename not in csv_files:
            csv_files.append(csv_filename)

    if mp4_filename:
        if mp4_filename.startswith("RESULTS/TEMP/") and os.path.exists(mp4_filename):
            dst_path = mp4_filename.replace("RESULTS/TEMP/", "RESULTS/").replace("-0-", f"-{id_hash}-")
            try:
                shutil.move(mp4_filename, dst_path)
                mp4_filename = dst_path
            except FileNotFoundError:
                mp4_filename = None

        if mp4_filename not in mp4_files:
           mp4_files.append(mp4_filename)

    if plot_portraits:
        portrait_params = plot_portraits.get_params()
        
    store[current_rec_key] = get_current_params()
    save_store(store)
    save_sweep_params()

    update_restore_options()
    update_id_hash()
    update_csv_count()
    update_mp4_count()
    with out:
        clear_output()

# ——————————————————————————————————————————————————————————————————————
def on_delete_params(button):
    key = restore_dropdown.value
    if not key:
        return
    store =  load_store()
    if key in store:
        store.pop(key)
        save_store(store)
        update_restore_options()
        update_id_hash()
        update_csv_count()
        update_mp4_count()
        copy_status.value = f"<span style=''>Entry {key} deleted from projects</span>"
    with out:
        clear_output()
        #print(f"The parameter setup '{key}' is deleted.")

# ——————————————————————————————————————————————————————————————————————
def on_params_copy(_):
    global current_rec_key
    if not current_rec_key:
        note = note_w.value or 'no note'
        ts   = time.strftime("%Y%m%d-%H%M")
        current_rec_key  = f"{ts} — {note}"
    param_record = {current_rec_key: get_current_params()}
    s = json.dumps(param_record, indent=2)
    pyperclip.copy(s)

# ——————————————————————————————————————————————————————————————————————
#clipboard_content = None

#def get_clipboard():
#    """Fetch clipboard content using JS into Python variable."""
#    code = """
#    navigator.clipboard.readText().then(text => {
#        IPython.notebook.kernel.execute("clipboard_content = `" + text.replace(/`/g, '\\`') + "`");
#    });
#    """
#    display(Javascript(code))

# ——————————————————————————————————————————————————————————————————————
def on_import_params(b):
    #global clipboard_content
    copy_status.value = "<span style='color:gray'>Reading clipboard...</span>"
    clipboard_content = pyperclip.paste()

     # Wait and then validate
    time.sleep(0.2)  # allow time for JS clipboard to populate variable
    if not isinstance(clipboard_content, str):
        copy_status.value = "<span style='color:red'>✖ Clipboard is not text</span>"
        return
    text = clipboard_content.replace('\u00A0', ' ')

    try:
        param_record = json.loads(text)
    except Exception as e:
        copy_status.value = f"<span style='color:red'>✖ JSON parse error: {e}</span>"
        return
    # Success!
    current_rec_key = next(iter(param_record))
    params = param_record[current_rec_key]
    load_parameters(params)

    s = current_rec_key.split("— ")
    if len(s)>1:
        note_w.value = s[1]

    #imported_params.append(obj)
    display_physics_params()
    if load_simulation_results2(): #if any
        accor_plots.selected_index = 0
    else:    
        show_nu_vector()

    setup_scene(b)
    copy_status.value = f"<span style='color:green'>✔ Parameters \"{note_w.value}\" pasted successfully. Please save it.</span>"

# ——————————————————————————————————————————————————————————————————————

def on_clear_proj(button):
    global current_rec_key, id_hash, csv_filename, mp4_filename, csv_files, mp4_files, calc_psd_from_cos_theta
    global om_B, I1_v, I2_v, I3_v, mB_v, gamma_v, gam_min, gam_eps, xi_v, th_eq, th_offs, th_v,  ph_v, ps_v, xi_v, thd_v, phd_v, psd_v, psd_ratio, ps_acc
    csv_files.clear()
    mp4_files.clear()
    current_rec_key = csv_filename = mp4_filename = None
    id_hash = '0'
    note_w.value = ''
    params_status1.set('')
    params_status2.set('')
    lb_result_sim.value = lb_result_ani.value = ''

    w_link.value = False
    psd_changing = True
    if button is clear_proj_btn:
        calc_psd_from_cos_theta = 1 # psd = -omega
        w_om.valRad = 0
        w_I1.value = 1e-8
        w_I2.value = 1e-8
        w_I3.value = 1e-8
        w_mB.value = 0
        w_gam.valRad = 0
        w_gam_eps.valRad = 0
        w_th_eq.valRad =	1e-9
        w_th.valRad = 1e-9
        w_th_eps.valRad = 0
        w_ph.valRad = 0
        w_ps.valRad = 0
        w_thd.valRad = 0
        w_phd.valRad = 0
        w_psd.valRad = 0
        w_xi.value = 0
        w_ps_acc.valRad = 0
        w_psd_ratio.value = 1
    else: #load default parameters
        calc_psd_from_cos_theta = 1 # psd = -omega
        w_om.valRad = om_B = 100*np.pi
        w_I1.value = I1_v = 1e-7
        w_I2.value = I2_v = 1e-7
        w_I3.value = I3_v = 1e-7
        w_mB.value = mB_v = 0.003
        w_gam_eps.valRad = gam_eps = 0.01
 
        #w_th_eq.valRad =	1e-9
        #w_th.valRad = 1e-9
        w_ph.valRad = ph_v = np.pi
        w_ps.valRad = ps_v = 0
        w_thd.valRad = thd_v = 0
        w_psd_ratio.value = psd_ratio = 1
        w_phd.valRad = phd_v = om_B
        w_psd.valRad = psd_v = -om_B 
        w_xi.value = xi_v = 0
        w_ps_acc.valRad = ps_acc = 0
        calc_gamma_min()
        w_gam.valRad = gam_v = gam_min + gam_eps
        w_th_eps.valRad = th_offs = 0
        calc_th_eq()
        th_v = th_eq + th_offs
        w_th.valRad = th_v
        show_nu_vector()
    
    set_psi_omega_cos_theta_btn_color()
    #w_solver.value = 'LSODA
    #w_solv_max_step.value = 0.0005
    #w_solv_rel_tol.value =
    #w_solv_abs_tol.value =
    #w_step_per_sec.value = 2500
    w_t_start.value = 0
    w_simtime.value = 10
    #w_scene_opts.value
    w_ani_beg.value = 0
    w_ani_spa.value = 0
    w_link.value = True
    clear_eigen_results()
    set_params_from_widgets()

# ——————————————————————————————————————————————————————————————————————
def update_record_subkey(record_key: str, subkey: str, value):
    """
    Update a single sub‐key in the JSON store for one record,
    leaving everything else untouched.
    """
    store = load_store()
    if record_key not in store:
        raise KeyError(f"No record found for key {current_rec_key!r}")
    store[record_key][subkey] = value
    save_store(store)


# ——————————————————————————————————————————————————————————————————————
def update_record_subkeys(record_key: str, subkeys: list[str], values: list):
    """
    Update multiple sub-keys in the JSON store for one record,
    leaving everything else untouched.

    Parameters
    ----------
    record_key : str
        The top-level key (record) to update in the store.
    subkeys : list[str]
        List of subkeys within the record to update.
    values : list
        List of new values corresponding to each subkey.
    """
    if len(subkeys) != len(values):
        raise ValueError("subkeys and values must have the same length")

    store = load_store()

    if record_key not in store:
        raise KeyError(f"No record found for key {record_key!r}")

    record = store[record_key]

    for key, val in zip(subkeys, values):
        record[key] = val

    save_store(store)

 # ——————————————————————————————————————————————————————————————————————
def update_file_list( record_key: str, files_key: str, file_list: list, fname: str, mode: Literal['a', 'r', 'd', 'c'] ):
    """
    1. Add or replace the last CSV entry for a given project record.
    On replace, the most‐recent file is removed from disk.
    2. Delete an entry on the list also delete it from the file system
    """
    global csv_filename, mp4_filename, t_vals, y_vals
    def parse_timestamp(fname: str) -> datetime:
        """
        Extract the YYYYMMDD_HHMMSS part by splitting on the last '-' and parse it.
        """
        ts_part = fname.rsplit('-', 1)[1]
        return datetime.strptime(ts_part, "%Y%m%d_%H%M%S")

    #—————————————————————————
    store = load_store()
    if record_key not in store:
        raise KeyError(f"No record '{record_key}' in store.")

    rec = store[record_key]
    files = rec.get(files_key, [])
    if mode == 'a':  # add
        files.append(fname)
    elif mode == 'r':  # replace
        if not files:
            # nothing to replace, just add
            files.append(fname)
        else:
            latest = files.pop()
            ## find the most‐recent by timestamp
            #timestamps = [
            #    ( parse_timestamp(f), f)
            #    for f in files
            #]
            ## pick the max
            #_, latest = max(timestamps, key=lambda x: x[0])
            try:
                os.remove(latest)
            except FileNotFoundError:
                pass  # already gone
            #files.remove(latest)
            files.append(fname)
    elif mode == 'd':  # delete
        if files:
            if fname in files:
                files.remove(fname)
        if fname == csv_filename:
            #print("tp: fname == csv_filename")
            csv_filename = None
            rec['csv_filename'] = ''
            t_vals = y_vals = None
        elif fname == mp4_filename:
            #print("tp: fname == mp4_filename")
            mp4_filename = None
            rec['mp4_filename'] = ''
        try:
            os.remove(fname)
        except FileNotFoundError:
            pass  # already gone
    elif mode == 'c':  # clean (keep only csv having same id_hash of the project
        if files:
            idstr = f'-{id_hash}-'
            files[:] = [item for item in files if idstr in item]
    else:
        return

    # write back and save
    rec[files_key] = files
    file_list[:] = files
    save_store(store)

    if files_key == "csv_files":
        update_csv_count()
    else:
        update_mp4_count()


# ——————————————————————————————————————————————————————————————————————
note_lbl = Label(value='Notes', layout=Layout( margin='1px 2px 0 2px'))
note_w   = Text(placeholder='Pin an entry by starting or ending the note with *', description='', layout=Layout(width='362px', margin='1px 5px 0 0'))
save_btn = Button(description='Save', button_style='success',layout=Layout(width='60px', height='26px', padding='0 0', margin='1px 5px 2px 0'))
restore_btn = Button(description='Load', button_style='info', layout=Layout(width='60px', height='26px', padding='0 0', margin='0 0 3px 0'))
#rename_btn = Button(description='Rename Params', button_style='info', layout=Layout(width='90px', padding='0 0'))
delete_btn = Button(description='Delete', button_style='danger', layout=Layout(width='60px', height='26px', padding='0 0',margin='0 0 3px 0'))
refresh_btn = Button(description='Refresh', button_style='warning', layout=Layout(width='60px', height='26px', padding='0 0', margin='0 0 3px 0'))
copy_btn = Button(description='Copy',icon='copy',layout=Layout(width='60px', height='26px', padding='0 0', margin='0 0 3px 0'))
paste_btn = Button(description='Paste',icon='paste',layout=Layout(width='60px', height='26px', padding='0 0', margin='0 0 2px 0'))
clear_proj_btn = Button(description='Clear', icon='eraser', button_style='warning', layout=Layout(width='60px', height='26px', padding='0 0',margin='2px 25px 3px 0'))
default_proj_btn = Button(description='Load Defaults', button_style='info', layout=Layout(width='90px', height='22px', padding='0 0',margin='2px 0 3px 0'), tooltip='Loads Daniel''s favorite parameters')
default_proj_btn.add_class('btn-center')

restore_dropdown = Select(options=[], rows=9, layout=Layout(width='400px', margin='0 3px 0 0'), description='')
#save_row = HBox([note_lbl, note_w, save_btn], layout=Layout(gap='10px'))
copy_status = W.HTML()

w_id_hash = W.HTML()

def update_id_hash():
    w_id_hash.value = f"<b>ID:</b> {id_hash}"

save_btn.on_click(on_save)
restore_btn.on_click(on_restore)
refresh_btn.on_click(update_restore_options)
#rename_btn.on_click(on_rename_params)
delete_btn.on_click(on_delete_params)
copy_btn.on_click(on_params_copy)
paste_btn.on_click(on_import_params)
clear_proj_btn.on_click(on_clear_proj)
default_proj_btn.on_click(on_clear_proj)
#evt = Event(source=restore_dropdown, watched_events=['dblclick'])

save_ui = VBox([
    HBox([
        VBox([
            W.HTML("<span class='sb'Save current parameters:</span>", layout=Layout(margin='1px 0 1px 0')),
            HBox([note_lbl, note_w]),
            HBox([
                W.HTML("<span class='sb'Saved parameters (select then click Load):</span>", layout=Layout(margin='2px 0 1px 0')),
                W.HTML("", layout=Layout(flex='1 1 auto')),
                default_proj_btn
                ], layout=Layout(width='100%', align_items='center',  margin='0', padding='0 5px 0 0')),
            restore_dropdown]),
        VBox([
            clear_proj_btn,
            save_btn, W.Box(layout=Layout(height="32px")), restore_btn, delete_btn, refresh_btn, copy_btn, paste_btn])
        ], layout=Layout(align_items='flex-start', gap='10px')),
         copy_status, out])
# initialize list
update_restore_options()

# ——————————————————————————————————————————————————————————————————————
def display_physics_params():
    params_resultsL.clear_output()
    params_resultsR.clear_output()
    with params_resultsL:
        g_min = math.degrees(gam_min) if DEGREE_MODE else gam_min
        display(Math(rf"""
        \begin{{alignedat}}{{2}}
        \omega_{{B}}        &={w_om.value: .6f}       \ \mathrm{{{velunit()}}}     &\quad&\text{{angular velocity of }}\vec{{B}}\\[2pt]
        I_1                 &={w_I1.value: .6g}       \ \mathrm{{kg\,m^2}}         &\quad&\text{{X Component  of moment of inertia}}\\[0pt]
        I_2                 &={w_I2.value: .6g}       \ \mathrm{{kg\,m^2}}         &\quad&\text{{Y component of moment of inertia}}\\[0pt]
        I_3                 &={w_I3.value: .6g}       \ \mathrm{{kg\,m^2}}         &\quad&\text{{Z component of moment of inertia}}\\[0pt]
        mB                  &={w_mB.value: .6g}       \ \mathrm{{Nm}}              &\quad&\text{{scalar multiplication of }}\vec{{m}}\text{{ and }}\vec{{B}}\\[3pt]
        \gamma_{{min}}      &={g_min: .7f}            \ \mathrm{{{angunit()}}}     &\quad&\text{{minimum }}\gamma\text{{ required for constant }}\theta\\[0pt]
        \gamma              &={w_gam.value: .7f}      \ \mathrm{{{angunit()}}}     &\quad&\text{{offset angle of }}\vec{{B}}\text{{ from the xy plane}}\\[3pt]
        \theta_0            &={w_th.value: .6f}       \ \mathrm{{{angunit()}}}     &\quad&\text{{initial value of angle }}\theta\\[3pt]
        \phi_0              &={w_ph.value: .6f}       \ \mathrm{{{angunit()}}}     &\quad&\text{{initial value of angle }}\phi\\[2pt]
        \end{{alignedat}}
        """))

    with params_resultsR:
        display(Math(rf"""
        \begin{{alignedat}}{{2}}
        \dot\theta          &={w_thd.value: .6f}      \ \mathrm{{{velunit()}}}      &\quad&\text{{initial velocity of }}\theta\\[2pt]
        \dot\phi            &={w_phd.value: .6f}      \ \mathrm{{{velunit()}}}      &\quad&\text{{initial velocity of }}\phi\\[2pt]
        \dot\psi            &={w_psd.value: .6f}      \ \mathrm{{{velunit()}}}      &\quad&\text{{initial velocity of }}\psi\\[2pt]
        \xi                 &={w_xi.value: .6f}       \ \mathrm{{s^{{-1}}}}         &\quad&\text{{damping factor}}\\[0pt]
        \ddot\psi           &={w_ps_acc.value: .6f}   \ \mathrm{{{accunit()}}}      &\quad&\text{{externally applied acceleration}}\\[2pt]
        \end{{alignedat}}
        """))
        #\omega_{{nat}}     &={w_om_nat.value: .6f}   \ \mathrm{{{velunit()}}}      &\quad&\text{{estimation of a natural frequency}}\\

def set_params_from_widgets():
    global om_B, I1_v, I2_v, I3_v, mB_v, gamma_v, gam_min, gam_eps, th_v, ph_v, ps_v, thd_v, phd_v,  psd_v, xi_v, ps_acc, psd_ratio
    om_B     = w_om.valRad
    I1_v	 = w_I1.value
    I2_v	 = w_I2.value
    I3_v	 = w_I3.value
    mB_v	 = w_mB.value
    gamma_v	 = w_gam.valRad
    gam_eps = w_gam_eps.valRad
    th_v	 = w_th.valRad
    ph_v	 = w_ph.valRad
    ps_v	 = w_ps.valRad
    thd_v	 = w_thd.valRad
    phd_v	 = w_phd.valRad
    psd_v	 = w_psd.valRad
    xi_v     = w_xi.value
    ps_acc	 = w_ps_acc.valRad
    psd_ratio = w_psd_ratio.value

# ——————————————————————————————————————————————————————————————————————
# — hook up the button click to your simulation function —
def on_run_clicked(button):
    global current_rec_key, csv_filename, csv_files, current_rec_key
    global om_B, I1_v, I2_v, I3_v, mB_v, gamma_v, gam_min, th_v, ph_v, ps_v, thd_v, phd_v,  psd_v, xi_v, ps_acc
    btn_simulate.disabled = True
    btn_simulate.button_style = 'warning'
    lb_result_sim.value = 'Running...'
    lb_result_ani.value = ''
    #plots_out.add_class('my-disabled')

    # clear previous run
    set_params_from_widgets()
    display_physics_params()

    #with status_out:
    #    status_out.clear_output()
    #    display(Math(r"\color{blue}{\text{Simulation is running, please wait}\ldots}"))

    res = run_sim(
        om_    = om_B,
        I1_    = I1_v,
        I2_    = I2_v,
        I3_    = I3_v,
        mB_    = mB_v,
        gam_   = gamma_v,
        th_    = th_v,
        ph_    = ph_v,
        ps_    = ps_v,
        thd_   = thd_v,
        phd_   = phd_v,
        psd_   = psd_v,
        xi_    = xi_v,
        ps_acc = ps_acc,
        #om_nat = om_nat,
        solver_params  = (
            w_solver.value,
            w_solv_max_step.value * 1e-3, #convert ms to secs
            w_solv_rel_tol.value,
            w_solv_abs_tol.value
        ),
        steps_per_sec  = int(w_step_per_sec.value),
        t_start        = w_t_start.value,
        simlength      = w_simtime.value
    )

    btn_simulate.disabled = False
    btn_simulate.button_style = 'success'

    btn_animate.disabled = False
    btn_animate.button_style = 'success'
    #plots_out.remove_class('my-disabled')

    if res:
        #if current_rec_key:
        #    write_csv()

        #if csv_filename:
        #    lb_result_sim.value = f"Complete ({res:.2f} secs). Results saved to {csv_filename}"
        #else:
        #    lb_result_sim.value = f"Complete ({res:.2f} secs). Results can be saved together with parameters by pressing 'Save'."
        lb_result_sim.value = f"{time.strftime('%H:%M:%S')}: Complete ({res:.2f} secs)."
        accor_plots.selected_index = 0
            
        #show_sim_save_dialog()
        #write_csv()
        #if current_rec_key:
        #     update_record_subkey(current_rec_key, 'csv_filename', csv_filename)

        # open the preview section
        # setup_scene(button)  # dont open!
        if current_rec_key:
            if not csv_files:
                update_file_list(current_rec_key, 'csv_files', csv_files, csv_filename, 'a') # add
                update_csv_count()
            else:
                sim_save_dlg.show()
    else:
         lb_result_sim.value = "Simulation failed. See log for details."

    #notify_when_notebook_idle()
    #notify_when_ui_responsive()
    
# ——————————————————————————————————————————————————————————————————————
def save_sim_choice(act):
    if write_csv() and current_rec_key:
        lb_result_sim.value += f" Results saved to {csv_filename}"
        update_file_list(current_rec_key, 'csv_files', csv_files, csv_filename, act)
        #update_record_subkey(current_rec_key, 'csv_filename', csv_filename)

# ——————————————————————————————————————————————————————————————————————
def save_anim_choice(act):
    update_file_list(current_rec_key, 'mp4_files', mp4_files, mp4_filename, act)

# ——————————————————————————————————————————————————————————————————————
btn_simulate.on_click(on_run_clicked)

#==================================================================================================
#==================================================================================================
calc_th_eq()
#empirical formula
#gam_min = sp.asin(mB_v/(2*I_v*om_B**2))
#gam_min = gamma_min_mb2(-psd_v/om_B)
calc_gamma_min()
update_th_eq(None)
gamma_v = gam_min + gam_eps
w_gam.valRad = gamma_v

#***************************************************************************************************************************************
# LINEARIZATION & EIGENVALUES BOX
eigen_box =  Output()
nat_freq_box = Output()
eigen_num  = Output()
eigen_resL =  Output(layout=Layout(overflow='auto', min_width='810px', min_height="810px", margin='2px 0 0 0')) # !! 4px margin adjust the plot top with the right table
eigen_resR =  Output(layout=Layout(overflow='auto', min_width='555px', min_height="810px", margin='0', padding='0'))

the_0, phi_0, psi_0, the_eq, phi_eq, psi_eq, phi_p, kappa = sp.symbols('theta_0 phi_0 psi_0 theta_eq phi_eq psi_eq phi_p kappa', real=True)
vars = sp.Matrix([em.th, em.ph, em.ps, em.th_d, em.ph_d, em.ps_d])

F = sp.Matrix([em.eom_axisym[0].rhs/em.I1, em.eom_axisym[1].rhs/em.I1, em.eom_axisym[2].rhs/em.I1])
#F = sp.Matrix([em.eom_iso[0].rhs/em.I, em.eom_iso[1].rhs/em.I, em.eom_iso[2].rhs/em.I])

J = F.jacobian(vars)

phi_0 = em.omega * em.t + phi_p
psi_0 = -kappa * em.omega * em.t

JE = sp.simplify(J.subs({em.th:the_0, em.ph:phi_0, em.ps:psi_0, em.th_d:0, em.ph_d: em.omega, em.ps_d: -kappa * em.omega}))
JEm = sp.Matrix(JE)

#------simplify J[0,0] ------
x = JEm[0,0]
k = em.omega**2 * em.I3 * cos(the_0)
x = sp.simplify(x-k)
x = x.collect(em.omega)
x = (sp.simplify(x) + k).collect(em.omega).collect(em.mB)
JEm[0, 0] = x
#------simplify J[0,4] ------
x = JEm[0,4].collect(2*cos(the_0))
JEm[0,4] = x
#------simplify J[2,0] ------
JEm[2, 0] = JEm[2, 0].collect(em.xi * em.omega*sin(the_0))
#------avoid tan in J[2,1] ------
JEm[2, 1] = JEm[2, 1].subs(tan(the_0),sin(the_0)/cos(the_0))
#------simplify J[2,3] ------
JEm[2, 3] = JEm[2, 3].collect(em.I3)

# substitute state variables with their symbols about their initial values
R0_th = sp.simplify(em.eom_axisym[0].rhs.subs({em.th:the_0, em.ph:phi_0, em.ps:psi_0, em.th_d:0, em.ph_d: em.omega, em.ps_d: -kappa * em.omega}))
R0_ph = sp.simplify(em.eom_axisym[1].rhs.subs({em.th:the_0, em.ph:phi_0, em.ps:psi_0, em.th_d:0, em.ph_d: em.omega, em.ps_d: -kappa * em.omega}))
R0_ps = sp.simplify(em.eom_axisym[2].rhs.subs({em.th:the_0, em.ph:phi_0, em.ps:psi_0, em.th_d:0, em.ph_d: em.omega, em.ps_d: -kappa * em.omega}))
# isotropic case
#R0_th = sp.simplify(em.eom_iso[0].rhs.subs({em.th:the_0, em.ph:phi_0, em.ps:psi_0, em.th_d:0, em.ph_d: em.omega, em.ps_d: -kappa * em.omega})/em.I)
#R0_ph = sp.simplify(em.eom_iso[1].rhs.subs({em.th:the_0, em.ph:phi_0, em.ps:psi_0, em.th_d:0, em.ph_d: em.omega, em.ps_d: -kappa * em.omega})/em.I)
#R0_ps = sp.simplify(em.eom_iso[2].rhs.subs({em.th:the_0, em.ph:phi_0, em.ps:psi_0, em.th_d:0, em.ph_d: em.omega, em.ps_d: -kappa * em.omega})/em.I)

JEmIso = sp.Matrix(sp.simplify(JEm.subs(em.I3, em.I1)))
JEmIso[0,0] = JEmIso[0,0].collect(em.mB)

def fmt_frac(mat):
    s = latex(mat).replace(r'\frac', r'\dfrac')
    return re.sub(r'\\\\', r'\\\\[9pt]', rem_par(s))

with eigen_box:
    display(Markdown(r"""
<span class='sb' style='max-width:1020px; display:block;'>
This box covers derivation of linearized equations of motion (Jacobians, state matrix) and eigenvalues. This allows one to obtain eigenvalue spectrum of the solution
with a selected parameter at the section <a onclick='event.preventDefault(); document.getElementById("eigenvalue-analysis").scrollIntoView({behavior:"smooth", block: "start"});'
   style='cursor:pointer; text-decoration:underline; color:#1a73e8; font-weight:600'>Eigenvalue Analysis</a> below.<br><br>

### Linearization of equations of motion
The equations of motion for an axisymmetric body admit equilibrium solutions in which all vectors remain time-invariant in the reference frame of the rotating magnetic field.
These equilibria have been confirmed by experimental observations and numerical simulations. Based on this fact, the equations of motion are linearized about the equilibrium
point / trajectory defined by the generalized coordinates $\theta,\;\phi,\;\psi$ and their first derivatives, using a first-order Taylor expansion.<br><br>

A Taylor expansion over a vector-valued nonlinear function $G(\mathbf{x})$ can be expressed as follows:<br>

Having a vector of variables $\mathbf{x}$ and the function $G$ as

$(1)\phantom{0} \qquad \mathbf{x} =
\begin{bmatrix}
x_1 \\ x_2 \\ \vdots \\ x_n
\end{bmatrix}
\in \mathbb{R}^n, \qquad
G : \mathbb{R}^n \;\to\; \mathbb{R}^m\,,
\qquad
G(\mathbf{x}) =
\begin{bmatrix}
G_1(\mathbf{x}) \\ G_2(\mathbf{x}) \\ \vdots \\ G_m(\mathbf{x})
\end{bmatrix}\,.
$

The Taylor expansion of $G$ around a point $\mathbf{x}_0$ up to first order is

$(2)\phantom{0} \qquad
G(\mathbf{x}) \;\approx\;
G(\mathbf{x}_0)
+ J_G(\mathbf{x}_0)\,(\mathbf{x}-\mathbf{x}_0)\,,
$

where the Jacobian matrix is defined as

$(3)\phantom{0} \qquad
J_G(\mathbf{x}_0) \;=\;
\begin{bmatrix}
\dfrac{\partial G_1}{\partial x_1} & \cdots & \dfrac{\partial G_1}{\partial x_n} \\
\vdots & \ddots & \vdots \\
\dfrac{\partial G_m}{\partial x_1} & \cdots & \dfrac{\partial G_m}{\partial x_n}
\end{bmatrix}_{\Large{x}=\Large{x}_0}\,.
$

If $\mathbf{x}_0$ corresponds to an equilibrium ($G(\mathbf{x}_0)=0$), this reduces to

$(4)\phantom{0} \qquad
G(\mathbf{x}) \;\approx\; J_G(\mathbf{x}_0)\,(\mathbf{x}-\mathbf{x}_0)\,.
$

Applying this formulation to the angular dynamics of the model, the three equations of motion can be expressed as

$(5)\phantom{0} \qquad \ddot s = F_{\textstyle s} (\theta, \, \phi, \, \psi, \, \dot \theta, \, \dot\phi, \, \dot\psi) ,\quad s \in \{\theta, \, \phi, \, \psi \}\,.$

The equilibrium condition can be defined as

$(6)\phantom{0} \qquad \ddot s = F_{\textstyle s} (\theta_0, \, \phi_0, \, \psi_0, \, \dot\theta_0, \, \dot\phi_0, \, \dot\psi_0) = 0 ,\quad s \in \{\theta, \, \phi, \, \psi \}\,.$

State variables that satisfy this equilibrium can be found as

$(7)\phantom{0} \qquad \theta_0 = \text{constant value determined by solving } F_\theta = 0\,,$

$(8)\phantom{0} \qquad \phi_0 = \omega t + \phi_p\,,$

$(9)\phantom{0} \qquad \psi_0 = - \kappa \,\omega t\,,$

$(10) \qquad \dot\theta_0 = 0\,,$

$(11) \qquad \dot\phi_0 = \omega\,,$

$(12) \qquad \dot\psi_0 = - \kappa \,\omega\,,$

where $\kappa$ is a coefficient relating the angular velocity term $\dot\psi$ to the angular velocity $\omega$ of the rotating field. It is set by the initial conditions and
remains constant when damping is zero, but converges to $\cos\theta$ in the presence of damping.

The phase $\phi_p$ may reach an equilibrium value of $\pi$ when damping is negligible, while it increases slightly in proportion to the damping.

It should be noted that the phase $\phi_p$ determines the azimuthal offset between the magnetic moment $\mathbf{\vec{m}}$ and the rotating magnetic field $\mathbf{\vec{B}}$.
A value of $\phi_p = $<span style="font-size:110%">$\;\pi$</span> corresponds to antiparallel alignment of $\mathbf{\vec{m}}$ with $\mathbf{\vec{B}}$, the key phase-lag
condition of the dynamics.</span>"""))

    display(Markdown(r"<div class='sb'>In equilibrium condition, the equation $F_{\textstyle \theta} = 0$ for an axisymmetric body can be written as</div>"))
    display(Math("(13) \\qquad " + rem_par(latex(sp.simplify(R0_th))) + " = 0\\,."))
    display(Markdown(r"<div class='sb'>Under zero damping condition, by setting $\phi_p = \pi$, this simplify to</div>"))
    display(Math("(14) \\qquad " + rem_par(latex(sp.simplify(R0_th.subs({phi_p:sp.pi})))) + " = 0\\,,"))
    display(Markdown(r"<div class='sb'>which can be solved numerically. In the case of $\kappa=1$, this simplify to</div>"))
    display(Math("(15) \\qquad " + rem_par(latex(sp.simplify(R0_th.subs({phi_p:sp.pi, kappa:1})))) + " = 0\\,,"))
    display(Markdown(r"<div class='sb'>and for $\kappa=1$ and for a body having isotropic moment of inertia $(I_1 = I_2 = I_3)$, it simplify to</div>"))
    x = R0_th.subs({phi_p:sp.pi, kappa:1, em.I3:em.I1})
    x = x.expand().collect(sin(the_0))
    display(Math("(16) \\qquad " + rem_par(latex(x)) + " = 0\\,,"))
    display(Markdown(r"<div class='sb'>allowing $\theta_0$ to be derived analytically.</div>"))

    ################################################################################################
    display(Markdown(r"""<div class='sb' style='max-width:1020px;'>

In our model, the generalized coordinates (Euler angles), their velocities and accelerations can be defined as

$(17) \qquad q = \begin{bmatrix}\theta \\ \phi \\ \psi \end{bmatrix},
      \qquad \dot q = \begin{bmatrix} \dot\theta \\ \dot\phi \\ \dot\psi \end{bmatrix}\,.$

We also define corresponding vectors that satisfy the equilibrium state as

$(18) \qquad q_0 = \begin{bmatrix}\theta_0 \\ \phi_0 \\ \psi_0 \end{bmatrix},
\qquad \dot q_0 = \begin{bmatrix} \dot\theta_0 \\ \dot\phi_0 \\ \dot\psi_0 \end{bmatrix}\,.$

This way the state vector $\mathbf{x}$ and equilibrium state vector $\mathbf{x_0}$ can be defined as:

$(19) \qquad
\mathbf{x =   \begin{bmatrix} q   \\ \dot q   \end{bmatrix} = \begin{bmatrix} \theta   \\ \phi   \\ \psi   \\ \dot\theta   \\ \dot\phi   \\ \dot\psi   \end{bmatrix} }\,, \qquad
\mathbf{x_0 = \begin{bmatrix} q_0 \\ \dot q_0 \end{bmatrix} = \begin{bmatrix} \theta_0 \\ \phi_0 \\ \psi_0 \\ \dot\theta_0 \\ \dot\phi_0 \\ \dot\psi_0 \end{bmatrix} }\,.$

The Jacobian $J(\mathbf{x})$ evaluated at the equilibrium point / trajectory $\mathbf{x_0}$ can be expressed as:

$
(20) \qquad J(\mathbf x) = \left.\dfrac{\partial\,(F_\theta,\,F_\phi,\,F_\psi)}{\partial\mathbf x}\right|_{\Large{x} = \Large{x}_0}
= \big[\,J(q) \;|\; J(\dot{q}))\,\big]\,,
$

where

$
(21) \qquad J(q)       = \left.\dfrac{\partial (F_\theta, F_\phi, F_\psi)}{\partial q}\right|_{\Large{x}_0}\,,
\qquad      J(\dot{q}) = \left.\dfrac{\partial (F_\theta, F_\phi, F_\psi)}{\partial \dot q}\right|_{\Large{x}_0}\,.
$

At the equilibrium point / trajectory, $J(q)$ and $J(\dot{q})$ are derived as
</div>"""))

    display(Markdown(rf"""
$(22) \qquad J(q) = {fmt_frac(JEm[:, :3])} \,,$
<div style='height:24px'></div>
$(23) \qquad J(\dot{{q}}) = {fmt_frac(JEm[:, 3:])}\,.$
<div class='sb' style='margin: 30px 0;'>For an isotropic body these Jacobians simplify as</div>
$(24) \qquad J(q) = {fmt_frac(JEmIso[:, :3])}\,,$
<div style='height:30px'></div>
$(25) \qquad J(\dot{{q}}) = {fmt_frac(JEmIso[:, 3:])}\,.$
"""))
    display(Markdown(r"""
<div class='sb' style='max-width:1020px; margin: 30px 0 0 0;'>
Using Jacobians, the equation of motion can be expressed in matrix form as<br><br>
$
(26) \qquad \ddot q = J(q)\, q + J(\dot q) \, \dot q \,,
$

<br>and in first order state matrix form as

$
(27) \qquad \dot{\begin{bmatrix}q \\ \dot q\end{bmatrix}} =
\underbrace{
  \begin{bmatrix}
    0_3 & I_3 \\
    J(q) &    J(\dot q)
  \end{bmatrix}
}_{\textstyle A}
\,
\begin{bmatrix}q \\ \dot q\end{bmatrix}\,,
$

where the term denoted as $A$ is the $6\mathord{\times}6$ state matrix, $0_3$ is a zero matrix and $I_3$ is an identity matrix in $3\mathord{\times}3$ form.

Using state vector $x$, this state matrix representation of equation of motion can be expressed as

$
(28) \qquad \mathbf{\dot x} = \underbrace{ \begin{bmatrix}
  0_3 & I_3 \\
  J(q) &    J(\dot q)
\end{bmatrix}}_{\textstyle A}\,\mathbf{x}\,.
$
</div>

<div class='sb' style='max-width:1020px;'>
In order to carry out the linearization, the state vector $\mathbf{x}$ is expressed as a perturbation around the equilibrium point / trajectory<br><br>

$
(29) \qquad \mathbf{x = x_0 + \delta x}\,.
$

Note that vector $\mathbf{\delta x}$ contains both small variations of coordinates $\mathbf{\delta q}$ and small variations of velocities $\mathbf{\delta \dot q}\,$.

$
(30) \qquad \mathbf{\delta x =   \begin{bmatrix} \delta q   \\ \delta \dot q \end{bmatrix}}\,.
$

This way, the linearized equations of motion in matrix form using Taylor expansion of $F$ around a point / trajectory $\boldsymbol{x}_0$ up to first order can be expressed as

$
(31) \qquad \dot x = F(\mathbf{x}) \;\approx\; F(\mathbf{x}_0) + J(\mathbf{x}_0)\,\mathbf{\delta x}\,.
$

Note that for the steady-rotation equilibrium where second derivatives of coordinates are zero, we have

$
(32) \qquad F(\mathbf{x}_0)=\dot{\mathbf x}_0
= \begin{bmatrix} \dot q_0 \\[4pt] \ddot q_0 \end{bmatrix}
= \begin{bmatrix} 0 \\ \dot\phi_0 \\ \dot\psi_0 \\[4pt] 0 \\ 0 \\ 0 \end{bmatrix},
$

so $F(\mathbf{x}_0)$ is not the zero vector (because $\dot\phi_0,\dot\psi_0\neq0$).
Subtract $\dot{\mathbf x}_0 = F(\mathbf{x}_0)$ from both sides of (29). Since
$\delta\dot{\mathbf x}=\dot{\mathbf x}-\dot{\mathbf x}_0$ we get

$
(33) \qquad \delta\dot{\mathbf x} \;=\; \dot{\mathbf x}-\dot{\mathbf x}_0
= F(\mathbf{x}_0)+J_x(\mathbf{x}_0)\,\delta\mathbf x - F(\mathbf{x}_0)
= J_x(\mathbf{x}_0)\,\delta\mathbf x.
$

Using the block form of the Jacobian introduced in (27)--(28),

$
(34) \qquad J_x(\mathbf{x}_0)=\begin{bmatrix}0_3 & I_3\\[4pt] J(q) & J(\dot q)\end{bmatrix}=A\,.
$

In the perturbation form, the state-matrix ($A$) representation of linearized equations of motion now can be expressed as

$
\boxed{(35)\qquad \delta\dot{\mathbf x} \;=\; A \,\delta\mathbf x\vphantom{\Big(}\,.}
$

</div>
<hr style='width:1340px; margin:2em auto 1em 0; height:2px; border:none;'> 
"""))

with nat_freq_box:
    display(Markdown(r"""
#### Notes

<div class='sb' style='max-width:1020px;'>There is also an empirical formula for the natural frequency
$\omega_N$ which gives close figures for both isotropic and axisymmetric bodies when $\gamma \approx \gamma_{\min}\,$<br><br>

$
(36) \qquad \omega_N = \sqrt{\,u \;-\;\dfrac{1}{2\,u} \,\biggl(\dfrac{m\!B}{I_c}\biggr)^{2}},
     \quad u\;=\;\sqrt{|\dot\phi\dot\psi|}\,,  \quad  I_{c} \;=\;
     \begin{cases}
     I_1, & \text{(isotropic case)},\\[6pt]
     I_3 + \lvert I_3-I_1 \rvert \cos\theta_0, & \text{(axisymmetric case)}.
     \end{cases}
$

The natural frequency bifurcates (splits into two branches) when $\gamma > \gamma_{\min}$ and $\theta(t)$ can be expressed as

$
(37) \qquad \theta(t) = a_\alpha \, \sin(\omega_\alpha \, t) + a_\beta \, \sin(\omega_\beta \, t)\,.
$

When amplitudes $a_\alpha$ and $a_\beta$ are equal, this can be also expressed as


$
(38) \qquad \theta(t) = 2 \, a \, \sin\Bigl(\dfrac{\omega_{\alpha} + \omega_{\beta}}{2} t\Bigr) \, \cos\Bigl(\dfrac{\omega_{\alpha} - \omega_{\beta}}{2} t\Bigr)\,,
$

which defines the oscillation as a center frequency in the sine term and the beat frequency in the cosine. The empirical formula in this case provides the center frequency.

</div>"""))

# —————————————— Natural frequency calculation ———————————————
calc_om0_displayed = False

def calculate_omega0(b):
    global calc_om0_displayed
    with nat_freq_box:
        if calc_om0_displayed:
            # Remove the last output (the previous calculation)
            nat_freq_box.outputs = nat_freq_box.outputs[:-1]

        om_v = w_om.valRad
        mB_v  = w_mB.value
        th_v = w_th_eq.valRad

        #Ic = I3_v + (I3_v - I1_v) * np.cos(th_v)
        Ic = I3_v + abs(I3_v - (I1_v+I2_v/2)) * np.cos(th_v)
        phd_v = w_phd.valRad
        psd_v = w_psd.valRad
        beta =  np.abs(phd_v*psd_v)

        om_zero = np.sqrt(beta - 0.5 * (mB_v / Ic)**2/beta)
        #print ("w I1: ", np.sqrt(beta - 0.5 * (mB_v / I1_v)**2/beta))
        #Ig = 2*I1_v*I3_v / (I1_v + I3_v)
        #print ("w Ig: ", np.sqrt(beta - 0.5 * (mB_v / Ig)**2/beta))
        #print(f"𝜔={om_B:.6f}, 𝛪1={I1_v:.6g}, 𝛪3={I3_v:.6g}, mB={mB_v:.6g}, 𝛾={float(gamma_v):.6f}")
        #print(f"𝜔₀: {omega_zero:.6f} rad/s,  {omega_zero/(2*np.pi):.6f} Hz [ simulation data 𝜃(t) ]")
        #print(f"𝜔₀: {o:.6f} rad/s,  {o/(2*np.pi):.6f} Hz [empirical formula]")
        display(Math(rf"(39)\qquad \omega_N \approx {om_zero:.6f}\: \text{{rad/s}} \; = \; {om_zero/(2*np.pi):.6f}\: \text{{rev/s}}"))
        calc_om0_displayed = True

btn_calc_om0 = Button(description='Calculate approximate center natural frequency with current parameters', button_style='info', layout=Layout(width='430px', height = '26px', padding='0', margin='0 0 0 0'))
btn_calc_om0.on_click(calculate_omega0)

with nat_freq_box:
    display(btn_calc_om0)

# ——————————————————————————— eigenvalue analysis —————————————————————————————
eig_param_options = [
    ("𝝎","omega"),
    ("𝜤₁","I1"),
    ("𝜤₃","I3"),
    ("𝒎𝑩", "mB"),
    ("𝜸", "gamma"),
    ("𝜓˙", "psi_d"),
    ("𝜿", "kappa"),
    ("ξ", "xi"),
]

eig_param_labels2 = {
    'omega':'\\omega',
    'I1'   : 'I_1',
    'I3'   : 'I_3',
    'mB'   : 'mB',
    'gamma': '\\gamma',
    'psi_d': '\\dot\\psi',
    'kappa': '\\kappa',
    'xi'   : '\\xi'
}

eig_param_labels = {internal: display for display, internal in eig_param_options}

def get_params(base_params, sweep_param, value):
    new_params = base_params.copy()
    new_params[sweep_param] = value
    # here you can enforce equilibrium conditions
    return new_params

# —————————————————————————————————————————————————————————————————————————————————
#try:
#    from scipy.optimize import linear_sum_assignment
#    have_scipy = True
#except Exception:
#    have_scipy = False
# —————————————————————————————————————————————————————————————————————————————————
def reorder_eigvals_steps(eigvals_steps):
    """
    eigvals_steps: array shape (n_steps, n_modes) complex
    Returns reordered array of same shape where modes are tracked continuously.
    Uses Hungarian if scipy available, otherwise greedy nearest neighbor.
    """
    eig = np.array(eigvals_steps, copy=True)
    n_steps, n_modes = eig.shape
    reordered = np.empty_like(eig)
    reordered[0] = eig[0]  # keep first step as-is (or sort it if you want)

    for k in range(1, n_steps):
        prev = reordered[k-1]
        cur  = eig[k]
        # cost = distance in complex plane
        cost = np.abs(prev.reshape(-1,1) - cur.reshape(1,-1))

        #if have_scipy:
        r, c = linear_sum_assignment(cost)
        # r are rows (prev) matched to c (cur)
        # build ordering arr so column j in reordered[k] corresponds to prev[j]
        order = np.argsort(r)   # r is permutation of 0..n_modes-1
        matched_cols = c[order]
        reordered[k] = cur[matched_cols]
        #else:
        #    # greedy: assign each prev to nearest unmatched cur
        #    prev_indices = list(range(n_modes))
        #    unmatched = set(range(n_modes))
        #    map_cols = [-1]*n_modes
        #    for i in prev_indices:
        #        # find nearest among unmatched
        #        d = cost[i]
        #        candidates = sorted(unmatched, key=lambda j: d[j])
        #        j = candidates[0]
        #        map_cols[i] = j
        #        unmatched.remove(j)
        #    reordered[k] = cur[map_cols]
    return reordered

# —————————————————————————————————————————————————————————————————————————————————
JEm_num=None

def calculate_eigenvalues(sweep_param, thetas, beg_pct, end_pct, steps, zoom_val) -> Tuple[float, float, float]:
    global th_eq, om_B, I1_v, I2_v, I3_v, mB_v, gamma_v, ph_v, phd_v, psd_v, xi_v, kappa_v, calc_psd_from_cos_theta
    global eigvals_all, eigvecs_all, sweep_vals

    om_B     = w_om.valRad
    I_approx = (w_I1.value + w_I2.value)/2
    I1_v     = I_approx
    I3_v     = w_I3.value
    mB_v     = w_mB.value
    gamma_v  = w_gam.valRad
    ph_v     = w_ph.valRad
    phd_v    = w_phd.valRad
    psd_v    = w_psd.valRad
    kappa_v  = -psd_v/phd_v if phd_v else 0
    xi_v      = w_xi.value

    params = {
        "omega"  : om_B,
        "I1"     : I1_v,
        "I3"     : I3_v,
        "mB"     : mB_v,
        "gamma"  : gamma_v,
        "psi_d"  : psd_v,
        "kappa"  : kappa_v,
        "xi"     : xi_v
    }

    base_val = zoom_val if zoom_val is not None else params[sweep_param]
    if base_val == 0:
        base_val = 1

    sgn = np.sign(base_val) or 1
    beg_val = (1 + sgn*beg_pct/100)*base_val
    end_val = (1 + sgn*end_pct/100)*base_val
    sweep_vals = np.linspace(beg_val, end_val, steps)

    eigvals_all = []
    eigvecs_all = []

    for i, val in enumerate(sweep_vals):
        P = params.copy()
        P[sweep_param] = val  # update sweep param

        om_B     = P["omega"]
        I1_v     = P["I1"]
        I2_v     = I1_v
        I3_v     = P["I3"]
        mB_v     = P["mB"]
        gamma_v  = P["gamma"]
        psd_v    = P["psi_d"]
        kappa_v  = P["kappa"]
        xi_v      = P["xi"] #damping coefficient

        try:
            th_eq, ph_v = calc_th_eq_sub()
        except Exception as e:
            eigen_sts_res.value = f"Finding equilibrium theta angle failed: {str(e)}"
            th_eq = w_th.valRad

        #print("theta=",th_eq)
        if sweep_param!='psi_d':
            if calc_psd_from_cos_theta == 2:
               psd_v   = -om_B * cos(th_eq)
            else:
               psd_v   = -om_B * kappa_v

        kappa_v = -psd_v / om_B if om_B else 0

        phd_v = om_B

        #ph_v = np.pi - np.asin(float(eq_phi_phase.subs({em.xi: xi_v * I1_v, em.th: th_eq, em.omega: om_B, em.mB: mB_v, em.gamma: gamma_v})))

        thetas[i] = float(th_eq)

        subs_j = {em.omega:om_B, em.I1:I1_v, em.I3:I3_v, em.mB:mB_v, em.gamma:gamma_v, the_0:th_eq, em.th_d:0, em.ph_d:phd_v, em.ps_d:psd_v, kappa: kappa_v, em.xi:xi_v * I1_v, phi_p:ph_v}
        JEm_num = clean_small_entries(JEm.subs(subs_j).evalf(), 1e-16)

        # top block [0_3x3 | I_3x3]0.55
        top = sp.Matrix.hstack(sp.zeros(3, 3), sp.eye(3))
        # vertical stack to form 6x6
        A_num = sp.Matrix.vstack(top, sp.Matrix(JEm_num))
        A_np = np.array(A_num, dtype=float)  # convert to float ndarray

        if np.isnan(A_np).any() or np.isinf(A_np).any():
            #display(Math(latex(A_num)))
            eigen_sts_res.value="State matrix A has errors on inifinites."
            return None

        vals, vecs = np.linalg.eig(A_np)

        eigvals_all.append(vals)
        eigvecs_all.append(vecs)

    eigvals_all = np.array(reorder_eigvals_steps(eigvals_all), dtype=np.complex128)

    eigvecs_all = np.array(eigvecs_all, dtype=np.complex128)

    tol = 1e-13
    #vals2 = eigvals_all.copy()                       # keep original
    vals2 = np.array(eigvals_all, dtype=np.complex128, copy=True) # keep original
    vals2.real[np.abs(vals2.real) < tol] = 0.0
    vals2.imag[np.abs(vals2.imag) < tol] = 0.0
    eigvals_all = vals2
    return base_val, beg_val, end_val

# —————————————————————————————————————————————————————————————————————————————————
fig_e = None
set_param_btn1 = set_param_btn2 = None

# —————————————————————————————————————————————————————————————————————————————————
def eigen_nz_cols(eigvals: np.ndarray, tol: float, mode: str = "c"):
    """
    Determine which column indices to keep after optionally removing a
    leading and/or trailing column that are (near) zero according to `mode`.

    Parameters
    ----------
    eigvals : array-like, shape (n_rows, n_cols) or (n,)
        Complex-valued array where columns are tested.
    tol : float
        Tolerance used for comparisons.
    mode : {"r", "i", "c"}
        - "real": consider a column zero if all real parts are within +/- tol.
        - "imag": consider a column zero if all imag parts are within +/- tol.
        - "complex": consider a column zero if both real and imag parts are within +/- tol.

    Returns
    -------
    keep_idx : list[int]
        Indices of columns to keep.
    """
    eigvals = np.asarray(eigvals)
    if eigvals.ndim == 1:
        eigvals = eigvals.reshape(-1, 1)

    ncols = eigvals.shape[1]
    if ncols == 0:
        return [], False, False

    def is_zero_col(col: np.ndarray) -> bool:
        if mode == "r":
            return np.all(np.abs(col.real) <= tol)
        if mode == "i":
            return np.all(np.abs(col.imag) <= tol)
        # complex
        return (np.all(np.abs(col.real) <= tol) and
                np.all(np.abs(col.imag) <= tol))

    # evaluate flags on original first/last columns
    zero_first = is_zero_col(eigvals[:, 0]) if ncols >= 1 else False
    zero_last  = is_zero_col(eigvals[:, -1]) if ncols >= 1 else False

    keep = list(range(ncols))
    if zero_first and keep:
        keep = keep[1:]
    # remove last only if there is still at least one column left to remove
    if zero_last and keep:
        keep = keep[:-1]

    return keep

# —————————————————————————————————————————————————————————————————————————————————
def set_param_value(no, value):
    if no==0:
        w_om.valRad = value
    elif no==1:
        w_I1.value = w_I2.value = value
    elif no==2:
        w_I3.value = value
    elif no==3:
        w_mB.value = value
    elif no==4:
        w_gam.valRad = value
    elif no==5:
        w_psd.valRad = value
    elif no==6:
        w_psd.valRad = value * w_om.valRad
    elif no==7:
        w_xi.value = value

def on_set_base_clicked(event):
    global base_x
    set_param_value(sweep_param_dd.index, base_x)

def on_set_bifurc_clicked(event):
    global stab_lim
    set_param_value(sweep_param_dd.index, stab_lim)

def on_click_fig_e(event):
    global base_x, stab_lim
    if event.button == MouseButton.LEFT and 523 <= event.x <= 559:
        if 36 >= event.y >= 23:
            set_param_value(sweep_param_dd.index, base_x)
        elif 18 >= event.y >= 5:
            set_param_value(sweep_param_dd.index, stab_lim)

    #with eigen_resR:
    #    print("Python handler called:", event.x, event.y, "button:", event.button)
    #
    #   if event.inaxes is btn_ax1:
    #       on_set_base_clicked(event)
    #       print("event.inaxes is btn_ax1")
    #   elif event.inaxes is btn_ax2:
    #       on_set_bifurc_clicked(event)
    #        print("event.inaxes is btn_ax2")

# ————————————————————————————————————————————————————————————————————————————
cid = None
# ———————————————————— eigenvalues plotting procedure ————————————————————————
def plot_eigenvalues(xlabel, swe_vals, eigvals, base_x, stability_lim, out_widget, single, positive):
    global fig_e, set_param_btn1, set_param_btn2

    #def update_ticks(event):
    #    ax_e.yaxis.set_major_locator( mticker.SymmetricalLogLocator(base=10, linthresh=1e-6, subs=np.arange(1, 10)))
    #    fig_e.canvas.draw_idle()

    def add_button_tooltip(fig, btn_ax, text,offset_y_px=7):
        tip = fig.text(0, 0, text, fontsize=8, va="bottom", ha="left",
            bbox=dict(boxstyle="round", fc="#fff4c2", ec="0.3"), visible=False, zorder=10, transform=IdentityTransform())

        renderer = fig.canvas.get_renderer()
        bbox = btn_ax.get_window_extent(renderer)
        tip.set_position((bbox.x0-80, bbox.y1+offset_y_px))

        def on_move(event):
            inside = (event.inaxes is btn_ax)
            if inside != tip.get_visible():
                tip.set_visible(inside)
                fig.canvas.draw_idle()    

        fig.canvas.mpl_connect("motion_notify_event", on_move)
   
    try:
        plt.close(fig_e)
    except NameError:
        pass  # fig_e doesn't exist yet

    fp = FontProperties(family='STIXGeneral', size=11)

    with out_widget:
        if single:
            # build list of column indices to keep
            keep_idx = eigen_nz_cols(eigvals, 1e-12, 'c')
            fig_e, ax_e = plt.subplots(figsize=(8,8))
            # hide header/footer, enable toolbar
            fig_e.canvas.header_visible = False
            fig_e.canvas.footer_visible = False
            fig_e.canvas.toolbar_visible = True
            #for i in range(eigvals_all.shape[1]):
            for i in keep_idx:
                ax_e.plot(swe_vals, eigvals[:, i].real, label=f"λ{i+1} Re", lw=glw, linestyle='-')
                ax_e.plot(swe_vals, eigvals[:, i].imag, label=f"λ{i+1} Im", lw=glw, linestyle='--')

            ax_e.axhline(0, color='black', lw=0.3)
            ax_e.set_xlabel(rf'${xlabel}$', fontproperties=fp)
            if base_x:
                ax_e.text(0.521, -0.09, f"(center={base_x:.8f})", transform=ax_imag.transAxes, ha='left', va='top', fontsize=9, color='black')
                ax_e.axvline(x=base_x, color='blue', linestyle='dotted', linewidth=0.8, zorder=5)
            if stability_lim:
                ax_e.axvline(x=stability_lim, color='C3', linestyle='dotted', linewidth=0.8, zorder=5)

            if False: #linear
                ax_e.set_yscale('linear')
                ax_e.yaxis.set_major_locator(mticker.AutoLocator())
                ax_e.yaxis.set_minor_locator(mticker.AutoMinorLocator(n=5))
                ax_e.set_ylabel("Eigenvalue parts", fontproperties=fp)
            else: # logarithmic scale
                threshold=1e-5 # 1e-6
                ax_e.set_ylabel("Eigenvalue parts (symlog scale)", fontproperties=fp)
                ax_e.set_yscale('symlog', linthresh=threshold, linscale=1)  # 👈 log scale for ±values
                subs_minor = np.logspace(0, 1, 6)[:-1] / 10 # four minor grid lines
                # Major and minor locators
                ax_e.yaxis.set_major_locator(mticker.SymmetricalLogLocator(base=10, linthresh=threshold, subs=[1.0]))
                ax_e.yaxis.set_minor_locator(mticker.SymmetricalLogLocator(base=10, linthresh=threshold, subs=subs_minor))

            ax_e.xaxis.set_major_locator(mticker.AutoLocator())
            ax_e.xaxis.set_minor_locator(mticker.AutoMinorLocator())

            ax_e.grid(True, which="both", ls="--", lw=0.3)
            ax_e.tick_params(axis='both', which='major', labelsize=8)
            ax_e.legend(fontsize=8)

            fig_e.tight_layout(pad=0.2)       # minimal padding
            fig_e.subplots_adjust(top=0.975, bottom=0.08, left=0.086, right=0.98)

            # --- Auto-scale limits to visible data only ---
            ax_e.autoscale(enable=True, axis='x', tight=True)
            ax_e.autoscale(enable=True, axis='y', tight=True)

            plt.show()
        else: #------------------------- real and imag parts are separated and ploted in linear scale---------------------------
            keep_idx_r = eigen_nz_cols(eigvals, 1e-12, 'r')
            keep_idx_i = eigen_nz_cols(eigvals, 1e-12, 'i')

            fig_e, (ax_real, ax_imag) = plt.subplots(2, 1, figsize=(8, 8), sharex=True, gridspec_kw={"hspace": 0.05})
            fig_e.canvas.header_visible = False
            fig_e.canvas.footer_visible = False
            fig_e.canvas.toolbar_visible = True

            ax_real.text(0.5, 1.008, rf"Eigenvalue Spectrum w.r.t. ${xlabel}$", ha='center', va='bottom', fontsize=10, fontweight='normal',
                transform=ax_real.transAxes, clip_on=False)                      # ensure it's not clipped by the axes box

            #fig_width = fig_e.get_size_inches()[0]
            #ax_real.text(0.5, 0.8, f"Figure width: {fig_width:.2f} in", ha='center', va='bottom', fontsize=10, fontweight='normal', transform=ax_real.transAxes, clip_on=False)

            # === Real parts ===
            #for i in range(eigvals_all.shape[1]):
            for i in keep_idx_r:
                ax_real.plot(swe_vals, eigvals[:, i].real, label=f"λ{i+1} Re", lw=glw, linestyle="-")

            ax_real.axhline(0, color='grey', lw=glw)
            ax_real.set_ylabel("Re(λ)")
            ax_real.yaxis.set_label_coords(-0.066, 0.46) # (horiz, vert)
            ax_real.xaxis.set_major_locator(mticker.AutoLocator())
            ax_real.xaxis.set_minor_locator(mticker.AutoMinorLocator())
            ax_real.yaxis.set_major_locator(mticker.AutoLocator())
            ax_real.yaxis.set_minor_locator(mticker.AutoMinorLocator(n=5))

            #ax_real.set_title(f"Eigenvalue Sweep wrt {sweep_param}")
            ax_real.grid(True, which='both', ls='--', lw=0.3)
            ax_real.legend(fontsize=8)

            # === Imaginary parts ===
            #for i in range(eigvals.shape[1]):
            for i in keep_idx_i:
                if not positive or eigvals[0, i].imag >=0:
                    ax_imag.plot(swe_vals, eigvals[:, i].imag, label=f"λ{i+1} Im", lw=glw, linestyle="-")

            if not positive:
                ax_imag.axhline(0, color="grey", lw=glw)

            ax_imag.set_xlabel(rf'${xlabel}$', fontproperties=fp, x=0.02)
            if base_x:
                ax_imag.text(0.40, -0.07, f"(center: {base_x:.10f})", transform=ax_imag.transAxes, ha="left", va="top", fontsize=9, color="blue")
                btn_ax1 = fig_e.add_axes([0.67, 0.031, 0.052, 0.020])  #[left, bottom, width, height]
                set_param_btn1 = mButton(btn_ax1, "set", hovercolor='0.975')
                set_param_btn1.on_clicked(on_set_base_clicked)

            if stability_lim:
                ax_imag.text(0.40, -0.12, f"(bifurc : {stability_lim:.10f})", transform=ax_imag.transAxes, ha="left", va="top", fontsize=9, color="brown")
                btn_ax2 = fig_e.add_axes([0.67, 0.006, 0.052, 0.020])  #[left, bottom, width, height]
                set_param_btn2 = mButton(btn_ax2, "set", hovercolor='0.975')
                set_param_btn2.on_clicked(on_set_bifurc_clicked)

            #ax_imag.set_xlabel(
            #    f"{xlabel}  (center=$\\small{{{base_x:.5f}}}$)",
            #    fontproperties=fp
            #)
            #ax_imag.set_xlabel(f"{xlabel} $\small A$ (center={base_x:.5f})", fontproperties=fp)
            ax_imag.set_ylabel("Im(λ)")
            ax_imag.yaxis.set_label_coords(-0.06, 0.46) # (horiz, vert)
            ax_imag.yaxis.set_major_locator(mticker.AutoLocator())
            ax_imag.yaxis.set_minor_locator(mticker.AutoMinorLocator(n=5))

            ax_imag.grid(True, which='both', ls='--', lw=0.5)
            ax_imag.legend(fontsize=8)

            for ax in (ax_real, ax_imag):
                ax.tick_params(axis='both', which='major', labelsize=8)
                ax.tick_params(axis='both', which='minor', length=3)
                ax.grid(False, which='minor')

                # draw a vertical line at base_x on BOTH axes
                if base_x:
                    ax.axvline(x=base_x, color='blue', linestyle='dotted', linewidth=0.8, zorder=5)
                if stability_lim:
                    ax.axvline(x=stability_lim, color='C3', linestyle='dotted', linewidth=0.8, zorder=5)

            #fig_e.tight_layout(pad=0.1)
            fig_e.subplots_adjust(hspace=0.01, top=0.970, bottom=0.08, left=0.086, right=0.99)
            ax_real.autoscale(enable=True, axis='x', tight=True)
            ax_imag.autoscale(enable=True, axis='x', tight=True)
            #fig_e.canvas.mpl_connect("draw_event", update_ticks)
            plt.show()

            cid = fig_e.canvas.mpl_connect('button_press_event', on_click_fig_e)
            add_button_tooltip(fig_e, btn_ax1, "Update the swept parameter with this center value.")
            add_button_tooltip(fig_e, btn_ax2, "Set the swept parameter to its stability/bifurcation limit.")

 # —————————————————————————————————————————————————————————————————————————————————
def odd_of(n):
    return n if n % 2 == 1 else n + 1

# —————————————————————————————————————————————————————————————————————————————————
#return the index of imaginar parts of eingenvalues bifurcates
def find_forking_point(eigvals_all, from_left)->int:
    # returns row index of the input array if diverging point found
    # otherwise -1 or -2
    use_zeros = False
    a_im = eigvals_all[:, 1].imag
    b_im = eigvals_all[:, 3].imag

    # Compare with tolerance
    same = np.isclose(a_im, b_im, atol=1e-6, rtol=1e-6)

    if same.any() == False:
        # additionally force "same" wherever b_im == 0
        same = np.isclose(b_im, 0.0, atol=1e-12)
        use_zeros = True
        #with eigen_num:
        #   print (same)

    diff_idx = np.where(~same)[0]
    #with eigen_num:
    #    print (diff_idx)
    #with eigen_box:
    #    print("find_forking_point")
    if diff_idx.size == 0:
        y= a_im - b_im
        indexes = [i for i in range(1, len(y)-1) if (y[i-1] > y[i] and y[i] < y[i+1])]
        if len(indexes)>0:
            return indexes[0] if from_left else indexes[-1]
        return -1
        #last_equal = eigvals_all.shape[0] - 1
        #note = "they never differ"
    elif diff_idx[0] == 0 and diff_idx[-1] == 0:
        y= a_im - b_im
        indexes = [i for i in range(1, len(y)-1) if (y[i-1] > y[i] and y[i] < y[i+1])]
        if len(indexes) > 0:
            return indexes[0] if from_left else indexes[-1]
        return -2;
        #last_equal = None
        #note = "they differ from the first row"
    else:
        #with eigen_box:
        #    print(f"forking: i_F={diff_idx[0]}, i_L={diff_idx[-1]}")
        if use_zeros:
            return diff_idx[-1] if diff_idx[0] == 0 else diff_idx[0] # ?
        else:
            return diff_idx[0] if abs(a_im[diff_idx[0]] - b_im[diff_idx[0]]) < abs(a_im[diff_idx[-1]] - b_im[diff_idx[-1]]) else diff_idx[-1]
        #last_equal = diff_idx[0] - 1
        #note = "found first difference at row index {}".format(diff_idx[0])
        #return 0

# —————————————————————————————————————————————————————————————————————————————————
def show_sweep_status(swp_range):
    err = 'no error'
    if eigen_sts_res.value[0]!='W':
        err = eigen_sts_res.value
    eigen_sts_res.value = rf'Calculation of eigenvalues over the sweep range [{swp_range[1]:.8e} to {swp_range[2]:.8e}] for the parameter $\;{eig_param_labels2[sweep_param_dd.value]}\;$ is completed ({err}).'


# —————————————————————————————————————————————————————————————————————————————————
# Calculate array of eigenvalues by sweeping a parameter and plot
def sweep_eigenvalues(sweep_param, beg_pct, end_pct, steps, zoom_on_bifur)-> Tuple[float, float, float]:
    global eigvals_all, sweep_vals, base_x, stab_lim

    with eigen_resL:
        clear_output()  # wait=True
    with eigen_resR:
        clear_output()  # wait=True
    try:
        theta_vals = np.empty(steps, dtype=float)

        stab_lim = None
        positive = False
        if zoom_on_bifur and not eigvals_all is None:
            positive = True
            idx = find_forking_point(eigvals_all, end_pct >= beg_pct)
            if idx>=0:
                stab_lim = sweep_vals[idx]
                #if idx > 0:spect_theta
                #    stab_lim = (stab_lim + sweep_vals[idx-1])/2

                rel_range = end_pct - beg_pct
                if rel_range > 50:
                    rel_range = 20
                elif rel_range > 10:
                    rel_range = 2
                elif rel_range > 1e-8:
                    rel_range *= 0.1

                rel_range = round(rel_range, 11)

                beg_pct= -rel_range/2
                end_pct = beg_pct + rel_range
                beg_range_box.value = beg_pct
                end_range_box.value = end_pct
                #positive = True

        calc_res = calculate_eigenvalues(sweep_param, theta_vals, beg_pct, end_pct, steps, stab_lim)
        if calc_res is None:
            return None

        base_x = calc_res[0]

        xlabel = eig_param_labels.get(sweep_param)

        idx = find_forking_point(eigvals_all, end_pct >= beg_pct)

        if idx>=0:
            stab_lim = sweep_vals[idx]
            #if idx > 0:
            #    stab_lim = (stab_lim + sweep_vals[idx-1])/2
        else:
             stab_lim = None

        plot_eigenvalues(eig_param_labels2[sweep_param], sweep_vals, eigvals_all, base_x, stab_lim, eigen_resL, False, positive)

        # === display a table of imaginary parts of swept eingenvalues ===
        with eigen_resR:
            kidx_re = eigen_nz_cols(eigvals_all, 1e-12, 'r')
            kidx_im = eigen_nz_cols(eigvals_all, 1e-12, 'i')

            # === Prepare data ===
            sweep_arr = np.asarray(sweep_vals)
            theta_arr = np.asarray(theta_vals, dtype=float)

            # pre-extract real/imag matrices
            eig_re = eigvals_all.real
            eig_im = eigvals_all.imag

            data_re = {xlabel: sweep_vals, "𝜃": theta_vals}
            #data_re = {xlabel: sweep_arr, "𝜃": theta_arr}
            data_im = data_re.copy()

            data_re.update({f"Re(λ{idx+1})": eig_re[:, idx] for idx in kidx_re})
            data_im.update({f"Im(λ{idx+1})": eig_im[:, idx] for idx in kidx_im})

            # Build DataFrames
            df_re = pd.DataFrame(data_re)
            df_im = pd.DataFrame(data_im)

            df_re.rename(columns={sweep_param: xlabel}, inplace=True)
            df_im.rename(columns={sweep_param: xlabel}, inplace=True)

            df_re = df_re.apply(lambda col: pd.to_numeric(col, errors='coerce'))
            df_im = df_im.apply(lambda col: pd.to_numeric(col, errors='coerce'))

             # find index of row whose first-column value is closest to target_value
            closest_idx = (df_im[xlabel] - float(base_x)).abs().idxmin()

            #for col in df.columns:
            #    try:
            #        if np.issubdtype(df[col].dtype, np.number):
            #            df_real[col] = df[col].apply(lambda v: np.real(v))
            #            df_imag[col] = df[col].apply(lambda v: np.imag(v))
            #    except Exception:
            #        pass

            # styling function: highlight only the closest row
            def _highlight_row(row):
                if row.name == closest_idx:
                    return ['background-color: #40c8ff'] * len(row)  # #4ddbff
                return [''] * len(row)
            #************************************************************************************#
            # helper to build styled HTML while removing Styler auto-id "T_..." that blocks CSS
            def styled_table_html(df_view, view_suffix):
                sid = f'eig_table_{view_suffix}'
                styler = df_view.style
                styler = styler.apply(_highlight_row, axis=1)
                # apply formatting map if provided; otherwise use default float_format in to_html later
                if fmt_map is not None:
                    styler = styler.format(fmt_map)
                # set the desired id attribute for the table
                styler = styler.set_table_attributes(f'id="{sid}"')
                # produce html and remove auto-generated id="T_xxx" inserted by Styler
                styler_h = styler.to_html()
                # remove only the Styler auto id attribute like id="T_...". Keep the id we set above.
                styler_h = re.sub(r'(<table\b[^>]*?)\s+id="T_[^"]+"', r'\1', styler_h, count=1, flags=re.IGNORECASE)
                return styler_h
            #************************************************************************************#
            #styler = (df.style
            #    .apply(_highlight_row, axis=1)
            #    .format(fmt_map)
            #    #.hide_index()
            #    .set_table_attributes('id="eig_table"')  # keep your table id
            #)

            fmt_map = {
                col: (
                    "{:.7e}" if i == 0 else
                    "{:.8f}" if i == 1 else
                    "{:.6f}"
                )
                for i, col in enumerate(df_im.columns)
            }

            # produce styled HTML fragments
            html_real = styled_table_html(df_re, 're')
            html_imag = styled_table_html(df_im, 'im')

            table_title = "Eigenvalues - Real Parts"

            # === Scrollable container with sticky header ===
            scrollable_html = f"""
<style>
.eig-output-root {{ margin-right: -8px; }}
/* target any table whose id begins with "eig_table_" (covers Styler ids) */
table[id^="eig_table_"] {{
  border-collapse: collapse;
  width: 100%;
  table-layout: fixed;
}}

/* table cells */
table[id^="eig_table_"] th,
table[id^="eig_table_"] td {{
  border: 0 solid #888;
  padding: 0 0;
  text-align: right;
  font-family: monospace;
  font-size: 9pt;
  height: 10px !important;
  line-height: 10px !important; /* 12=63 lines, 13 = 58 lines, 14=53 lines */
  overflow: hidden;
  white-space: nowrap;
}}

/* sticky header */
table[id^="eig_table_"] thead th {{
  position: sticky;
  top: 0;
  background-color: #333;  /* dark header background */
  color: white;            /* white text */
  font-weight: 600;
  text-align: center;
  height: 16px;
  z-index: 2;
}}

/* first column(s) size hints (adjust nth-child index if needed) */
table[id^="eig_table_"] thead th:first-child,
table[id^="eig_table_"] tbody th {{
  width: 30px;
  min-width: 30px;
  max-width: 40px;
  box-sizing: border-box;
  padding-right: 2px;
  text-align: right;
  white-space: nowrap;
  overflow: hidden;
}}

/* parameter column has exponent format and needs be larger */
table[id^="eig_table_"] td:nth-child(2),
table[id^="eig_table_"] th:nth-child(2) {{
    width: 100px;
    box-sizing: border-box;
}}

/* the theta column */
table[id^="eig_table_"] td:nth-child(3),
table[id^="eig_table_"] th:nth-child(3) {{
    width: 78px;
    box-sizing: border-box;
}}

table[id^="eig_table_"] thead th {{
  text-align: center !important;
}}

/* separate eigenvalues column from columns at their left */
table[id^="eig_table_"] td:nth-child(3) {{
  border-right: 1px solid #aaa; /* heavier vertical line */
}}

/* small title above the table */
.table-title {{
  display: flex;
  align-items: center;
  justify-content: space-between;
  font-family: sans-serif;
  font-size: 13px;
  font-weight: 600;
  margin-top: 6px;
  margin-bottom: 0;
  margin-left: 2px;
  margin-right: 2px;

  color: color-mix(in srgb, CanvasText 80%, Canvas 20%);
  min-width: 554px;
  width: 554px;
}}

/* left side wrapper stacks title and small subtitle - not used now!! */
.table-title .title-block {{
  display: flex;
  align-items: center;
  gap: 12px;
  min-width: 0; /* allow truncation if container is narrow */
}}

/* tab buttons */
.tabbar {{
  display:inline-flex;
  gap:6px;
  align-items:center;
  margin-right:2px;
  margin-bottom:3px;
}}

.tabbar button {{
  background:ButtonFace;
  border:0px;
  padding:2px 7px;
  font-size:12px;
  cursor:pointer;
}}

.tabbar button.tab-active {{
  background:Highlight;
  color:HighlightText;
  border-color:ButtonBorder;
}}

/* THE FIX: hide inactive view, show active one */
.table-view {{ display: none; }}
.table-view.active {{ display: block; }}

/* container scrolling */
.scroll-container {{
  margin-right: 0;
  margin-top: 3px;  /* 0 */
  max-height: 774px;
  min-width: 554px;
  max-width: 554px;
  width: 554px;
  overflow-y: auto;
  overflow-x: auto;
  border: 1px solid #888;
}}
</style>

<script>
(function() {{
  const tabReal = document.getElementById('eig_table_tab_re');
  const tabImag = document.getElementById('eig_table_tab_im');
  const viewReal = document.getElementById('eig_table_view_re');
  const viewImag = document.getElementById('eig_table_view_im');
  const titleEl = document.getElementById('eig_table_title');
  const TITLE_REAL = 'Eigenvalues - Real Parts';
  const TITLE_IMAG = 'Eigenvalues - Imaginary Parts';

  function center_base_row() {{
    setTimeout(() => {{
      const idxEl = document.querySelector('[id$="_level0_row' + {closest_idx} + '"]');
      if (idxEl) idxEl.closest('tr').scrollIntoView({{ block: 'center', inline: 'nearest' }});
    }}, 30);
  }}

  function activate(which) {{
    if (which === 're') {{
      tabReal.classList.add('tab-active');
      tabImag.classList.remove('tab-active');
      tabReal.setAttribute('aria-selected','true');
      tabImag.setAttribute('aria-selected','false');
      viewReal.classList.add('active');
      viewImag.classList.remove('active');
      if (titleEl) titleEl.textContent = TITLE_REAL;
    }} else {{
      tabImag.classList.add('tab-active');
      tabReal.classList.remove('tab-active');
      tabImag.setAttribute('aria-selected','true');
      tabReal.setAttribute('aria-selected','false');
      viewImag.classList.add('active');
      viewReal.classList.remove('active');
      if (titleEl) titleEl.textContent = TITLE_IMAG;
    }}
    // keep scroll position consistent (optional)
    // document.querySelector('.scroll-container').scrollTop = 0;

    localStorage.setItem('eig_table_view', which);
    center_base_row();
  }}

  if (tabReal && tabImag) {{
    tabReal.addEventListener('click', () => activate('re'));
    tabImag.addEventListener('click', () => activate('im'));
  }}

  const storedView = (() => {{
    try {{ return localStorage.getItem('eig_table_view'); }} catch(e) {{ return null; }}
  }})();
  if (storedView === 're' || storedView === 'im') {{
    // delay activation slightly so DOM elements are present
    setTimeout(() => activate(storedView), 100);
  }}

  // keyboard support
  document.addEventListener('keydown', (ev) => {{
    if (ev.key === 'ArrowLeft') activate('re');
    if (ev.key === 'ArrowRight') activate('im');
    if (ev.key === 'r' || ev.key === 'R') activate('re');
    if (ev.key === 'i' || ev.key === 'I') activate('im');
  }});
}})();
</script>

<div class="eig-output-root">
<div class="table-title">
  <div style="display:flex; gap:12px; align-items:center;">
    <div id="eig_table_title">{table_title}</div>
    <div style="font-size:11px; color:#666;">(click buttons at the right to switch)</div>
  </div>
  <div class="tabbar" role="tablist" aria-label="Show real or imaginary parts">
    <button id="eig_table_tab_re" class="tab-active" role="tab" aria-selected="true">Real</button>
    <button id="eig_table_tab_im" role="tab" aria-selected="false">Imag</button>
  </div>
</div>
<div class="scroll-container">
  <div id="eig_table_view_re" class="table-view active" role="region" aria-label="Real parts table">
    {html_real}
  </div>
  <div id="eig_table_view_im" class="table-view" role="region" aria-label="Imaginary parts table">
    {html_imag}
  </div>
</div>
</div>
"""
            display(HTML(scrollable_html))

        return calc_res
    except Exception as e:
        eigen_sts_res.value = str(e)
        return None
# —————————————————————————————————————————————————————————————————————————————————
def save_sweep_params(_=None):
    global calc_psd_from_cos_theta
    if current_rec_key:
        subkeys=[
          "sweep_name",
          "sweep_beg",
          "sweep_end",
          "sweep_steps",
          "sweep_cos"]

        values=[
            sweep_param_dd.value,
            beg_range_box.value,
            end_range_box.value,
            step_box.value,
            calc_psd_from_cos_theta]

        update_record_subkeys(current_rec_key, subkeys, values)

# —————————————————————————————————————————————————————————————————————————————————styl
# === User interface ===
# --- globals (module-scope) ---
last_plot_beg = -10
last_plot_end = 10

user_edited_beg = True
user_edited_end = True

suppress_observers = False   # set True while programmatically changing widgets

sweep_param_dd = Dropdown(
    options=eig_param_options,
    value='gamma',
    description='Sweep ',
    layout=Layout(width='105px', margin='5px 3px 0 0'),
    style= {'description_width': '41px'}) #'font_size': '12px'
sweep_param_dd.add_class("big-font")

beg_range_box = FloatText(
    value=last_plot_beg, description='Begin %',
    layout=Layout(width='155px', margin='5px 3px 0 0'),
    style= {'description_width': '60px'})
beg_range_box.add_class("big-font")

end_range_box = FloatText(
    value=last_plot_end, description='End %',
    layout=Layout(width='142px', margin='5px 3px 0 0'),
    style= {'description_width': '47px'})
end_range_box.add_class("big-font")

zoom_i_bt = Button(description="+", style={'button_color':'#42c5f5', 'text_color':'Canvas', 'font_weight':'800'}, layout=Layout(width='16px', margin='5px 2px 0 5px', padding='0'), tooltip='Increase the range (zoom out) by 2x') # top right bottom left
zoom_o_bt = Button(description="−", style={'button_color':'#42c5f5', 'text_color':'Canvas', 'font_weight':'800'}, layout=Layout(width='16px', margin='5px 2px 0 0', padding='0'), tooltip='Decrease the range (zoom in) by 1/2') # top right bottom left
zoom_R_bt = Button(description="↺", style={'button_color':'#42c5f5', 'text_color':'Canvas', 'font_weight':'800'}, layout=Layout(width='16px', margin='5px 0 0 0', padding='0'), tooltip='Reset the range to (10, -10)') # top right bottom left

# ————————————————————————————————————————
# helper: observer callbacks for FloatText
def _on_eig_beg_change(change):
    global user_edited_beg, last_plot_beg, suppress_observers
    if change['name'] != 'value':
        return
    if suppress_observers:
        # programmatic update — treat as NOT a user edit
        return
    # user changed the value manually
    user_edited_beg = True

def _on_eig_end_change(change):
    global user_edited_end, last_plot_end, suppress_observers
    if change['name'] != 'value':
        return
    if suppress_observers:
        return
    user_edited_end = True

def eig_range_scale(scale):
    global last_plot_beg, last_plot_end
    if scale == 0:
        last_plot_beg = -10
        last_plot_end = 10
    else:
        last_plot_beg *= scale
        last_plot_end *= scale
    user_edited_beg = user_edited_end = True
    beg_range_box.value = last_plot_beg
    end_range_box.value = last_plot_end

def _on_eig_range_plus(_):
    eig_range_scale(2.0)

def _on_eig_range_minus(_):
    eig_range_scale(0.5)

def _on_eig_range_reset(_):
    eig_range_scale(0)

beg_range_box.observe(_on_eig_beg_change, names='value')
end_range_box.observe(_on_eig_end_change, names='value')

zoom_i_bt.on_click(_on_eig_range_plus)
zoom_o_bt.on_click(_on_eig_range_minus)
zoom_R_bt.on_click(_on_eig_range_reset)
# ————————————————————————————————————————

step_box = IntText(
    value=101,
    description="Steps",
    layout=Layout(width="112px", margin="5px 3px 0 0"),
    style={"description_width": "42px"})

eigen_sweep_plot_btn = Button(
    description="Plot",
    layout=Layout(width='58px', margin='5px 0 0 5px'),
    button_style='info',
    tooltip='Generate plot of eigenvalues with respect to the selected parameter and its range.',
    icon='play')

eigen_sweep_plot_btn.add_class("big-font")

zoom_bifurcation_btn = Button(
    description="Zoom on Bifurcation",
    layout=Layout(width='155px', margin='5px 0 0 5px'),
    button_style='info',
    tooltip='Center and zoom the plot around bifurcation of eigenfrequencies',
    icon='search')

save_sweep_btn= Button(
    description="Save Config",
    layout=Layout(width='115px', margin='5px 0 0 5px'),
    button_style='success',
    tooltip='Save the sweeping configuration in the current record.',
    icon='save')

save_sweep_btn.on_click(save_sweep_params)

plot_mode_toggle = ToggleButtons(
    options=[('Sep', 'separate'),('Com', 'combined')],
    value='separate',
    description='',
    layout=Layout(width='104px', margin='4px 0 0 5px'),        # container width
    style={'button_width': '48px'}                             # each button width
)
eigen_sts_lbl = Label(value='Status:',layout=Layout(margin='0 2px 0 0px'), style={'font_weight':'400'})
eigen_sts_res = Label(value='Ready',  layout=Layout(margin='0 0 0 5px'), style={'font_weight':'600'})

# —————————————————————————————————————————————————————————————————————————————————
def _on_eig_plot_mode_change(change):
    global eigvals_all, sweep_vals
    if change['name'] != 'value':
        return
    mode = change['new']
    with eigen_resL:
        clear_output(wait=True)

    if eigvals_all is None or sweep_vals is None:
        with eigen_resL:
            display("No eigenvalues available. Run the sweep first.")
    else:
        plot_eigenvalues(eig_param_labels.get(sweep_param_dd.value), sweep_vals, eigvals_all, None, None, eigen_resL, (mode == 'combined'), False)

plot_mode_toggle.observe(_on_eig_plot_mode_change)

# —————————————————————————————————————————————————————————————————————————————————
def run_eigen_sweep_plot(_=None):
    global last_plot_beg, last_plot_end, user_edited_beg, user_edited_end

    suppress_observers = True
    eigen_sts_res.value = 'Working...'
    try:
        if last_plot_beg is not None and not user_edited_beg:
            beg_range_box.value = last_plot_beg
        else:
            last_plot_beg = float(beg_range_box.value)

        if last_plot_end is not None and not user_edited_end:
            end_range_box.value = last_plot_end
        else:
            last_plot_end = float(end_range_box.value)
    finally:
        suppress_observers = False

    user_edited_beg = False
    user_edited_end = False

    swp_range = sweep_eigenvalues(sweep_param_dd.value, last_plot_beg, last_plot_end, max(1, odd_of(step_box.value)), False)
    show_sweep_status(swp_range)

# —————————————————————————————————————————————————————————————————————————————————
def zoom_on_bifurcation(button=None):
    global suppress_observers
    suppress_observers = True
    eigen_sts_res.value = 'Working...'

    try:
        swp_range = sweep_eigenvalues(sweep_param_dd.value, beg_range_box.value, end_range_box.value, max(1, odd_of(step_box.value)), True)
    finally:
        suppress_observers = False
        show_sweep_status(swp_range)

# —————————————————————————————————————————————————————————————————————————————————
zoom_bifurcation_btn.on_click(zoom_on_bifurcation)
eigen_sweep_plot_btn.on_click(run_eigen_sweep_plot)

ui = VBox([HBox([sweep_param_dd, beg_range_box, end_range_box, zoom_i_bt, zoom_o_bt, zoom_R_bt, step_box, eigen_sweep_plot_btn, zoom_bifurcation_btn, plot_mode_toggle, save_sweep_btn],
                layout=Layout(margin='0', padding='0')),
          HBox([eigen_sts_lbl, eigen_sts_res],layout=Layout(margin='5px 0 0 4px', padding='0'))])

with eigen_num:
    display(Markdown(r"""<hr id='eigenvalue-analysis' style='width:1340px; margin:1em auto 1em 0; height:2px; border:none;'>

### Eigenvalue Analysis


<style>
.big-font select {
  font-size: 12pt !important;
  padding-left: 4px !important;
  padding-right: 4px !important;
}

.zoom-adjust .widget-label {
    font-weight: 800 !important;
    font-size: 18px !important;
    line-height: 14px !important;   /* reduces vertical spacing */
    margin-top: -4px !important;    /* moves '+' upward */
    padding: 0 !important;
}
</style>

<div class='sb' style='max-width:1020px;'>This section allows one to calculate eigenvalues of the state matrix $A$ by varying the selected sweep parameter over the specified range,
while one provides or adjusts the initial condition ($x_0$) vector in order to obtain the equilibrium state (if possible). Results are shown in a plot where the x-axis is the swept
parameter and the y-axis shows eigenvalues. The imaginary parts of eigenvalues might correspond to the system’s natural frequencies. Matrix $A$ has six eigenvalues but the first one
is zero, and the sixth one's real part may be non-zero in the presence of damping. If any eigenvalue stays zero for the entire swept range, it will be excluded.<br><br>

Note that eigenvalues are valid only when all state variables are at their equilibrium values. In the absence of damping, the rotation phase $\phi$ is equal to $\pi$ at equilibrium;
with damping it shifts slightly depending on the damping ratio (See Section C). This value is automatically calculated here on each step of the sweep range. There is also a button 
labeled $[\phi_0 = f(\xi, \ldots)]$ on the simulation section for setting $\phi$ initial value below the damping entry.<br>

The array containing calculated eigenvalues and eigenvectors for the swept range is available as <span style="font-family:monospace; font-weight:normal; font-size:10pt;">
eigvals_all, eigvecs_all</span> and the selected parameter swept steps are in <span style="font-family:monospace; font-weight:normal; font-size:10pt;">sweep_vals</span> for further evaluation.
</div>
"""))

    display(ui)

# RESEARCH CONTENT / ABSTRACT
research_box = Output(layout=Layout(padding='0', margin='0', overflow_x='visible', width='1040px'))
# QUICK GUIDE
guide_box = Output(layout=Layout(padding='0', margin='0', overflow_x='visible', width='1040px'))
# EQUATIONS OF MOTION
r_matrix_out = Output(layout=Layout(padding='0', margin='0', width='1150px'))
# DERIVATION OF EQUATIONS OF MOTION
derivation_out = Output(layout=Layout(padding='0', margin='0', overflow_x='visible', width='1150px'))   # ← allow the math to render without a scrollbar
# Evaluating State Variables in the Presence of Damping
damping_out = Output()
# Semi-empirical Stability Criterion
semi_empiric_box =  Output()


#hr = W.HTML( value="<hr style='border:none; border-top:2px solid #909090; margin:0; width:100%;'>",layout=Layout(width='99.7%'))

accor_research = Accordion(children=[research_box],layout=Layout(padding='0', margin='0 0 4px 0',width='99.7%'))
accor_research.set_title(0, '📖  Abstract/Research Context')

accor_guide = Accordion(children=[guide_box],layout=Layout(padding='0', margin='0 0 4px 0',width='99.7%'))
accor_guide.set_title(0, '📖  Quick Guide')

accor_drv = Accordion(children=[derivation_out], layout=Layout(padding='0', margin='0 0 4px 0',width='99.7%'))
accor_drv.set_title(0, 'A.\u2003Derivation of Equations of Motion')

accor_eqs = Accordion(children=[r_matrix_out], layout=Layout(padding='0', margin='0 0 4px 0', width='99.7%'))
accor_eqs.set_title(0, 'B.\u2003Basic Definitions and Equations of Motion')

accor_sup1 = Accordion(children=[damping_out],layout=Layout(padding='0', margin='0 0 4px 0',width='99.7%'))
accor_sup1.set_title(0, 'C.\u2003Evaluating State Variables in the Presence of Damping')

accor_eigen = Accordion(children=[VBox([eigen_box, nat_freq_box, eigen_num,
                                        HBox([eigen_resL, eigen_resR],layout=Layout(padding='0', margin='0'))
                                       ])],layout=Layout(padding='0', margin='0 0 4px 0',width='99.7%'))
accor_eigen.set_title(0, 'D.\u2003Linearization and Eigenvalue Analysis')

accor_empiric = Accordion(children=[semi_empiric_box],layout=Layout(padding='0', margin='0 0 4px 0',width='99.7%'))
accor_empiric.set_title(0, 'E.\u2003Semi-empirical Stability Criterion')

title_out = Output(layout=Layout(padding='0', margin='0'))
with title_out:
    display(top_title)

js_out = Output(layout=dict( height="1px", width="1px", padding="0", margin="0", border="0", overflow="hidden"))    # hide overflow and disable scrollbars
display(js_out)


btns_combo= HBox([bt_psi_om_cos_theta, bt_psi_eq_om])

display(HTML("""
<style>
/* Reduce vertical padding */
.widget-toggle-buttons.tight-toggle button {
    padding-top: 1px !important;
    padding-bottom: 1px !important;
    font-size: 12px !important;
    line-height: 1 !important;
    height: 24px !important;
}
.smallpad-btn .widget-toggle-button {
    padding: 0 !important;
    min-height: 0 !important;
    line-height: 1 !important;
}
</style>
"""))

unit_btn = W.ToggleButtons(
    options=['Rad','Deg'],
    value='Deg' if DEGREE_MODE else 'Rad',
    layout=Layout(height='24px', width='80px', margin='0', padding='0'),
    style={'button_width': '36px', 'description_width': 'initial'},
    _dom_classes=['smallpad-btn'],
    tooltip="Show anlge and angular velocities in radians (radians/seconds) or degrees (revolution/per second) in parameter fields."
)

# Optional: eliminate internal padding with some CSS tweaks
unit_btn.add_class("tight-toggle")

def unit_btn_handler(change):
    global deg_mode_action_suppress
    if deg_mode_action_suppress:
        return
    set_RadDeg_mode(change['new'] == 'Deg', True)

def set_RadDeg_mode(deg: bool, flag):
    global DEGREE_MODE
    save_w_link = w_link.value
    w_link.value = False
    DEGREE_MODE = deg
    # Switch all angles to degrees or to radians:
    AngleText.update_all_modes()
    LabeledFloatA.update_all()
    # Switch all ang‑vel to rad/s or to cycles/sec:
    AngVelocityText.update_all_modes()
    LabeledFloatB.update_all()
    w_link.value = save_w_link
    if flag:
        show_nu_vector()
        display_physics_params()

unit_btn.observe(lambda change: set_RadDeg_mode(change['new']=='Deg', True), names='value')

hdr_setup_params    = Label(value=r'\(\text{Setup Parameters}\)',   layout=Layout(padding='5px 0 0 35px', width='auto'))
hdr_initial_angles  = Label(value=r'\(\text{Initial Angles}\)',     layout=Layout(justify_self='start',padding='5px 0 0 35px', width='auto'))
hdr_initial_vel     = Label(value=r'\(\text{Initial Velocities}\)', layout=Layout(justify_self='start',padding='5px 0 0 35px', width='auto'))
hdr_test_params     = Label(value=r'\(\text{Damping \& Test Params}\)',   layout=Layout(justify_self='start',padding='5px 0 0 30px', width='auto'))

w_csv_count         = W.HTML(value='', layout=Layout(margin='2px 0 0 8px'))
w_mp4_count         = W.HTML(value='', layout=Layout(margin='2px 0 0 8px'))

def place(wid, row, col, align='start'):
    wid.layout.grid_area = f"r{row}c{col}"
    wid.layout.justify_self = align  # 'start', 'center', or 'end'
    return wid

def clear_csv_files(button):
    global csv_files, csv_filename
    lb_result_sim.value = ''
    csv_files.clear()
    csv_filename = None
    update_csv_count()

def clear_mp4_files(button):
    global mp4_filename, mp4_files
    lb_result_ani.value = ''
    mp4_files.clear()
    mp4_filename = None
    update_mp4_count()

def make_ck(desc, tip, value=True, width='90px'):
    ck = Checkbox(
        value=value,
        description=desc,
        tooltip=tip,
        layout=Layout(margin='0', width=width),
        style={'description_width': '0'}
    )
    return ck
    
btn_clear_csvs =  Button(description='Clear', button_style='warning', layout=Layout(width='40px', height = '26px', padding='0', margin='0 0 0 5px'),
    tooltip="Clear the list of simulation results within the project.")
btn_clear_mp4s =  Button(description='Clear', button_style='warning', layout=Layout(width='40px', height = '26px', padding='0', margin='0 0 0 5px'),
    tooltip="Clear the list of animation results within the project.")

btn_clear_csvs.on_click(clear_csv_files)
btn_clear_mp4s.on_click(clear_mp4_files)

ck_th      = Checkbox(value=True,  layout=Layout(margin='0', width='85px'), style={'description_width': '0'}, description=r'$\theta$')
ck_php     = Checkbox(value=False, layout=Layout(margin='0', width='85px'), style={'description_width': '0'}, description=r'$\theta ∠$')
ck_phd_psd = Checkbox(value=False, layout=Layout(margin='0', width='85px'), style={'description_width': '0'}, description=r'$\dot\phi,\,\dot\psi$')
ck_nuXY    = Checkbox(value=False, layout=Layout(margin='0', width='85px'), style={'description_width': '0'}, description=r'$\nu_X,\,\nu_Y$')
ck_nuZ     = Checkbox(value=False, layout=Layout(margin='0', width='85px'), style={'description_width': '0'}, description=r'$\nu_Z$')
ck_kinE    = Checkbox(value=False, layout=Layout(margin='0', width='85px'), style={'description_width': '0'}, description=r'🚀$T$')
ck_magE    = Checkbox(value=False, layout=Layout(margin='0', width='85px'), style={'description_width': '0'}, description=r'🧲$V$')
ck_totE    = Checkbox(value=False, layout=Layout(margin='0', width='85px'), style={'description_width': '0'}, description=r'$T\!+\!V$')
ck_diss    = Checkbox(value=False, layout=Layout(margin='0', width='85px'), style={'description_width': '0'}, description=r'♨ $\xi$')
ck_polar   = Checkbox(value=False, layout=Layout(margin='0', width='85px'), style={'description_width': '0'}, description=r'$\Large{\oplus}_{\Large{\theta,\,\phi}}$')
ck_spher   = Checkbox(value=False, layout=Layout(margin='0 10px 0 0', width='90px'), style={'description_width': '0'}, description=r'$\Large{🌐}_{\Large{\theta,\,\phi}}$')

phase_portr_dd = Dropdown(
    options=[("—", 0), ("𝜽, 𝜽'",  1), ("𝝓-𝝎𝒕, 𝝓˙", 2), ("𝝍, 𝝍˙", 3), ("𝜽, T",  4), ("𝜽, V",  5), ("𝜽, H",  6)],
    value=0,
    description='Phase portrait',
    layout=Layout(width='155px', margin='0 0 0 0'),
    style= {'description_width': '80px'})

phase_portr_dd.add_class("sanino")

def on_phase_portr_select(change):
    global plot_portraits, I1_v, I2_v, I3_v, mB_v, om_B, gamma_v, xi_v, portrait_params
    val = change["new"]   # the newly selected value
    if val == 0:
        plots_out[11].layout.display = "None"
        if plot_portraits:
            portrait_params = plot_portraits.get_params()
        plot_portraits = None
    elif t_vals is not None:
        plot_flg[11] = False
        params = I1_v, I2_v, I3_v, mB_v, om_B, gamma_v, xi_v*(I1_v+I2_v)*0.5
        plot_sim_results(t_vals, y_vals[0], y_vals[1], y_vals[2], y_vals[3], y_vals[4], y_vals[5], params, 11)
  
phase_portr_dd.observe(on_phase_portr_select, names="value")
'''
ck_tooltips = [
    'Plots time evolution of theta angle and its frequency spectrum',
    'Plots the phase of the theta angle with the rotating field.',
    'Plots angular velocities of rotations phi and psi.',
    'Plots x and y components of angular velocity vector nu in the body frame.',
    'Plots z component of angular velocity vector nu in the body and the lab frame.',
    'Plots kinetic energy of the angular motion.',
    'Plots potential energy related to magnetic torque.',
    'Plots the total energy.',
    'Plots dissipation energy due to the damping defined by the parameter xi.',
    'Plots the magnetic moment vector in polar coordinates.',
    'Plots the magnetic moment vector in spherical coordinates mapped to a user defined plane.']
'''    
    
ck_plots = [ ck_th, ck_php, ck_phd_psd, ck_nuXY, ck_nuZ, ck_kinE, ck_magE, ck_totE, ck_diss, ck_polar, ck_spher]

for i, ck in enumerate(ck_plots):
    ck.observe(lambda change, i=i: on_toggle_plot_view(change, i), names="value")

#ck_boxes = [] 
#for ck, tip in zip(ck_plots, ck_tooltips):
#    ck_boxes.append(HBox([ck], layout=Layout(tooltip=tip, margin='0', padding='0')))
    
children = [
    # ROW 0: column headers
    place(hdr_setup_params, 0, 0), place(hdr_initial_angles, 0, 1), place(hdr_initial_vel, 0, 2), place(hdr_test_params, 0, 3), place(HBox([unit_btn], layout=Layout(justify_content='flex-end')),0,4),
    # ROW 1:
    place(box_om,      1, 0), place(box_th_eq,  1, 1), place(box_thd,        1, 2), place(box_ps_acc, 1, 3),
    # ROW 2:
    place(box_I1,      2, 0), place(box_th_eps, 2, 1), place(box_phd,        2, 2), place(box_xi,     2, 3),
    # ROW 3:
    place(box_I2,      3, 0), place(box_th0,    3, 1), place(box_psd,        3, 2), place(HBox([bt_set_phi_zero, bt_set_ext_psdd],layout=Layout(justify_content='flex-start', width='100%')),  3, 3),
    # ROW 4:
    place(box_I3,      4, 0), place(box_ph,     4, 1), place(w_psd_ratio_om, 4, 2), 
    # ROW 5:
    place(box_mB,      5, 0), place(box_ps,     5, 1), place(btns_combo,     5, 2),  place(w_link,     5, 3),
    # ROW 6:
    place(box_eps_gam, 6, 0),  place(box_t_start, 6, 1),  place(params_status1, 6, 2),
    # ROW 7:
    place(box_gam,     7, 0),  place(HBox([bt_load_time_instance,bt_load_from_end, ck_mod360_phi_ps], layout=Layout(justify_content='flex-start', width='195px', margin='0 0 0 32px', padding='0 0')),7,1),  place(params_status2, 7, 2)
]

grid_layout = Layout(
    display                = 'grid',
    grid_template_columns  = '250px 250px 250px 195px 100px',
    grid_template_rows     = 'auto auto auto auto auto auto auto auto',  # 8 rows
    padding                = '5px 5px',
    # Define the CSS‐style grid template (7 rows × 4 columns).
    # Notice how the final line repeats “r6c1” in columns 1 and 2.
    grid_template_areas    = '''
    "r0c0 r0c1 r0c2 r0c3 r0c4"
    "r1c0 r1c1 r1c2 r1c3 r1c3"
    "r2c0 r2c1 r2c2 r2c3 r2c3"
    "r3c0 r3c1 r3c2 r3c3 r3c3"
    "r4c0 r4c1 r4c2 r4c3 r4c3"
    "r5c0 r5c1 r5c2 r5c3 r5c3"
    "r6c0 r6c1 r6c2 r6c2 r6c2"
    "r7c0 r7c1 r7c2 r7c2 r7c2"
'''
)

# ——————————————————————
# (4) Create & Display the GridBox
# ——————————————————————
params_grid = GridBox(children=children, layout=grid_layout)

utility = VBox([W.HTML("Conversions between Euler rotations and angular velocities")], layout=Layout(margin='0'))
accor_utility = Accordion(children=[utility],layout=Layout(padding='0', margin='4px 0 4px 0', width='99.7%'))
accor_utility.set_title(0, 'F.\u2003Conversions')
ui = VBox([
        title_out,
        #Latex(r"Solution of the dynamics of a free body having magnetic moment →m subject to a rotating field →B\large \text{Solution of the dynamics of a free body having magnetic moment } \vec{m} \text{ subject to a rotating field } \vec{B}",layout=W.Layout(margin="0 0 0 0")),
        accor_research,
        accor_guide,
        accor_drv,
        accor_eqs,
        accor_sup1,
        accor_eigen,
        accor_empiric,
        params_grid,
        accor_utility,
        #hr,
        HBox([save_ui,
              VBox([
                  HBox([
                        HBox([W.HTML("<span class='sb'>Create Simulation and Plots:</span>", layout=Layout(margin='2px 0 1px 0')), w_csv_count], layout=Layout(width='311px')),
                        HBox([btn_clear_csvs, W.Box(layout=Layout(flex='1 1 auto')), w_id_hash], layout=Layout(width='220px'))
                       ], layout=Layout(width='535px')),
                  HBox([box_simsteps, box_simtime, btn_simulate, btn_load_sim, btn_del_sim], layout=Layout(margin='0')),
                  HBox([w_solver, box_maxstep,box_rel_tol,box_abs_tol],layout=Layout(margin='0')),
                  HBox([lb_sts, lb_result_sim]),
                  HBox([
                        HBox([W.HTML("<span class='sb'>Create Animation:</span>", layout=Layout(margin='2px 0 1px 0')), w_mp4_count], layout=Layout(width='205px')), #315
                        w_interpolate,
                        btn_clear_mp4s
                       ], layout=Layout(width='535px')),
                  HBox([box_ani_beg, box_ani_spa, btn_animate, btn_load_anim, btn_del_anim], layout=Layout(margin='0')),
                  HBox([box_ani_fps, box_ani_len, btn_preview], layout=Layout(margin='0')),
                  w_scene_opts,
                  HBox([progress_ani, box_result_ani], layout=Layout(height='36px', margin='2px 0 0 0'))
                  ])
             ]),
        #hr
    ])

params_resultsL = Output(layout=Layout(overflow='visible', margin='0 30px 0 0'))
params_resultsR = Output(layout=Layout(overflow='visible'))

plot_opts = HBox(ck_plots+[phase_portr_dd], layout=Layout(height='30px', margin='0 0 10px 0')) #ck_plots
    
#status_out = Output(layout=Layout(margin='0', padding='0'))

plots_out = [ Output(layout=Layout(margin='0', padding='0')) for _ in range(N_PLOTS+1) ]

plots_box = VBox(plots_out, layout=Layout(margin='0', padding='0', min_height='550px') )

video_out =  Output(layout=Layout(height='auto', overflow='auto', margin='10px 0 0 0', padding='0')) #,width 630px enough, border='1px solid lightgray'

accor_params = Accordion(children=[HBox([params_resultsL, params_resultsR], layout=Layout(height='240px'))], layout=Layout(padding='0', margin='0', width='99.7%')) #,layout=Layout(gap='50px',align_items='flex-start')),
accor_params.set_title(0, 'G.\u2003Physics Parameters')
accor_plots = Accordion(children=[VBox([plot_opts, plots_box], layout=Layout(padding='0', margin='0'))], layout=Layout(padding='0', margin='4px 0 0 0', width='99.7%')) # top right bottom left  #,layout=Layout(gap='50px',align_items='flex-start')),
accor_plots.set_title(0, 'H.\u2003Plots')
accor_scene = Accordion(children=[video_out], layout=Layout(padding='0', margin='4px 0 0 0', width='99.7%')) # top right bottom left  #,layout=Layout(gap='50px',align_items='flex-start')),
accor_scene.set_title(0, 'H.\u2003Animation')
page = VBox([ui,
    accor_params,
    #HBox([params_resultsL,params_resultsR],layout=Layout(height='200px')), #,layout=Layout(gap='50px',align_items='flex-start')),
    #iHTML( value="<hr style='border:none; border-top:2px solid #909090; margin:0; width:100%;'>",layout=Layout(width='99%')),
    #status_out,
    accor_plots,
    accor_scene
], layout=Layout(padding='0', margin='0'))
#page.layout = Layout(height='3000px', overflow_y='visible')  # force all children to show
set_params_from_widgets()
show_nu_vector()
display_physics_params()
display(page)

sim_save_dlg = SimSaveDialog(save_sim_choice)
display(sim_save_dlg.dialog)

anim_save_dlg = AnimSaveDialog(save_anim_choice)
display(anim_save_dlg.dialog)

csv_dialog = SelectDeleteDialog(
    title_select="Load a simulation result",
    title_delete="Delete a simulation result",
    on_select=on_csv_selected,
    on_delete=on_csv_delete,
    on_clean=on_csv_clean
)

mp4_dialog = SelectDeleteDialog(
    title_select="Load an animation",
    title_delete="Delete an animation",
    on_select=on_mp4_selected,
    on_delete=on_mp4_delete,
    on_clean=on_mp4_clean
)

#SelectDialog("Select A Simulation Result", on_csv_selected)
#display(csv_dialog.dialog)

#mp4_dialog = SelectDialog("Select A Animation", on_mp4_selected)
#display(mp4_dialog.dialog)

# —————————————————————————————————————————————————————————————————————————————————
acc_R = ResearchContext(accor_research)
acc_Q = QuickGuideAccordion(accor_guide)
acc_A = DerivationAccordion(accor_drv, em)
acc_B = R_MatrixAccordion(accor_eqs, em)
acc_C = SupBoxAccordion(accor_sup1, em, (w_om, w_I1, w_I2, w_mB, w_gam, w_th_eq, w_xi))
acc_E = SemiEmpAccordion(accor_empiric)

ModuleNotFoundError: No module named 'eom_diag5'